In [ ]:
# Experiment 002 — BGE-M3
# Cell 1: Load and verify frozen inputs from Experiment 001

import json
import importlib.util
from pathlib import Path

from google.colab import drive
drive.mount("/content/drive")

BASE = Path("/content/drive/MyDrive/maxmin_research")

# ------------------------------------------------------------
# Load frozen artifacts
# ------------------------------------------------------------

with open(BASE / "frozen/msmarco_xi_frozen_benchmark.json", "r") as f:
    benchmark = json.load(f)

with open(BASE / "frozen/msmarco_xi_native_corpus.json", "r") as f:
    native_corpus = json.load(f)

with open(BASE / "frozen/msmarco_xi_sentence_corpus.json", "r") as f:
    sentence_corpus = json.load(f)

# ------------------------------------------------------------
# Load the EXACT frozen sentence splitter from 001
# ------------------------------------------------------------

splitter_path = BASE / "frozen/sentence_splitter.py"

spec = importlib.util.spec_from_file_location(
    "sentence_splitter",
    splitter_path
)
sentence_splitter = importlib.util.module_from_spec(spec)
spec.loader.exec_module(sentence_splitter)

split_sentences = sentence_splitter.split_sentences

# ------------------------------------------------------------
# Verify frozen dataset
# ------------------------------------------------------------

assert len(benchmark) == 459
assert len(native_corpus) == 8973
assert len(sentence_corpus) == 31792

dev = [q for q in benchmark if q["split"] == "dev"]
test = [q for q in benchmark if q["split"] == "test"]

assert len(dev) == 225
assert len(test) == 234

languages = sorted(set(q["language"] for q in benchmark))

assert len(languages) == 9

print("=" * 65)
print("EXPERIMENT 002 — FROZEN INPUT VERIFICATION")
print("=" * 65)

print(f"Base path:              {BASE}")
print(f"Languages:              {languages}")
print(f"Native passages:        {len(native_corpus)}")
print(f"Sentence records:       {len(sentence_corpus)}")
print(f"Benchmark queries:      {len(benchmark)}")
print(f"DEV queries:            {len(dev)}")
print(f"TEST queries:           {len(test)}")

print("\nQueries per language:")
for lang in languages:
    n = sum(q["language"] == lang for q in benchmark)
    d = sum(q["language"] == lang and q["split"] == "dev" for q in benchmark)
    t = sum(q["language"] == lang and q["split"] == "test" for q in benchmark)
    print(f"{lang:10s} total={n:3d}  dev={d:2d}  test={t:2d}")

# ------------------------------------------------------------
# Verify sentence splitter against frozen sentence corpus
# ------------------------------------------------------------

sample = native_corpus[0]
recomputed = split_sentences(sample["text"])

frozen_sentences = [
    row["text"]
    for row in sentence_corpus
    if row["parent_id"] == sample["parent_id"]
]

assert recomputed == frozen_sentences

print("\n✓ Frozen benchmark verified")
print("✓ Frozen native corpus verified")
print("✓ Frozen sentence corpus verified")
print("✓ Exact 001 sentence splitter loaded and verified")
print("✓ Ready to load BGE-M3")

Mounted at /content/drive
EXPERIMENT 002 — FROZEN INPUT VERIFICATION
Base path:              /content/drive/MyDrive/maxmin_research
Languages:              ['bengali', 'gujarati', 'hindi', 'kannada', 'malayalam', 'marathi', 'punjabi', 'tamil', 'telugu']
Native passages:        8973
Sentence records:       31792
Benchmark queries:      459
DEV queries:            225
TEST queries:           234

Queries per language:
bengali    total= 51  dev=25  test=26
gujarati   total= 51  dev=25  test=26
hindi      total= 51  dev=25  test=26
kannada    total= 51  dev=25  test=26
malayalam  total= 51  dev=25  test=26
marathi    total= 51  dev=25  test=26
punjabi    total= 51  dev=25  test=26
tamil      total= 51  dev=25  test=26
telugu     total= 51  dev=25  test=26

✓ Frozen benchmark verified
✓ Frozen native corpus verified
✓ Frozen sentence corpus verified
✓ Exact 001 sentence splitter loaded and verified
✓ Ready to load BGE-M3


In [ ]:
# Experiment 002 — BGE-M3
# Cell 2: Load and verify BGE-M3

import torch
import numpy as np
from sentence_transformers import SentenceTransformer

print("=" * 65)
print("EXPERIMENT 002 — BGE-M3 ENVIRONMENT")
print("=" * 65)

# ------------------------------------------------------------
# GPU check
# ------------------------------------------------------------

assert torch.cuda.is_available(), "CUDA GPU is required for this experiment."

device = "cuda"

print("GPU:", torch.cuda.get_device_name(0))
print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)

# ------------------------------------------------------------
# Load BGE-M3
# ------------------------------------------------------------

MODEL_NAME = "BAAI/bge-m3"

print("\nLoading:", MODEL_NAME)

bge_model = SentenceTransformer(
    MODEL_NAME,
    device=device
)

# Use the model's native maximum sequence length.
print("\nModel:", MODEL_NAME)
print("Embedding dimension:", bge_model.get_sentence_embedding_dimension())
print("Max sequence length:", bge_model.max_seq_length)
print("Device:", bge_model.device)

# ------------------------------------------------------------
# Sanity embedding
# ------------------------------------------------------------

test_texts = [
    "यह एक परीक्षण वाक्य है।",
    "This is a test sentence."
]

test_emb = bge_model.encode(
    test_texts,
    batch_size=2,
    normalize_embeddings=True,
    convert_to_numpy=True,
    show_progress_bar=False
)

print("\nTest embedding shape:", test_emb.shape)

norms = np.linalg.norm(test_emb, axis=1)

print(
    "Test embedding norms:",
    f"min={norms.min():.7f}, "
    f"mean={norms.mean():.7f}, "
    f"max={norms.max():.7f}"
)

assert test_emb.shape[1] == 1024
assert np.allclose(norms, 1.0, atol=1e-5)

print("\n✓ BGE-M3 loaded successfully")
print("✓ 1024-dimensional embeddings confirmed")
print("✓ L2 normalization verified")
print("✓ GPU inference ready")

EXPERIMENT 002 — BGE-M3 ENVIRONMENT
GPU: Tesla T4
PyTorch: 2.11.0+cu130
CUDA: 13.0

Loading: BAAI/bge-m3


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]


Model: BAAI/bge-m3
Embedding dimension: 1024
Max sequence length: 8192
Device: cuda:0


/tmp/ipykernel_6920/3357344511.py:39: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("Embedding dimension:", bge_model.get_sentence_embedding_dimension())



Test embedding shape: (2, 1024)
Test embedding norms: min=1.0000000, mean=1.0000000, max=1.0000001

✓ BGE-M3 loaded successfully
✓ 1024-dimensional embeddings confirmed
✓ L2 normalization verified
✓ GPU inference ready


In [ ]:
# Experiment 002 — BGE-M3
# Cell 3: Check token lengths before embedding

tokenizer = bge_model.tokenizer
MAX_LEN = bge_model.max_seq_length

texts = [row["text"] for row in sentence_corpus]

# Tokenize in batches to avoid excessive RAM use
lengths = []

batch_size = 512

for i in range(0, len(texts), batch_size):
    batch = texts[i:i + batch_size]

    encoded = tokenizer(
        batch,
        add_special_tokens=True,
        padding=False,
        truncation=False
    )

    lengths.extend(len(ids) for ids in encoded["input_ids"])

lengths = np.array(lengths)

print("=" * 65)
print("BGE-M3 TOKEN LENGTH CHECK — SENTENCE CORPUS")
print("=" * 65)

print(f"Sentences checked:     {len(lengths)}")
print(f"Model max length:      {MAX_LEN}")
print(f"Minimum:               {lengths.min()}")
print(f"Mean:                  {lengths.mean():.2f}")
print(f"Median:                {np.median(lengths):.0f}")
print(f"P90:                   {np.percentile(lengths, 90):.0f}")
print(f"P95:                   {np.percentile(lengths, 95):.0f}")
print(f"P99:                   {np.percentile(lengths, 99):.0f}")
print(f"Maximum:               {lengths.max()}")

over_limit = lengths > MAX_LEN

print(f"\nSentences > {MAX_LEN} tokens: {over_limit.sum()}")
print(f"Percentage over limit:       {over_limit.mean() * 100:.4f}%")

assert len(lengths) == len(sentence_corpus)

print("\n✓ All 31,792 sentences checked")
print("✓ No truncation decision made yet")

BGE-M3 TOKEN LENGTH CHECK — SENTENCE CORPUS
Sentences checked:     31792
Model max length:      8192
Minimum:               3
Mean:                  31.02
Median:                27
P90:                   55
P95:                   66
P99:                   94
Maximum:               2663

Sentences > 8192 tokens: 0
Percentage over limit:       0.0000%

✓ All 31,792 sentences checked
✓ No truncation decision made yet


In [ ]:
# Experiment 002 — BGE-M3
# Cell 4: Use fp16 inference and verify normalization

print("Converting BGE-M3 to fp16...")

bge_model.half()

print("Model dtype:", next(bge_model.parameters()).dtype)

# Small fp16 sanity test
test_emb_fp16 = bge_model.encode(
    test_texts,
    batch_size=2,
    normalize_embeddings=False,
    convert_to_numpy=True,
    show_progress_bar=False
)

# Explicit L2 normalization, matching our final embedding protocol
test_emb_fp16 = test_emb_fp16 / np.linalg.norm(
    test_emb_fp16,
    axis=1,
    keepdims=True
)

norms = np.linalg.norm(test_emb_fp16, axis=1)

print("Test embedding shape:", test_emb_fp16.shape)
print(
    "Normalized norms:",
    f"min={norms.min():.7f}, "
    f"mean={norms.mean():.7f}, "
    f"max={norms.max():.7f}"
)

assert next(bge_model.parameters()).dtype == torch.float16
assert test_emb_fp16.shape == (2, 1024)
assert np.allclose(norms, 1.0, atol=1e-5)

print("\n✓ BGE-M3 fp16 verified")
print("✓ Explicit L2 normalization verified")
print("✓ Ready for full sentence embedding")

Converting BGE-M3 to fp16...
Model dtype: torch.float16
Test embedding shape: (2, 1024)
Normalized norms: min=1.0000000, mean=1.0000000, max=1.0000000

✓ BGE-M3 fp16 verified
✓ Explicit L2 normalization verified
✓ Ready for full sentence embedding


In [ ]:
# Experiment 002 — BGE-M3
# Cell 5: Embed all frozen sentences and save to Drive

import time
import json
import numpy as np

SENT_CACHE = BASE / "cache/bge_m3_sentence_embeddings"
SENT_CACHE.mkdir(parents=True, exist_ok=True)

EMB_PATH = SENT_CACHE / "bge_m3_sentence_embeddings.npy"
ID_PATH = SENT_CACHE / "sentence_ids.json"

# ------------------------------------------------------------
# Resume protection
# ------------------------------------------------------------

if EMB_PATH.exists() and ID_PATH.exists():
    print("Existing BGE-M3 sentence embedding cache found.")
    print("Path:", EMB_PATH)

    bge_sentence_embeddings = np.load(EMB_PATH)
    with open(ID_PATH, "r") as f:
        bge_sentence_ids = json.load(f)

    print("Shape:", bge_sentence_embeddings.shape)
    print("IDs:", len(bge_sentence_ids))

    if (
        bge_sentence_embeddings.shape == (31792, 1024)
        and bge_sentence_ids == [r["sentence_id"] for r in sentence_corpus]
    ):
        print("\n✓ Existing cache matches frozen sentence corpus exactly.")
        print("✓ No embedding recomputation needed.")

    else:
        raise ValueError("Existing cache does not match frozen sentence corpus.")

else:
    texts = [r["text"] for r in sentence_corpus]
    original_ids = [r["sentence_id"] for r in sentence_corpus]

    # --------------------------------------------------------
    # Get exact token lengths (already verified in Cell 3)
    # --------------------------------------------------------

    lengths = []
    length_batch = 512

    for i in range(0, len(texts), length_batch):
        batch = texts[i:i + length_batch]

        encoded = tokenizer(
            batch,
            add_special_tokens=True,
            padding=False,
            truncation=False
        )

        lengths.extend(len(x) for x in encoded["input_ids"])

    lengths = np.asarray(lengths)

    assert len(lengths) == len(texts)

    # --------------------------------------------------------
    # Length-sort so padding waste is minimized
    # --------------------------------------------------------

    sorted_indices = np.argsort(lengths)

    # Dynamic batch sizes based on token length.
    # This avoids large batches around unusually long sentences.
    def choose_batch_size(max_tokens):
        if max_tokens <= 64:
            return 128
        elif max_tokens <= 128:
            return 64
        elif max_tokens <= 256:
            return 32
        elif max_tokens <= 512:
            return 16
        elif max_tokens <= 1024:
            return 8
        else:
            return 2

    start = time.time()

    embeddings_sorted = []
    sorted_original_indices = []

    pos = 0
    total = len(sorted_indices)

    while pos < total:
        first_idx = sorted_indices[pos]
        batch_size = choose_batch_size(lengths[first_idx])

        end = min(pos + batch_size, total)
        batch_indices = sorted_indices[pos:end]

        batch_texts = [texts[i] for i in batch_indices]
        max_len = int(lengths[batch_indices].max())

        # Encode without internal normalization,
        # then explicitly L2-normalize ourselves.
        batch_emb = bge_model.encode(
            batch_texts,
            batch_size=len(batch_texts),
            normalize_embeddings=False,
            convert_to_numpy=True,
            show_progress_bar=False
        ).astype(np.float32)

        norms = np.linalg.norm(
            batch_emb,
            axis=1,
            keepdims=True
        )

        batch_emb /= norms

        embeddings_sorted.append(batch_emb)
        sorted_original_indices.extend(batch_indices.tolist())

        pos = end

        if len(embeddings_sorted) % 50 == 0 or pos == total:
            elapsed = time.time() - start
            print(
                f"{pos:5d}/{total} "
                f"({pos/total*100:6.2f}%) | "
                f"max_tokens={max_len:4d} | "
                f"elapsed={elapsed/60:.2f} min"
            )

    # --------------------------------------------------------
    # Restore exact original sentence order
    # --------------------------------------------------------

    sorted_embeddings = np.vstack(embeddings_sorted)

    bge_sentence_embeddings = np.empty_like(sorted_embeddings)

    for sorted_pos, original_idx in enumerate(sorted_original_indices):
        bge_sentence_embeddings[original_idx] = sorted_embeddings[sorted_pos]

    bge_sentence_ids = original_ids.copy()

    elapsed = time.time() - start

    # --------------------------------------------------------
    # Integrity checks
    # --------------------------------------------------------

    assert bge_sentence_embeddings.shape == (31792, 1024)
    assert bge_sentence_ids == original_ids

    final_norms = np.linalg.norm(
        bge_sentence_embeddings,
        axis=1
    )

    print("\nEmbedding complete.")
    print(f"Elapsed time: {elapsed/60:.2f} min")
    print("Shape:", bge_sentence_embeddings.shape)
    print(
        "Norms:",
        f"min={final_norms.min():.7f}, "
        f"mean={final_norms.mean():.7f}, "
        f"max={final_norms.max():.7f}"
    )

    assert np.allclose(final_norms, 1.0, atol=1e-5)

    # --------------------------------------------------------
    # Save to Drive
    # --------------------------------------------------------

    np.save(EMB_PATH, bge_sentence_embeddings)

    with open(ID_PATH, "w") as f:
        json.dump(bge_sentence_ids, f)

    print("\n✓ Saved BGE-M3 sentence embeddings")
    print("✓ Saved sentence IDs")
    print("✓ Original order restored")
    print("✓ All embeddings explicitly L2-normalized")

 6400/31792 ( 20.13%) | max_tokens=  15 | elapsed=0.04 min
12800/31792 ( 40.26%) | max_tokens=  23 | elapsed=0.12 min
19200/31792 ( 60.39%) | max_tokens=  32 | elapsed=0.23 min
25600/31792 ( 80.52%) | max_tokens=  44 | elapsed=0.38 min
31040/31792 ( 97.63%) | max_tokens=  79 | elapsed=0.57 min
31792/31792 (100.00%) | max_tokens=2663 | elapsed=0.68 min

Embedding complete.
Elapsed time: 0.68 min
Shape: (31792, 1024)
Norms: min=0.9999999, mean=1.0000000, max=1.0000001

✓ Saved BGE-M3 sentence embeddings
✓ Saved sentence IDs
✓ Original order restored
✓ All embeddings explicitly L2-normalized


In [ ]:
# Experiment 002 — BGE-M3
# Cell 6: Recompute official-code Max–Min segmentation using BGE sentence embeddings

import json
import math
import time
import numpy as np

# ------------------------------------------------------------
# Frozen Max–Min parameters from Experiment 001
# ------------------------------------------------------------

HARD_THR = 0.6
INIT_CONST = 1.5
C = 0.9

def sigmoid(x):
    return 1.0 / (1.0 + math.exp(-x))


# ------------------------------------------------------------
# Sentence embedding lookup
# ------------------------------------------------------------

assert bge_sentence_embeddings.shape == (31792, 1024)

sentence_id_to_idx = {
    row["sentence_id"]: i
    for i, row in enumerate(sentence_corpus)
}

# Group frozen sentence records by parent
parent_sentences = {}

for row in sentence_corpus:
    parent_sentences.setdefault(row["parent_id"], []).append(row)

assert len(parent_sentences) == 8973


# ------------------------------------------------------------
# Official-code Max–Min implementation
# ------------------------------------------------------------

def maxmin_segment_parent(parent_id):
    sentences = parent_sentences[parent_id]

    n = len(sentences)

    if n == 0:
        return []

    # Sentence vectors in exact frozen sentence order
    vecs = np.vstack([
        bge_sentence_embeddings[
            sentence_id_to_idx[s["sentence_id"]]
        ]
        for s in sentences
    ])

    chunks = []

    # Current chunk stores sentence indices
    current = [0]

    # Running minimum pairwise similarity inside current chunk
    min_sim = None

    for k in range(1, n):

        candidate = vecs[k]

        # ----------------------------------------------------
        # Singleton current chunk
        # ----------------------------------------------------
        if len(current) == 1:
            prev_idx = current[-1]

            sim_prev = float(
                np.dot(vecs[prev_idx], candidate)
            )

            merge = (
                INIT_CONST * sim_prev > HARD_THR
            )

            if merge:
                current.append(k)

                # First pairwise similarity in the chunk
                min_sim = sim_prev

            else:
                chunks.append(current)
                current = [k]
                min_sim = None

        # ----------------------------------------------------
        # Multi-sentence current chunk
        # ----------------------------------------------------
        else:
            # Similarity of candidate to every sentence
            # already in the current chunk
            sims = np.dot(
                vecs[current],
                candidate
            )

            max_sim = float(np.max(sims))

            # Existing minimum pairwise similarity
            # within current chunk.
            #
            # Since only the candidate is being added,
            # the new running minimum is:
            # min(previous minimum,
            #     minimum similarity(candidate, C))
            candidate_min_sim = float(np.min(sims))

            threshold = max(
                C
                * min_sim
                * sigmoid(len(current) - 1),
                HARD_THR
            )

            merge = max_sim > threshold

            if merge:
                min_sim = min(
                    min_sim,
                    candidate_min_sim
                )
                current.append(k)

            else:
                chunks.append(current)
                current = [k]
                min_sim = None

    # Final open chunk
    chunks.append(current)

    return chunks


# ------------------------------------------------------------
# Run across all frozen passages
# ------------------------------------------------------------

start = time.time()

bge_maxmin_segmentation = []

total_chunks = 0

for i, passage in enumerate(native_corpus):

    parent_id = passage["parent_id"]

    chunk_indices = maxmin_segment_parent(parent_id)

    record = {
        "parent_id": parent_id,
        "language": passage["language"],
        "source_query_id": passage["source_query_id"],
        "passage_idx": passage["passage_idx"],
        "num_sentences": len(parent_sentences[parent_id]),
        "num_chunks": len(chunk_indices),
        "chunks": [
            {
                "chunk_idx": j,
                "sentence_indices": indices,
                "num_sentences": len(indices)
            }
            for j, indices in enumerate(chunk_indices)
        ]
    }

    bge_maxmin_segmentation.append(record)
    total_chunks += len(chunk_indices)

    if (i + 1) % 1000 == 0 or (i + 1) == len(native_corpus):
        print(
            f"{i+1:5d}/{len(native_corpus)} "
            f"({(i+1)/len(native_corpus)*100:6.2f}%)"
        )

elapsed = time.time() - start

# ------------------------------------------------------------
# Structural summary
# ------------------------------------------------------------

chunk_sizes = [
    c["num_sentences"]
    for row in bge_maxmin_segmentation
    for c in row["chunks"]
]

singleton_count = sum(s == 1 for s in chunk_sizes)
one_chunk_passages = sum(
    row["num_chunks"] == 1
    for row in bge_maxmin_segmentation
)

print("\n" + "=" * 65)
print("BGE-M3 MAX–MIN SEGMENTATION")
print("=" * 65)

print("Passages:", len(bge_maxmin_segmentation))
print("Total chunks:", total_chunks)
print("Chunks/passage:", total_chunks / len(bge_maxmin_segmentation))
print("Sentences/chunk:", np.mean(chunk_sizes))
print("Median sentences/chunk:", np.median(chunk_sizes))
print(
    "Singleton chunks:",
    singleton_count,
    f"({singleton_count / len(chunk_sizes) * 100:.2f}%)"
)
print(
    "One-chunk passages:",
    one_chunk_passages,
    f"({one_chunk_passages / len(bge_maxmin_segmentation) * 100:.2f}%)"
)
print("Max sentences/chunk:", max(chunk_sizes))
print(f"Elapsed: {elapsed:.2f} sec")

# ------------------------------------------------------------
# Save frozen BGE segmentation
# ------------------------------------------------------------

out_path = (
    BASE /
    "frozen/maxmin_segmentation_bge_m3.json"
)

with open(out_path, "w") as f:
    json.dump(
        {
            "method": "max_min",
            "embedding_model": "BAAI/bge-m3",
            "hard_thr": HARD_THR,
            "init_const": INIT_CONST,
            "c": C,
            "sigmoid_variant": "sigmoid(|C|-1)",
            "implementation_reference": "official_code",
            "max_chunk_size": None,
            "similarity_window": None,
            "num_passages": len(bge_maxmin_segmentation),
            "num_sentences": 31792,
            "num_chunks": total_chunks,
            "segmentation": bge_maxmin_segmentation
        },
        f
    )

print("\n✓ BGE-M3 Max–Min segmentation saved")
print("✓ Official-code parameters preserved")
print("✓ Boundaries recomputed from BGE-M3 embeddings")
print("✓ Qwen boundaries were NOT reused")

 1000/8973 ( 11.14%)
 2000/8973 ( 22.29%)
 3000/8973 ( 33.43%)
 4000/8973 ( 44.58%)
 5000/8973 ( 55.72%)
 6000/8973 ( 66.87%)
 7000/8973 ( 78.01%)
 8000/8973 ( 89.16%)
 8973/8973 (100.00%)

BGE-M3 MAX–MIN SEGMENTATION
Passages: 8973
Total chunks: 16763
Chunks/passage: 1.8681600356625432
Sentences/chunk: 1.8965578953647915
Median sentences/chunk: 2.0
Singleton chunks: 6672 (39.80%)
One-chunk passages: 4179 (46.57%)
Max sentences/chunk: 135
Elapsed: 1.18 sec

✓ BGE-M3 Max–Min segmentation saved
✓ Official-code parameters preserved
✓ Boundaries recomputed from BGE-M3 embeddings
✓ Qwen boundaries were NOT reused


In [ ]:
# Experiment 002 — BGE-M3
# Cell 7: Check token lengths of native passages and BGE Max–Min chunks

import numpy as np
import pandas as pd

# ------------------------------------------------------------
# Build exact parent -> sentence list
# ------------------------------------------------------------

parent_sentences = {}

for row in sentence_corpus:
    parent_sentences.setdefault(row["parent_id"], []).append(row)

assert len(parent_sentences) == 8973

# ------------------------------------------------------------
# Build BGE Max–Min chunk records
# ------------------------------------------------------------

bge_chunk_corpus = []

for seg in bge_maxmin_segmentation:

    parent_id = seg["parent_id"]
    sentences = parent_sentences[parent_id]

    for chunk in seg["chunks"]:

        indices = chunk["sentence_indices"]

        chunk_text = " ".join(
            sentences[i]["text"]
            for i in indices
        )

        bge_chunk_corpus.append({
            "chunk_id": f"{parent_id}_bge_maxmin_{chunk['chunk_idx']}",
            "parent_id": parent_id,
            "language": seg["language"],
            "chunk_idx": chunk["chunk_idx"],
            "text": chunk_text,
            "num_sentences": chunk["num_sentences"]
        })

assert len(bge_chunk_corpus) == 16763

# ------------------------------------------------------------
# Token-length helper
# ------------------------------------------------------------

def get_token_lengths(texts, batch_size=256):

    lengths = []

    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]

        encoded = tokenizer(
            batch,
            add_special_tokens=True,
            padding=False,
            truncation=False
        )

        lengths.extend(
            len(ids)
            for ids in encoded["input_ids"]
        )

    return np.asarray(lengths)


# ------------------------------------------------------------
# Native passage lengths
# ------------------------------------------------------------

native_texts = [
    row["text"]
    for row in native_corpus
]

native_lengths = get_token_lengths(native_texts)

# ------------------------------------------------------------
# BGE Max–Min chunk lengths
# ------------------------------------------------------------

chunk_texts = [
    row["text"]
    for row in bge_chunk_corpus
]

chunk_lengths = get_token_lengths(chunk_texts)

# ------------------------------------------------------------
# Report
# ------------------------------------------------------------

MAX_LEN = 8192

print("=" * 70)
print("BGE-M3 TOKEN LENGTH CHECK")
print("=" * 70)

print("\nNATIVE PASSAGES")
print("-" * 40)
print("Count:", len(native_lengths))
print("Min:", native_lengths.min())
print("Mean:", f"{native_lengths.mean():.2f}")
print("Median:", np.median(native_lengths))
print("P90:", np.percentile(native_lengths, 90))
print("P95:", np.percentile(native_lengths, 95))
print("P99:", np.percentile(native_lengths, 99))
print("Max:", native_lengths.max())
print(f">{MAX_LEN}:", (native_lengths > MAX_LEN).sum())
print(
    f"% > {MAX_LEN}:",
    f"{(native_lengths > MAX_LEN).mean() * 100:.4f}%"
)

print("\nBGE MAX–MIN CHUNKS")
print("-" * 40)
print("Count:", len(chunk_lengths))
print("Min:", chunk_lengths.min())
print("Mean:", f"{chunk_lengths.mean():.2f}")
print("Median:", np.median(chunk_lengths))
print("P90:", np.percentile(chunk_lengths, 90))
print("P95:", np.percentile(chunk_lengths, 95))
print("P99:", np.percentile(chunk_lengths, 99))
print("Max:", chunk_lengths.max())
print(f">{MAX_LEN}:", (chunk_lengths > MAX_LEN).sum())
print(
    f"% > {MAX_LEN}:",
    f"{(chunk_lengths > MAX_LEN).mean() * 100:.4f}%"
)

# ------------------------------------------------------------
# Exact records over the limit
# ------------------------------------------------------------

if (native_lengths > MAX_LEN).any():
    print("\nLongest native passages:")
    idxs = np.argsort(native_lengths)[-10:][::-1]

    for idx in idxs:
        if native_lengths[idx] > MAX_LEN:
            print(
                native_corpus[idx]["parent_id"],
                "tokens=",
                int(native_lengths[idx])
            )

if (chunk_lengths > MAX_LEN).any():
    print("\nLongest BGE Max–Min chunks:")
    idxs = np.argsort(chunk_lengths)[-10:][::-1]

    for idx in idxs:
        if chunk_lengths[idx] > MAX_LEN:
            row = bge_chunk_corpus[idx]
            print(
                row["chunk_id"],
                "tokens=",
                int(chunk_lengths[idx]),
                "sentences=",
                row["num_sentences"]
            )

print("\n✓ BGE-M3 length check complete")
print("✓ No embedding performed in this cell")

BGE-M3 TOKEN LENGTH CHECK

NATIVE PASSAGES
----------------------------------------
Count: 8973
Min: 10
Mean: 104.81
Median: 96.0
P90: 149.0
P95: 174.0
P99: 235.0
Max: 2719
>8192: 0
% > 8192: 0.0000%

BGE MAX–MIN CHUNKS
----------------------------------------
Count: 16763
Min: 3
Mean: 57.03
Median: 51.0
P90: 102.0
P95: 117.0
P99: 163.0
Max: 2663
>8192: 0
% > 8192: 0.0000%

✓ BGE-M3 length check complete
✓ No embedding performed in this cell


In [ ]:
# Experiment 002 — BGE-M3
# Cell 8: Embed all native passages and save to Drive

import time
import json
import numpy as np

NATIVE_CACHE = BASE / "cache/bge_m3_native_embeddings"
NATIVE_CACHE.mkdir(parents=True, exist_ok=True)

EMB_PATH = NATIVE_CACHE / "bge_m3_native_embeddings.npy"
ID_PATH = NATIVE_CACHE / "parent_ids.json"

native_texts = [row["text"] for row in native_corpus]
native_ids = [row["parent_id"] for row in native_corpus]

# ------------------------------------------------------------
# Resume protection
# ------------------------------------------------------------

if EMB_PATH.exists() and ID_PATH.exists():

    print("Existing BGE-M3 native embedding cache found.")

    bge_native_embeddings = np.load(EMB_PATH)

    with open(ID_PATH, "r") as f:
        bge_native_ids = json.load(f)

    print("Shape:", bge_native_embeddings.shape)
    print("IDs:", len(bge_native_ids))

    if (
        bge_native_embeddings.shape == (8973, 1024)
        and bge_native_ids == native_ids
    ):
        print("\n✓ Existing cache matches frozen native corpus exactly.")
        print("✓ No recomputation needed.")

    else:
        raise ValueError(
            "Existing BGE native cache does not match frozen corpus."
        )

else:

    # --------------------------------------------------------
    # Token lengths
    # --------------------------------------------------------

    lengths = []

    for i in range(0, len(native_texts), 256):

        batch = native_texts[i:i + 256]

        encoded = tokenizer(
            batch,
            add_special_tokens=True,
            padding=False,
            truncation=False
        )

        lengths.extend(
            len(x)
            for x in encoded["input_ids"]
        )

    lengths = np.asarray(lengths)

    assert len(lengths) == len(native_texts)
    assert lengths.max() <= MAX_LEN

    # --------------------------------------------------------
    # Length-sorted embedding
    # --------------------------------------------------------

    sorted_indices = np.argsort(lengths)

    def choose_batch_size(max_tokens):

        if max_tokens <= 128:
            return 64
        elif max_tokens <= 256:
            return 32
        elif max_tokens <= 512:
            return 16
        elif max_tokens <= 1024:
            return 8
        elif max_tokens <= 2048:
            return 4
        else:
            return 2

    start = time.time()

    embeddings_sorted = []
    sorted_original_indices = []

    pos = 0
    total = len(sorted_indices)

    while pos < total:

        first_idx = sorted_indices[pos]
        batch_size = choose_batch_size(
            int(lengths[first_idx])
        )

        end = min(pos + batch_size, total)

        batch_indices = sorted_indices[pos:end]

        batch_texts = [
            native_texts[i]
            for i in batch_indices
        ]

        batch_emb = bge_model.encode(
            batch_texts,
            batch_size=len(batch_texts),
            normalize_embeddings=False,
            convert_to_numpy=True,
            show_progress_bar=False
        ).astype(np.float32)

        # Explicit L2 normalization
        norms = np.linalg.norm(
            batch_emb,
            axis=1,
            keepdims=True
        )

        batch_emb /= norms

        embeddings_sorted.append(batch_emb)
        sorted_original_indices.extend(
            batch_indices.tolist()
        )

        pos = end

        if pos % 1000 == 0 or pos == total:

            elapsed = time.time() - start

            print(
                f"{pos:5d}/{total} "
                f"({pos/total*100:6.2f}%) | "
                f"elapsed={elapsed/60:.2f} min"
            )

    # --------------------------------------------------------
    # Restore original corpus order
    # --------------------------------------------------------

    sorted_embeddings = np.vstack(
        embeddings_sorted
    )

    bge_native_embeddings = np.empty_like(
        sorted_embeddings
    )

    for sorted_pos, original_idx in enumerate(
        sorted_original_indices
    ):
        bge_native_embeddings[original_idx] = (
            sorted_embeddings[sorted_pos]
        )

    bge_native_ids = native_ids.copy()

    # --------------------------------------------------------
    # Integrity checks
    # --------------------------------------------------------

    assert bge_native_embeddings.shape == (
        8973, 1024
    )

    assert bge_native_ids == native_ids

    norms = np.linalg.norm(
        bge_native_embeddings,
        axis=1
    )

    print("\nEmbedding complete.")
    print(f"Elapsed: {time.time() - start:.2f} sec")
    print("Shape:", bge_native_embeddings.shape)
    print(
        "Norms:",
        f"min={norms.min():.7f}, "
        f"mean={norms.mean():.7f}, "
        f"max={norms.max():.7f}"
    )

    assert np.allclose(
        norms,
        1.0,
        atol=1e-5
    )

    # --------------------------------------------------------
    # Save
    # --------------------------------------------------------

    np.save(
        EMB_PATH,
        bge_native_embeddings
    )

    with open(ID_PATH, "w") as f:
        json.dump(bge_native_ids, f)

    print("\n✓ BGE-M3 native embeddings saved")
    print("✓ Parent IDs saved")
    print("✓ Original corpus order restored")
    print("✓ Explicit L2 normalization verified")

 8000/8973 ( 89.16%) | elapsed=0.51 min
 8973/8973 (100.00%) | elapsed=0.67 min

Embedding complete.
Elapsed: 40.01 sec
Shape: (8973, 1024)
Norms: min=0.9999999, mean=1.0000000, max=1.0000001

✓ BGE-M3 native embeddings saved
✓ Parent IDs saved
✓ Original corpus order restored
✓ Explicit L2 normalization verified


In [ ]:
# Experiment 002 — BGE-M3
# Cell 9: Embed all frozen queries and save to Drive

import json
import numpy as np

QUERY_CACHE = BASE / "cache/bge_m3_query_embeddings"
QUERY_CACHE.mkdir(parents=True, exist_ok=True)

EMB_PATH = QUERY_CACHE / "bge_m3_query_embeddings.npy"
ID_PATH = QUERY_CACHE / "query_keys.json"

# Exact frozen query order
query_records = benchmark
query_keys = [
    (q["language"], q["query_id"])
    for q in query_records
]

query_texts = [
    q["query"]
    for q in query_records
]

# ------------------------------------------------------------
# Resume protection
# ------------------------------------------------------------

if EMB_PATH.exists() and ID_PATH.exists():

    print("Existing BGE-M3 query embedding cache found.")

    bge_query_embeddings = np.load(EMB_PATH)

    with open(ID_PATH, "r") as f:
        cached_query_keys = [
            tuple(x) for x in json.load(f)
        ]

    print("Shape:", bge_query_embeddings.shape)
    print("Query keys:", len(cached_query_keys))

    if (
        bge_query_embeddings.shape == (459, 1024)
        and cached_query_keys == query_keys
    ):
        print("\n✓ Existing cache matches frozen benchmark exactly.")
        print("✓ No recomputation needed.")

    else:
        raise ValueError(
            "Existing BGE query cache does not match frozen benchmark."
        )

else:

    print("Embedding 459 frozen queries...")

    # BGE-M3 retrieval queries are encoded directly.
    # No Qwen instruction prompt is reused.

    bge_query_embeddings = bge_model.encode(
        query_texts,
        batch_size=64,
        normalize_embeddings=False,
        convert_to_numpy=True,
        show_progress_bar=True
    ).astype(np.float32)

    # Explicit L2 normalization
    norms = np.linalg.norm(
        bge_query_embeddings,
        axis=1,
        keepdims=True
    )

    bge_query_embeddings /= norms

    # --------------------------------------------------------
    # Integrity checks
    # --------------------------------------------------------

    assert bge_query_embeddings.shape == (459, 1024)

    final_norms = np.linalg.norm(
        bge_query_embeddings,
        axis=1
    )

    print("\nEmbedding complete.")
    print("Shape:", bge_query_embeddings.shape)
    print(
        "Norms:",
        f"min={final_norms.min():.7f}, "
        f"mean={final_norms.mean():.7f}, "
        f"max={final_norms.max():.7f}"
    )

    assert np.allclose(
        final_norms,
        1.0,
        atol=1e-5
    )

    # --------------------------------------------------------
    # Save
    # --------------------------------------------------------

    np.save(
        EMB_PATH,
        bge_query_embeddings
    )

    with open(ID_PATH, "w") as f:
        json.dump(
            [list(x) for x in query_keys],
            f
        )

    print("\n✓ BGE-M3 query embeddings saved")
    print("✓ Query keys saved")
    print("✓ Frozen query order preserved")
    print("✓ Explicit L2 normalization verified")

Embedding 459 frozen queries...


Batches:   0%|          | 0/8 [00:00<?, ?it/s]


Embedding complete.
Shape: (459, 1024)
Norms: min=0.9999999, mean=1.0000000, max=1.0000001

✓ BGE-M3 query embeddings saved
✓ Query keys saved
✓ Frozen query order preserved
✓ Explicit L2 normalization verified


In [ ]:
# Experiment 002 — BGE-M3
# Cell 10: Build Fixed-2 corpus and determine embedding reuse

import hashlib
import json
from collections import Counter
from pathlib import Path

FIXED2_DIR = BASE / "frozen"
FIXED2_CACHE_DIR = BASE / "cache/bge_m3_fixed2_embeddings"

FIXED2_CACHE_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# Build Fixed-2 corpus using the EXACT frozen splitter
# ------------------------------------------------------------

bge_fixed2_corpus = []

for passage in native_corpus:

    parent_id = passage["parent_id"]
    language = passage["language"]

    sentences = split_sentences(passage["text"])

    # Verify against frozen sentence corpus
    frozen_sentences = [
        row["text"]
        for row in sentence_corpus
        if row["parent_id"] == parent_id
    ]

    assert sentences == frozen_sentences

    chunks = [
        sentences[i:i + 2]
        for i in range(0, len(sentences), 2)
    ]

    for chunk_idx, chunk_sentences in enumerate(chunks):

        chunk_text = " ".join(chunk_sentences)

        bge_fixed2_corpus.append({
            "chunk_id": f"{parent_id}_fixed2_{chunk_idx}",
            "parent_id": parent_id,
            "language": language,
            "chunk_idx": chunk_idx,
            "text": chunk_text,
            "num_sentences": len(chunk_sentences)
        })

# ------------------------------------------------------------
# Frozen-count verification
# ------------------------------------------------------------

assert len(bge_fixed2_corpus) == 18171

print("=" * 70)
print("BGE-M3 FIXED-2 CORPUS")
print("=" * 70)

print("Total chunks:", len(bge_fixed2_corpus))

language_counts = Counter(
    row["language"]
    for row in bge_fixed2_corpus
)

print("\nChunks by language:")
for lang in sorted(language_counts):
    print(f"{lang:10s} {language_counts[lang]}")

# ------------------------------------------------------------
# Build text hashes
# ------------------------------------------------------------

def text_hash(text):
    return hashlib.sha256(
        text.encode("utf-8")
    ).hexdigest()

# Sentence hash -> index
sentence_hash_to_idx = {}

for i, row in enumerate(sentence_corpus):
    h = text_hash(row["text"])
    sentence_hash_to_idx.setdefault(h, []).append(i)

# Native text hash -> index
native_hash_to_idx = {}

for i, row in enumerate(native_corpus):
    h = text_hash(row["text"])
    native_hash_to_idx.setdefault(h, []).append(i)

# ------------------------------------------------------------
# Determine reuse/new counts
# ------------------------------------------------------------

reuse_sentence = 0
reuse_native = 0
needs_new = 0

new_unique_hashes = set()
duplicate_new_instances = 0

for row in bge_fixed2_corpus:

    h = text_hash(row["text"])

    if row["num_sentences"] == 1 and h in sentence_hash_to_idx:
        reuse_sentence += 1

    elif h in native_hash_to_idx:
        # Covers whole-passage identical text,
        # especially 2-sentence native passages.
        reuse_native += 1

    else:
        if h in new_unique_hashes:
            duplicate_new_instances += 1
        else:
            new_unique_hashes.add(h)

        needs_new += 1

print("\nEmbedding reuse plan:")
print("Single-sentence chunks → sentence embeddings:", reuse_sentence)
print("Chunks identical to native passage:", reuse_native)
print("New chunk instances:", needs_new)
print("New unique chunk texts:", len(new_unique_hashes))
print("Duplicate instances among new texts:", duplicate_new_instances)

# ------------------------------------------------------------
# Save frozen Fixed-2 corpus
# ------------------------------------------------------------

corpus_path = FIXED2_DIR / "fixed2_chunk_corpus_bge_m3.json"

with open(corpus_path, "w") as f:
    json.dump(
        bge_fixed2_corpus,
        f,
        ensure_ascii=False
    )

print("\n✓ Fixed-2 corpus saved")
print("✓ Frozen sentence splitter verified")
print("✓ 18,171 chunks confirmed")
print("✓ Embedding reuse/new-work plan computed")
print("✓ No BGE chunk embeddings generated in this cell")

BGE-M3 FIXED-2 CORPUS
Total chunks: 18171

Chunks by language:
bengali    1875
gujarati   2028
hindi      2038
kannada    2086
malayalam  2021
marathi    2111
punjabi    2029
tamil      1995
telugu     1988

Embedding reuse plan:
Single-sentence chunks → sentence embeddings: 4550
Chunks identical to native passage: 1946
New chunk instances: 11675
New unique chunk texts: 11320
Duplicate instances among new texts: 355

✓ Fixed-2 corpus saved
✓ Frozen sentence splitter verified
✓ 18,171 chunks confirmed
✓ Embedding reuse/new-work plan computed
✓ No BGE chunk embeddings generated in this cell


In [ ]:
# Experiment 002 — BGE-M3
# Cell 11: Token-length check for NEW unique Fixed-2 texts only

import numpy as np

# Recreate the exact set of new unique texts from Cell 10
new_unique_texts = {}

for row in bge_fixed2_corpus:

    h = text_hash(row["text"])

    # Single-sentence chunks are reused from sentence embeddings.
    if row["num_sentences"] == 1 and h in sentence_hash_to_idx:
        continue

    # Identical-to-native chunks are reused from native embeddings.
    if h in native_hash_to_idx:
        continue

    # Keep exactly one instance of each new text.
    if h not in new_unique_texts:
        new_unique_texts[h] = row["text"]

new_unique_hashes = list(new_unique_texts.keys())
new_unique_texts_list = [
    new_unique_texts[h]
    for h in new_unique_hashes
]

assert len(new_unique_texts_list) == 11320

# ------------------------------------------------------------
# Tokenize without truncation
# ------------------------------------------------------------

new_lengths = []

for i in range(0, len(new_unique_texts_list), 256):

    batch = new_unique_texts_list[i:i + 256]

    encoded = tokenizer(
        batch,
        add_special_tokens=True,
        padding=False,
        truncation=False
    )

    new_lengths.extend(
        len(ids)
        for ids in encoded["input_ids"]
    )

new_lengths = np.asarray(new_lengths)

assert len(new_lengths) == 11320

MAX_LEN = 8192

print("=" * 70)
print("BGE-M3 — NEW FIXED-2 TEXT LENGTH CHECK")
print("=" * 70)

print("Unique new texts:", len(new_lengths))
print("Minimum:", new_lengths.min())
print("Mean:", f"{new_lengths.mean():.2f}")
print("Median:", np.median(new_lengths))
print("P90:", np.percentile(new_lengths, 90))
print("P95:", np.percentile(new_lengths, 95))
print("P99:", np.percentile(new_lengths, 99))
print("Maximum:", new_lengths.max())

over_limit = new_lengths > MAX_LEN

print(f"\n> {MAX_LEN} tokens:", over_limit.sum())
print(
    f"% > {MAX_LEN}:",
    f"{over_limit.mean() * 100:.4f}%"
)

assert over_limit.sum() == 0

# ------------------------------------------------------------
# Save exact hash ordering for the upcoming embedding step
# ------------------------------------------------------------

new_hash_path = (
    BASE / "cache/bge_m3_fixed2_embeddings"
    / "new_unique_text_hashes.json"
)

with open(new_hash_path, "w") as f:
    json.dump(new_unique_hashes, f)

print("\n✓ 11,320 unique new texts verified")
print("✓ No text exceeds BGE-M3's 8,192-token limit")
print("✓ Exact new-text hash list saved")
print("✓ Ready for BGE-M3 Fixed-2 embedding")

BGE-M3 — NEW FIXED-2 TEXT LENGTH CHECK
Unique new texts: 11320
Minimum: 4
Mean: 51.06
Median: 48.0
P90: 81.0
P95: 92.0
P99: 114.0
Maximum: 1253

> 8192 tokens: 0
% > 8192: 0.0000%

✓ 11,320 unique new texts verified
✓ No text exceeds BGE-M3's 8,192-token limit
✓ Exact new-text hash list saved
✓ Ready for BGE-M3 Fixed-2 embedding


In [ ]:
# Experiment 002 — BGE-M3
# Cell 12: Embed ONLY new unique Fixed-2 texts

import time
import json
import numpy as np

FIXED2_CACHE_DIR = BASE / "cache/bge_m3_fixed2_embeddings"
FIXED2_CACHE_DIR.mkdir(parents=True, exist_ok=True)

NEW_HASHES_PATH = FIXED2_CACHE_DIR / "new_unique_text_hashes.json"
NEW_EMB_PATH = FIXED2_CACHE_DIR / "new_unique_text_embeddings.npy"

# ------------------------------------------------------------
# Load exact new-text hash list from Cell 11
# ------------------------------------------------------------

with open(NEW_HASHES_PATH, "r") as f:
    new_unique_hashes = json.load(f)

assert len(new_unique_hashes) == 11320

new_unique_texts = [
    new_unique_texts[h]
    for h in new_unique_hashes
]

assert len(new_unique_texts) == 11320

# ------------------------------------------------------------
# Resume protection
# ------------------------------------------------------------

if NEW_EMB_PATH.exists():

    print("Existing new Fixed-2 embedding cache found.")

    new_unique_embeddings = np.load(NEW_EMB_PATH)

    print("Shape:", new_unique_embeddings.shape)

    assert new_unique_embeddings.shape == (11320, 1024)

    norms = np.linalg.norm(
        new_unique_embeddings,
        axis=1
    )

    print(
        "Norms:",
        f"min={norms.min():.7f}, "
        f"mean={norms.mean():.7f}, "
        f"max={norms.max():.7f}"
    )

    assert np.allclose(norms, 1.0, atol=1e-5)

    print("\n✓ Existing cache verified")
    print("✓ No recomputation needed")

else:

    # --------------------------------------------------------
    # Token lengths
    # --------------------------------------------------------

    lengths = []

    for i in range(0, len(new_unique_texts), 256):

        batch = new_unique_texts[i:i + 256]

        encoded = tokenizer(
            batch,
            add_special_tokens=True,
            padding=False,
            truncation=False
        )

        lengths.extend(
            len(x)
            for x in encoded["input_ids"]
        )

    lengths = np.asarray(lengths)

    assert len(lengths) == 11320
    assert lengths.max() <= MAX_LEN

    # --------------------------------------------------------
    # Dynamic batch sizing
    # --------------------------------------------------------

    def choose_batch_size(max_tokens):

        if max_tokens <= 128:
            return 64
        elif max_tokens <= 256:
            return 32
        elif max_tokens <= 512:
            return 16
        elif max_tokens <= 1024:
            return 8
        else:
            return 4

    sorted_indices = np.argsort(lengths)

    start = time.time()

    embeddings_sorted = []
    sorted_original_indices = []

    pos = 0
    total = len(sorted_indices)

    while pos < total:

        first_idx = sorted_indices[pos]

        batch_size = choose_batch_size(
            int(lengths[first_idx])
        )

        end = min(
            pos + batch_size,
            total
        )

        batch_indices = sorted_indices[pos:end]

        batch_texts = [
            new_unique_texts[i]
            for i in batch_indices
        ]

        batch_emb = bge_model.encode(
            batch_texts,
            batch_size=len(batch_texts),
            normalize_embeddings=False,
            convert_to_numpy=True,
            show_progress_bar=False
        ).astype(np.float32)

        # Explicit L2 normalization
        norms = np.linalg.norm(
            batch_emb,
            axis=1,
            keepdims=True
        )

        batch_emb /= norms

        embeddings_sorted.append(batch_emb)
        sorted_original_indices.extend(
            batch_indices.tolist()
        )

        pos = end

        if pos % 1000 < batch_size or pos == total:

            elapsed = time.time() - start

            print(
                f"{pos:5d}/{total} "
                f"({pos/total*100:6.2f}%) | "
                f"elapsed={elapsed/60:.2f} min"
            )

    # --------------------------------------------------------
    # Restore original hash order
    # --------------------------------------------------------

    sorted_embeddings = np.vstack(
        embeddings_sorted
    )

    new_unique_embeddings = np.empty_like(
        sorted_embeddings
    )

    for sorted_pos, original_idx in enumerate(
        sorted_original_indices
    ):
        new_unique_embeddings[original_idx] = (
            sorted_embeddings[sorted_pos]
        )

    # --------------------------------------------------------
    # Integrity checks
    # --------------------------------------------------------

    assert new_unique_embeddings.shape == (
        11320, 1024
    )

    final_norms = np.linalg.norm(
        new_unique_embeddings,
        axis=1
    )

    print("\nEmbedding complete.")
    print(
        f"Elapsed: {(time.time() - start)/60:.2f} min"
    )
    print("Shape:", new_unique_embeddings.shape)
    print(
        "Norms:",
        f"min={final_norms.min():.7f}, "
        f"mean={final_norms.mean():.7f}, "
        f"max={final_norms.max():.7f}"
    )

    assert np.allclose(
        final_norms,
        1.0,
        atol=1e-5
    )

    # --------------------------------------------------------
    # Save
    # --------------------------------------------------------

    np.save(
        NEW_EMB_PATH,
        new_unique_embeddings
    )

    print("\n✓ Saved 11,320 unique new Fixed-2 embeddings")
    print("✓ Hash ordering preserved")
    print("✓ Duplicate chunk texts can reuse these vectors")
    print("✓ No unnecessary duplicate embedding performed")

 1024/11320 (  9.05%) | elapsed=0.01 min
 2048/11320 ( 18.09%) | elapsed=0.03 min
 3008/11320 ( 26.57%) | elapsed=0.05 min
 4032/11320 ( 35.62%) | elapsed=0.08 min
 5056/11320 ( 44.66%) | elapsed=0.11 min
 6016/11320 ( 53.14%) | elapsed=0.14 min
 7040/11320 ( 62.19%) | elapsed=0.18 min
 8000/11320 ( 70.67%) | elapsed=0.22 min
 9024/11320 ( 79.72%) | elapsed=0.27 min
10048/11320 ( 88.76%) | elapsed=0.32 min
11008/11320 ( 97.24%) | elapsed=0.38 min
11320/11320 (100.00%) | elapsed=0.47 min

Embedding complete.
Elapsed: 0.47 min
Shape: (11320, 1024)
Norms: min=0.9999999, mean=1.0000000, max=1.0000001

✓ Saved 11,320 unique new Fixed-2 embeddings
✓ Hash ordering preserved
✓ Duplicate chunk texts can reuse these vectors
✓ No unnecessary duplicate embedding performed


In [ ]:
# Experiment 002 — BGE-M3
# Cell 13: Assemble complete Fixed-2 embedding matrix

import json
import numpy as np

FIXED2_CACHE_DIR = BASE / "cache/bge_m3_fixed2_embeddings"

NEW_EMB_PATH = FIXED2_CACHE_DIR / "new_unique_text_embeddings.npy"

# ------------------------------------------------------------
# Load cached new embeddings
# ------------------------------------------------------------

new_unique_embeddings = np.load(NEW_EMB_PATH)

assert new_unique_embeddings.shape == (11320, 1024)
assert len(new_unique_hashes) == 11320

new_hash_to_idx = {
    h: i
    for i, h in enumerate(new_unique_hashes)
}

# ------------------------------------------------------------
# Build full Fixed-2 embedding matrix
# ------------------------------------------------------------

N = len(bge_fixed2_corpus)

bge_fixed2_embeddings = np.empty(
    (N, 1024),
    dtype=np.float32
)

fixed2_ids = []

reuse_sentence_count = 0
reuse_native_count = 0
reuse_new_count = 0

for i, row in enumerate(bge_fixed2_corpus):

    text = row["text"]
    h = text_hash(text)

    # --------------------------------------------------------
    # Single-sentence chunk → sentence embedding
    # --------------------------------------------------------
    if (
        row["num_sentences"] == 1
        and h in sentence_hash_to_idx
    ):

        sentence_idx = sentence_hash_to_idx[h][0]

        bge_fixed2_embeddings[i] = (
            bge_sentence_embeddings[sentence_idx]
        )

        reuse_sentence_count += 1

    # --------------------------------------------------------
    # Chunk identical to a native passage → native embedding
    # --------------------------------------------------------
    elif h in native_hash_to_idx:

        native_idx = native_hash_to_idx[h][0]

        bge_fixed2_embeddings[i] = (
            bge_native_embeddings[native_idx]
        )

        reuse_native_count += 1

    # --------------------------------------------------------
    # Otherwise → newly embedded unique Fixed-2 text
    # --------------------------------------------------------
    else:

        assert h in new_hash_to_idx

        new_idx = new_hash_to_idx[h]

        bge_fixed2_embeddings[i] = (
            new_unique_embeddings[new_idx]
        )

        reuse_new_count += 1

    fixed2_ids.append(row["chunk_id"])


# ------------------------------------------------------------
# Integrity checks
# ------------------------------------------------------------

assert bge_fixed2_embeddings.shape == (
    18171, 1024
)

assert len(fixed2_ids) == 18171
assert len(set(fixed2_ids)) == 18171

norms = np.linalg.norm(
    bge_fixed2_embeddings,
    axis=1
)

print("=" * 70)
print("BGE-M3 FIXED-2 — COMPLETE EMBEDDING ASSEMBLY")
print("=" * 70)

print("Shape:", bge_fixed2_embeddings.shape)

print("\nEmbedding sources:")
print(
    "Sentence embeddings:",
    reuse_sentence_count
)
print(
    "Native embeddings:",
    reuse_native_count
)
print(
    "New Fixed-2 embeddings:",
    reuse_new_count
)

print(
    "\nTotal:",
    reuse_sentence_count
    + reuse_native_count
    + reuse_new_count
)

print(
    "\nNorms:",
    f"min={norms.min():.7f}, "
    f"mean={norms.mean():.7f}, "
    f"max={norms.max():.7f}"
)

assert (
    reuse_sentence_count
    + reuse_native_count
    + reuse_new_count
) == 18171

assert np.allclose(
    norms,
    1.0,
    atol=1e-5
)

# ------------------------------------------------------------
# Save complete matrix + IDs
# ------------------------------------------------------------

FULL_EMB_PATH = (
    FIXED2_CACHE_DIR /
    "bge_m3_fixed2_embeddings.npy"
)

FULL_ID_PATH = (
    FIXED2_CACHE_DIR /
    "fixed2_chunk_ids.json"
)

np.save(
    FULL_EMB_PATH,
    bge_fixed2_embeddings
)

with open(FULL_ID_PATH, "w") as f:
    json.dump(fixed2_ids, f)

print("\n✓ Complete 18,171 × 1,024 Fixed-2 matrix saved")
print("✓ Chunk IDs saved")
print("✓ All embeddings explicitly L2-normalized")
print("✓ No duplicate text was unnecessarily embedded")

BGE-M3 FIXED-2 — COMPLETE EMBEDDING ASSEMBLY
Shape: (18171, 1024)

Embedding sources:
Sentence embeddings: 4550
Native embeddings: 1946
New Fixed-2 embeddings: 11675

Total: 18171

Norms: min=0.9999999, mean=1.0000000, max=1.0000001

✓ Complete 18,171 × 1,024 Fixed-2 matrix saved
✓ Chunk IDs saved
✓ All embeddings explicitly L2-normalized
✓ No duplicate text was unnecessarily embedded


In [ ]:
# Experiment 002 — BGE-M3
# Cell 14: Build Max–Min chunk corpus and determine embedding reuse

import hashlib
from collections import Counter

MM_CACHE_DIR = BASE / "cache/bge_m3_maxmin_embeddings"
MM_CACHE_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# Load / verify BGE Max–Min segmentation
# ------------------------------------------------------------

bge_mm_seg = bge_maxmin_segmentation

assert len(bge_mm_seg) == 8973
assert sum(r["num_chunks"] for r in bge_mm_seg) == 16763

# ------------------------------------------------------------
# Parent -> frozen sentence records
# ------------------------------------------------------------

parent_sentences = {}

for row in sentence_corpus:
    parent_sentences.setdefault(
        row["parent_id"], []
    ).append(row)

assert len(parent_sentences) == 8973

# ------------------------------------------------------------
# Build BGE Max–Min chunk corpus
# ------------------------------------------------------------

bge_mm_corpus = []

for seg in bge_mm_seg:

    parent_id = seg["parent_id"]
    sentences = parent_sentences[parent_id]

    for chunk in seg["chunks"]:

        indices = chunk["sentence_indices"]

        chunk_text = " ".join(
            sentences[i]["text"]
            for i in indices
        )

        bge_mm_corpus.append({
            "chunk_id": (
                f"{parent_id}_bge_maxmin_{chunk['chunk_idx']}"
            ),
            "parent_id": parent_id,
            "language": seg["language"],
            "chunk_idx": chunk["chunk_idx"],
            "text": chunk_text,
            "num_sentences": chunk["num_sentences"]
        })

assert len(bge_mm_corpus) == 16763

print("=" * 70)
print("BGE-M3 MAX–MIN CORPUS")
print("=" * 70)

print("Total chunks:", len(bge_mm_corpus))

language_counts = Counter(
    row["language"]
    for row in bge_mm_corpus
)

print("\nChunks by language:")
for lang in sorted(language_counts):
    print(f"{lang:10s} {language_counts[lang]}")

# ------------------------------------------------------------
# Hash lookup for existing BGE embeddings
# ------------------------------------------------------------

def text_hash(text):
    return hashlib.sha256(
        text.encode("utf-8")
    ).hexdigest()

# Sentence text hashes
sentence_hash_to_idx = {}

for i, row in enumerate(sentence_corpus):
    h = text_hash(row["text"])
    sentence_hash_to_idx.setdefault(h, []).append(i)

# Native text hashes
native_hash_to_idx = {}

for i, row in enumerate(native_corpus):
    h = text_hash(row["text"])
    native_hash_to_idx.setdefault(h, []).append(i)

# Fixed-2 text hash -> embedding row
fixed2_hash_to_idx = {}

for i, row in enumerate(bge_fixed2_corpus):
    h = text_hash(row["text"])
    fixed2_hash_to_idx.setdefault(h, []).append(i)

# ------------------------------------------------------------
# Determine exact reuse / new-work plan
# ------------------------------------------------------------

reuse_sentence = 0
reuse_native = 0
reuse_fixed2 = 0
needs_new = 0

new_unique_hashes = []
new_unique_set = set()

for row in bge_mm_corpus:

    h = text_hash(row["text"])

    # --------------------------------------------------------
    # Singleton → sentence embedding
    # --------------------------------------------------------
    if (
        row["num_sentences"] == 1
        and h in sentence_hash_to_idx
    ):
        reuse_sentence += 1

    # --------------------------------------------------------
    # Entire passage → native embedding
    # Only applies when this passage stayed as one chunk.
    # --------------------------------------------------------
    elif h in native_hash_to_idx:
        reuse_native += 1

    # --------------------------------------------------------
    # Identical text already embedded for Fixed-2
    # --------------------------------------------------------
    elif h in fixed2_hash_to_idx:
        reuse_fixed2 += 1

    # --------------------------------------------------------
    # Truly new text
    # --------------------------------------------------------
    else:
        needs_new += 1

        if h not in new_unique_set:
            new_unique_set.add(h)
            new_unique_hashes.append(h)

# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

print("\nEmbedding reuse plan:")
print(
    "Sentence embeddings:",
    reuse_sentence
)
print(
    "Native embeddings:",
    reuse_native
)
print(
    "Existing Fixed-2 embeddings:",
    reuse_fixed2
)
print(
    "New chunk instances:",
    needs_new
)
print(
    "New unique chunk texts:",
    len(new_unique_hashes)
)

print(
    "\nTotal accounted for:",
    reuse_sentence
    + reuse_native
    + reuse_fixed2
    + needs_new
)

assert (
    reuse_sentence
    + reuse_native
    + reuse_fixed2
    + needs_new
) == 16763

# Every genuinely new item must be multi-sentence
new_hash_set = set(new_unique_hashes)

for row in bge_mm_corpus:
    if text_hash(row["text"]) in new_hash_set:
        assert row["num_sentences"] >= 2

# ------------------------------------------------------------
# Save frozen corpus + new hash list
# ------------------------------------------------------------

CORPUS_PATH = (
    BASE / "frozen/maxmin_chunk_corpus_bge_m3.json"
)

HASH_PATH = (
    MM_CACHE_DIR / "new_unique_text_hashes.json"
)

with open(CORPUS_PATH, "w") as f:
    json.dump(
        bge_mm_corpus,
        f,
        ensure_ascii=False
    )

with open(HASH_PATH, "w") as f:
    json.dump(new_unique_hashes, f)

print("\n✓ 16,763 BGE Max–Min chunks constructed")
print("✓ Reuse sources identified")
print("✓ New unique-text list saved")
print("✓ No Max–Min chunk embeddings generated yet")

BGE-M3 MAX–MIN CORPUS
Total chunks: 16763

Chunks by language:
bengali    1689
gujarati   1914
hindi      1902
kannada    1940
malayalam  1859
marathi    1894
punjabi    1915
tamil      1845
telugu     1805

Embedding reuse plan:
Sentence embeddings: 6672
Native embeddings: 3173
Existing Fixed-2 embeddings: 4489
New chunk instances: 2429
New unique chunk texts: 2407

Total accounted for: 16763

✓ 16,763 BGE Max–Min chunks constructed
✓ Reuse sources identified
✓ New unique-text list saved
✓ No Max–Min chunk embeddings generated yet


In [ ]:
# Experiment 002 — BGE-M3
# Cell 15: Token-length check for NEW Max–Min texts only

import numpy as np

MM_CACHE_DIR = BASE / "cache/bge_m3_maxmin_embeddings"
HASH_PATH = MM_CACHE_DIR / "new_unique_text_hashes.json"

# Load exact frozen new-text hash list
with open(HASH_PATH, "r") as f:
    mm_new_hashes = json.load(f)

assert len(mm_new_hashes) == 2407

# Build hash -> text from the frozen Max–Min corpus
mm_hash_to_text = {}

for row in bge_mm_corpus:
    h = text_hash(row["text"])

    if h in set(mm_new_hashes):
        mm_hash_to_text[h] = row["text"]

mm_new_texts = [
    mm_hash_to_text[h]
    for h in mm_new_hashes
]

assert len(mm_new_texts) == 2407

# ------------------------------------------------------------
# Token lengths
# ------------------------------------------------------------

mm_new_lengths = []

for i in range(0, len(mm_new_texts), 256):

    batch = mm_new_texts[i:i + 256]

    encoded = tokenizer(
        batch,
        add_special_tokens=True,
        padding=False,
        truncation=False
    )

    mm_new_lengths.extend(
        len(ids)
        for ids in encoded["input_ids"]
    )

mm_new_lengths = np.asarray(mm_new_lengths)

assert len(mm_new_lengths) == 2407

MAX_LEN = 8192

print("=" * 70)
print("BGE-M3 — NEW MAX–MIN TEXT LENGTH CHECK")
print("=" * 70)

print("Unique new texts:", len(mm_new_lengths))
print("Minimum:", mm_new_lengths.min())
print("Mean:", f"{mm_new_lengths.mean():.2f}")
print("Median:", np.median(mm_new_lengths))
print("P90:", np.percentile(mm_new_lengths, 90))
print("P95:", np.percentile(mm_new_lengths, 95))
print("P99:", np.percentile(mm_new_lengths, 99))
print("Maximum:", mm_new_lengths.max())

over_limit = mm_new_lengths > MAX_LEN

print(f"\n> {MAX_LEN} tokens:", over_limit.sum())
print(
    f"% > {MAX_LEN}:",
    f"{over_limit.mean() * 100:.4f}%"
)

assert over_limit.sum() == 0

print("\n✓ 2,407 unique new Max–Min texts verified")
print("✓ No text exceeds BGE-M3's 8,192-token limit")
print("✓ Ready for the final Max–Min embedding workload")

BGE-M3 — NEW MAX–MIN TEXT LENGTH CHECK
Unique new texts: 2407
Minimum: 6
Mean: 68.87
Median: 66.0
P90: 110.0
P95: 124.0
P99: 158.0
Maximum: 1007

> 8192 tokens: 0
% > 8192: 0.0000%

✓ 2,407 unique new Max–Min texts verified
✓ No text exceeds BGE-M3's 8,192-token limit
✓ Ready for the final Max–Min embedding workload


In [ ]:
# Experiment 002 — BGE-M3
# Cell 16: Embed ONLY new unique Max–Min texts

import time
import json
import numpy as np

MM_CACHE_DIR = BASE / "cache/bge_m3_maxmin_embeddings"
NEW_HASHES_PATH = MM_CACHE_DIR / "new_unique_text_hashes.json"
NEW_EMB_PATH = MM_CACHE_DIR / "new_unique_text_embeddings.npy"

# ------------------------------------------------------------
# Load exact hash order
# ------------------------------------------------------------

with open(NEW_HASHES_PATH, "r") as f:
    mm_new_hashes = json.load(f)

assert len(mm_new_hashes) == 2407

# Rebuild hash -> text from frozen Max–Min corpus
mm_hash_to_text = {}

for row in bge_mm_corpus:
    h = text_hash(row["text"])
    if h in set(mm_new_hashes):
        mm_hash_to_text[h] = row["text"]

mm_new_texts = [
    mm_hash_to_text[h]
    for h in mm_new_hashes
]

assert len(mm_new_texts) == 2407

# ------------------------------------------------------------
# Resume protection
# ------------------------------------------------------------

if NEW_EMB_PATH.exists():

    print("Existing BGE-M3 Max–Min new-text cache found.")

    mm_new_embeddings = np.load(NEW_EMB_PATH)

    print("Shape:", mm_new_embeddings.shape)

    assert mm_new_embeddings.shape == (2407, 1024)

    norms = np.linalg.norm(
        mm_new_embeddings,
        axis=1
    )

    print(
        "Norms:",
        f"min={norms.min():.7f}, "
        f"mean={norms.mean():.7f}, "
        f"max={norms.max():.7f}"
    )

    assert np.allclose(norms, 1.0, atol=1e-5)

    print("\n✓ Existing cache verified")
    print("✓ No recomputation needed")

else:

    # --------------------------------------------------------
    # Token lengths in exact hash order
    # --------------------------------------------------------

    lengths = []

    for i in range(0, len(mm_new_texts), 256):

        batch = mm_new_texts[i:i + 256]

        encoded = tokenizer(
            batch,
            add_special_tokens=True,
            padding=False,
            truncation=False
        )

        lengths.extend(
            len(x)
            for x in encoded["input_ids"]
        )

    lengths = np.asarray(lengths)

    assert len(lengths) == 2407
    assert lengths.max() <= MAX_LEN

    # --------------------------------------------------------
    # Length-sorted dynamic batching
    # --------------------------------------------------------

    sorted_indices = np.argsort(lengths)

    def choose_batch_size(max_tokens):

        if max_tokens <= 128:
            return 64
        elif max_tokens <= 256:
            return 32
        elif max_tokens <= 512:
            return 16
        elif max_tokens <= 1024:
            return 8
        else:
            return 4

    start = time.time()

    embeddings_sorted = []
    sorted_original_indices = []

    pos = 0
    total = len(sorted_indices)

    while pos < total:

        first_idx = sorted_indices[pos]

        batch_size = choose_batch_size(
            int(lengths[first_idx])
        )

        end = min(
            pos + batch_size,
            total
        )

        batch_indices = sorted_indices[pos:end]

        batch_texts = [
            mm_new_texts[i]
            for i in batch_indices
        ]

        batch_emb = bge_model.encode(
            batch_texts,
            batch_size=len(batch_texts),
            normalize_embeddings=False,
            convert_to_numpy=True,
            show_progress_bar=False
        ).astype(np.float32)

        # Explicit L2 normalization
        batch_emb /= np.linalg.norm(
            batch_emb,
            axis=1,
            keepdims=True
        )

        embeddings_sorted.append(batch_emb)
        sorted_original_indices.extend(
            batch_indices.tolist()
        )

        pos = end

        if pos % 500 < batch_size or pos == total:
            elapsed = time.time() - start

            print(
                f"{pos:4d}/{total} "
                f"({pos/total*100:6.2f}%) | "
                f"elapsed={elapsed:.2f} sec"
            )

    # --------------------------------------------------------
    # Restore original hash order
    # --------------------------------------------------------

    sorted_embeddings = np.vstack(
        embeddings_sorted
    )

    mm_new_embeddings = np.empty_like(
        sorted_embeddings
    )

    for sorted_pos, original_idx in enumerate(
        sorted_original_indices
    ):
        mm_new_embeddings[original_idx] = (
            sorted_embeddings[sorted_pos]
        )

    # --------------------------------------------------------
    # Integrity checks
    # --------------------------------------------------------

    assert mm_new_embeddings.shape == (
        2407, 1024
    )

    final_norms = np.linalg.norm(
        mm_new_embeddings,
        axis=1
    )

    elapsed = time.time() - start

    print("\nEmbedding complete.")
    print(f"Elapsed: {elapsed:.2f} sec")
    print("Shape:", mm_new_embeddings.shape)

    print(
        "Norms:",
        f"min={final_norms.min():.7f}, "
        f"mean={final_norms.mean():.7f}, "
        f"max={final_norms.max():.7f}"
    )

    assert np.allclose(
        final_norms,
        1.0,
        atol=1e-5
    )

    # --------------------------------------------------------
    # Save
    # --------------------------------------------------------

    np.save(
        NEW_EMB_PATH,
        mm_new_embeddings
    )

    print("\n✓ Saved 2,407 unique new Max–Min embeddings")
    print("✓ Hash order preserved")
    print("✓ Explicit L2 normalization verified")

 512/2407 ( 21.27%) | elapsed=0.68 sec
1024/2407 ( 42.54%) | elapsed=1.71 sec
1536/2407 ( 63.81%) | elapsed=3.13 sec
2048/2407 ( 85.09%) | elapsed=5.21 sec
2407/2407 (100.00%) | elapsed=8.29 sec

Embedding complete.
Elapsed: 8.38 sec
Shape: (2407, 1024)
Norms: min=0.9999999, mean=1.0000000, max=1.0000001

✓ Saved 2,407 unique new Max–Min embeddings
✓ Hash order preserved
✓ Explicit L2 normalization verified


In [ ]:
# Experiment 002 — BGE-M3
# Cell 17: Assemble complete Max–Min embedding matrix

import json
import numpy as np

MM_CACHE_DIR = BASE / "cache/bge_m3_maxmin_embeddings"

NEW_EMB_PATH = (
    MM_CACHE_DIR / "new_unique_text_embeddings.npy"
)
NEW_HASHES_PATH = (
    MM_CACHE_DIR / "new_unique_text_hashes.json"
)

# ------------------------------------------------------------
# Load newly embedded Max–Min texts
# ------------------------------------------------------------

mm_new_embeddings = np.load(NEW_EMB_PATH)

with open(NEW_HASHES_PATH, "r") as f:
    mm_new_hashes = json.load(f)

assert mm_new_embeddings.shape == (2407, 1024)
assert len(mm_new_hashes) == 2407

new_hash_to_idx = {
    h: i
    for i, h in enumerate(mm_new_hashes)
}

# ------------------------------------------------------------
# Build lookup tables for existing BGE embeddings
# ------------------------------------------------------------

# Sentence text hash -> sentence embedding index
sentence_hash_to_idx = {}

for i, row in enumerate(sentence_corpus):
    h = text_hash(row["text"])
    sentence_hash_to_idx.setdefault(h, []).append(i)

# Native text hash -> native embedding index
native_hash_to_idx = {}

for i, row in enumerate(native_corpus):
    h = text_hash(row["text"])
    native_hash_to_idx.setdefault(h, []).append(i)

# Fixed-2 text hash -> Fixed-2 embedding index
fixed2_hash_to_idx = {}

for i, row in enumerate(bge_fixed2_corpus):
    h = text_hash(row["text"])
    fixed2_hash_to_idx.setdefault(h, []).append(i)

# ------------------------------------------------------------
# Assemble
# ------------------------------------------------------------

N = len(bge_mm_corpus)

bge_mm_embeddings = np.empty(
    (N, 1024),
    dtype=np.float32
)

mm_chunk_ids = []

source_sentence = 0
source_native = 0
source_fixed2 = 0
source_new = 0

for i, row in enumerate(bge_mm_corpus):

    h = text_hash(row["text"])

    # --------------------------------------------------------
    # 1. Singleton chunk → cached sentence embedding
    # --------------------------------------------------------
    if (
        row["num_sentences"] == 1
        and h in sentence_hash_to_idx
    ):
        sentence_idx = sentence_hash_to_idx[h][0]

        bge_mm_embeddings[i] = (
            bge_sentence_embeddings[sentence_idx]
        )

        source_sentence += 1

    # --------------------------------------------------------
    # 2. Whole native passage → cached native embedding
    # --------------------------------------------------------
    elif h in native_hash_to_idx:
        native_idx = native_hash_to_idx[h][0]

        bge_mm_embeddings[i] = (
            bge_native_embeddings[native_idx]
        )

        source_native += 1

    # --------------------------------------------------------
    # 3. Identical text already embedded in Fixed-2
    # --------------------------------------------------------
    elif h in fixed2_hash_to_idx:
        fixed2_idx = fixed2_hash_to_idx[h][0]

        bge_mm_embeddings[i] = (
            bge_fixed2_embeddings[fixed2_idx]
        )

        source_fixed2 += 1

    # --------------------------------------------------------
    # 4. Genuinely new Max–Min text
    # --------------------------------------------------------
    else:
        assert h in new_hash_to_idx

        new_idx = new_hash_to_idx[h]

        bge_mm_embeddings[i] = (
            mm_new_embeddings[new_idx]
        )

        source_new += 1

    mm_chunk_ids.append(row["chunk_id"])

# ------------------------------------------------------------
# Integrity checks
# ------------------------------------------------------------

assert bge_mm_embeddings.shape == (
    16763, 1024
)

assert len(mm_chunk_ids) == 16763
assert len(set(mm_chunk_ids)) == 16763

total_sources = (
    source_sentence
    + source_native
    + source_fixed2
    + source_new
)

assert total_sources == 16763

norms = np.linalg.norm(
    bge_mm_embeddings,
    axis=1
)

print("=" * 70)
print("BGE-M3 MAX–MIN — COMPLETE EMBEDDING ASSEMBLY")
print("=" * 70)

print("Shape:", bge_mm_embeddings.shape)

print("\nEmbedding sources:")
print("Sentence embeddings:", source_sentence)
print("Native embeddings:", source_native)
print("Fixed-2 embeddings:", source_fixed2)
print("New Max–Min embeddings:", source_new)

print("\nTotal:", total_sources)

print(
    "\nNorms:",
    f"min={norms.min():.7f}, "
    f"mean={norms.mean():.7f}, "
    f"max={norms.max():.7f}"
)

assert np.allclose(
    norms,
    1.0,
    atol=1e-5
)

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

FULL_EMB_PATH = (
    MM_CACHE_DIR /
    "bge_m3_maxmin_embeddings.npy"
)

FULL_ID_PATH = (
    MM_CACHE_DIR /
    "maxmin_chunk_ids.json"
)

np.save(
    FULL_EMB_PATH,
    bge_mm_embeddings
)

with open(FULL_ID_PATH, "w") as f:
    json.dump(mm_chunk_ids, f)

print("\n✓ Complete 16,763 × 1,024 Max–Min matrix saved")
print("✓ Chunk IDs saved")
print("✓ All embedding norms verified")
print("✓ No unnecessary duplicate embedding performed")

BGE-M3 MAX–MIN — COMPLETE EMBEDDING ASSEMBLY
Shape: (16763, 1024)

Embedding sources:
Sentence embeddings: 6672
Native embeddings: 3173
Fixed-2 embeddings: 4489
New Max–Min embeddings: 2429

Total: 16763

Norms: min=0.9999999, mean=1.0000000, max=1.0000001

✓ Complete 16,763 × 1,024 Max–Min matrix saved
✓ Chunk IDs saved
✓ All embedding norms verified
✓ No unnecessary duplicate embedding performed


In [ ]:
# Experiment 002 — BGE-M3
# Cell 18: Build exact language-restricted retrieval indexes

from collections import defaultdict
import numpy as np

# ------------------------------------------------------------
# Load / verify embedding matrices
# ------------------------------------------------------------

assert bge_native_embeddings.shape == (8973, 1024)
assert bge_fixed2_embeddings.shape == (18171, 1024)
assert bge_mm_embeddings.shape == (16763, 1024)
assert bge_query_embeddings.shape == (459, 1024)

# ------------------------------------------------------------
# Native parent lookup
# ------------------------------------------------------------

native_parent_ids = [
    row["parent_id"]
    for row in native_corpus
]

native_languages = [
    row["language"]
    for row in native_corpus
]

native_lang_indices = defaultdict(list)

for i, lang in enumerate(native_languages):
    native_lang_indices[lang].append(i)

# ------------------------------------------------------------
# Fixed-2 chunk lookup
# ------------------------------------------------------------

fixed2_chunk_ids = [
    row["chunk_id"]
    for row in bge_fixed2_corpus
]

fixed2_parent_ids = [
    row["parent_id"]
    for row in bge_fixed2_corpus
]

fixed2_languages = [
    row["language"]
    for row in bge_fixed2_corpus
]

fixed2_lang_indices = defaultdict(list)

for i, lang in enumerate(fixed2_languages):
    fixed2_lang_indices[lang].append(i)

# ------------------------------------------------------------
# Max–Min chunk lookup
# ------------------------------------------------------------

mm_chunk_ids = [
    row["chunk_id"]
    for row in bge_mm_corpus
]

mm_parent_ids = [
    row["parent_id"]
    for row in bge_mm_corpus
]

mm_languages = [
    row["language"]
    for row in bge_mm_corpus
]

mm_lang_indices = defaultdict(list)

for i, lang in enumerate(mm_languages):
    mm_lang_indices[lang].append(i)

# ------------------------------------------------------------
# Exact relevance mapping
# ------------------------------------------------------------

passage_key_to_parent = {}

for passage in native_corpus:

    key = (
        passage["language"],
        passage["source_query_id"],
        passage["passage_idx"]
    )

    assert key not in passage_key_to_parent

    passage_key_to_parent[key] = passage["parent_id"]

# Verify every frozen benchmark passage maps exactly
missing = []

for q in benchmark:

    for passage_idx in range(len(q["passages"])):

        key = (
            q["language"],
            q["query_id"],
            passage_idx
        )

        if key not in passage_key_to_parent:
            missing.append(key)

assert len(missing) == 0

# ------------------------------------------------------------
# Language-count verification
# ------------------------------------------------------------

languages = sorted(set(q["language"] for q in benchmark))

print("=" * 70)
print("EXPERIMENT 002 — RETRIEVAL INDEX VERIFICATION")
print("=" * 70)

print("\nLanguage candidate counts:")

for lang in languages:

    print(
        f"{lang:10s}"
        f" native={len(native_lang_indices[lang]):4d}"
        f" fixed2={len(fixed2_lang_indices[lang]):4d}"
        f" maxmin={len(mm_lang_indices[lang]):4d}"
    )

# ------------------------------------------------------------
# Sanity checks
# ------------------------------------------------------------

assert all(
    len(native_lang_indices[lang]) == 997
    for lang in languages
)

assert sum(len(v) for v in native_lang_indices.values()) == 8973
assert sum(len(v) for v in fixed2_lang_indices.values()) == 18171
assert sum(len(v) for v in mm_lang_indices.values()) == 16763

print("\nFrozen benchmark passages checked:", sum(
    len(q["passages"])
    for q in benchmark
))

print("Missing exact parent mappings:", len(missing))

print("\n✓ Native index verified")
print("✓ Fixed-2 index verified")
print("✓ Max–Min index verified")
print("✓ Own-language retrieval pools verified")
print("✓ Exact (language, query_id, passage_idx) mapping verified")
print("✓ No retrieval metrics computed yet")

EXPERIMENT 002 — RETRIEVAL INDEX VERIFICATION

Language candidate counts:
bengali    native= 997 fixed2=1875 maxmin=1689
gujarati   native= 997 fixed2=2028 maxmin=1914
hindi      native= 997 fixed2=2038 maxmin=1902
kannada    native= 997 fixed2=2086 maxmin=1940
malayalam  native= 997 fixed2=2021 maxmin=1859
marathi    native= 997 fixed2=2111 maxmin=1894
punjabi    native= 997 fixed2=2029 maxmin=1915
tamil      native= 997 fixed2=1995 maxmin=1845
telugu     native= 997 fixed2=1988 maxmin=1805

Frozen benchmark passages checked: 4563
Missing exact parent mappings: 0

✓ Native index verified
✓ Fixed-2 index verified
✓ Max–Min index verified
✓ Own-language retrieval pools verified
✓ Exact (language, query_id, passage_idx) mapping verified
✓ No retrieval metrics computed yet


In [ ]:
# Experiment 002 — BGE-M3
# Cell 19: Exact retrieval + per-query evaluation

import json
import numpy as np
from sklearn.metrics import ndcg_score

# ------------------------------------------------------------
# Sanity checks
# ------------------------------------------------------------

assert bge_query_embeddings.shape == (459, 1024)
assert bge_native_embeddings.shape == (8973, 1024)
assert bge_fixed2_embeddings.shape == (18171, 1024)
assert bge_mm_embeddings.shape == (16763, 1024)

# Query key -> embedding row
query_key_to_idx = {
    (q["language"], q["query_id"]): i
    for i, q in enumerate(benchmark)
}

assert len(query_key_to_idx) == 459

# ------------------------------------------------------------
# Exact parent-level evaluation
# ------------------------------------------------------------

def evaluate_method(
    name,
    embeddings,
    corpus_parent_ids,
    corpus_languages
):
    per_query = []

    for q in benchmark:

        lang = q["language"]
        qkey = (lang, q["query_id"])
        q_idx = query_key_to_idx[qkey]

        query_vec = bge_query_embeddings[q_idx]

        # Own-language candidate indices only
        lang_indices = [
            i for i, l in enumerate(corpus_languages)
            if l == lang
        ]

        lang_embeddings = embeddings[lang_indices]
        lang_parent_ids = [
            corpus_parent_ids[i]
            for i in lang_indices
        ]

        # Exact cosine because all vectors are normalized
        scores = lang_embeddings @ query_vec

        # ----------------------------------------------------
        # Collapse chunk scores to parent scores
        # parent score = maximum chunk score
        # ----------------------------------------------------

        parent_scores = {}

        for parent_id, score in zip(
            lang_parent_ids,
            scores
        ):
            score = float(score)

            if (
                parent_id not in parent_scores
                or score > parent_scores[parent_id]
            ):
                parent_scores[parent_id] = score

        parent_ids = list(parent_scores.keys())

        parent_score_array = np.asarray(
            [parent_scores[p] for p in parent_ids],
            dtype=np.float32
        )

        # ----------------------------------------------------
        # Exact frozen relevance mapping
        # ----------------------------------------------------

        relevant_parents = set()

        for passage_idx, label in enumerate(
            q["is_selected"]
        ):
            if label == 1:

                key = (
                    lang,
                    q["query_id"],
                    passage_idx
                )

                assert key in passage_key_to_parent

                relevant_parents.add(
                    passage_key_to_parent[key]
                )

        # Binary relevance vector over the complete
        # own-language parent pool
        y_true = np.asarray(
            [
                1.0 if p in relevant_parents else 0.0
                for p in parent_ids
            ],
            dtype=np.float32
        )

        # ----------------------------------------------------
        # nDCG@10
        # ----------------------------------------------------

        ndcg10 = float(
            ndcg_score(
                y_true.reshape(1, -1),
                parent_score_array.reshape(1, -1),
                k=10
            )
        )

        # ----------------------------------------------------
        # Ranked parent list
        # ----------------------------------------------------

        order = np.argsort(
            -parent_score_array,
            kind="mergesort"
        )

        ranked_parents = [
            parent_ids[i]
            for i in order
        ]

        top10 = ranked_parents[:10]

        # ----------------------------------------------------
        # MRR@10 / Recall@1 / Recall@10
        # ----------------------------------------------------

        first_relevant_rank = None

        for rank, parent_id in enumerate(
            top10,
            start=1
        ):
            if parent_id in relevant_parents:
                first_relevant_rank = rank
                break

        if first_relevant_rank is None:
            mrr10 = 0.0
            recall10 = 0.0
        else:
            mrr10 = 1.0 / first_relevant_rank
            recall10 = 1.0

        recall1 = (
            1.0
            if len(top10) >= 1
            and top10[0] in relevant_parents
            else 0.0
        )

        per_query.append({
            "language": lang,
            "query_id": q["query_id"],
            "ndcg10": ndcg10,
            "mrr10": mrr10,
            "recall1": recall1,
            "recall10": recall10
        })

    # --------------------------------------------------------
    # Aggregate
    # --------------------------------------------------------

    summary = {
        "method": name,
        "n_queries": len(per_query),
        "ndcg10": float(
            np.mean([x["ndcg10"] for x in per_query])
        ),
        "mrr10": float(
            np.mean([x["mrr10"] for x in per_query])
        ),
        "recall1": float(
            np.mean([x["recall1"] for x in per_query])
        ),
        "recall10": float(
            np.mean([x["recall10"] for x in per_query])
        ),
        "per_query": per_query
    }

    return summary


# ------------------------------------------------------------
# Run all three BGE conditions
# ------------------------------------------------------------

print("=" * 75)
print("EXPERIMENT 002 — BGE-M3 RETRIEVAL EVALUATION")
print("=" * 75)

results = {}

results["native"] = evaluate_method(
    "BGE-Native",
    bge_native_embeddings,
    native_parent_ids,
    native_languages
)

print("✓ BGE-Native complete")

results["fixed2"] = evaluate_method(
    "BGE-Fixed2",
    bge_fixed2_embeddings,
    fixed2_parent_ids,
    fixed2_languages
)

print("✓ BGE-Fixed2 complete")

results["maxmin"] = evaluate_method(
    "BGE-MaxMin",
    bge_mm_embeddings,
    mm_parent_ids,
    mm_languages
)

print("✓ BGE-Max–Min complete")

# ------------------------------------------------------------
# Print DEV / TEST aggregates
# ------------------------------------------------------------

for split_name in ["dev", "test"]:

    print("\n" + "=" * 75)
    print(f"{split_name.upper()} RESULTS")
    print("=" * 75)

    split_keys = {
        (q["language"], q["query_id"])
        for q in benchmark
        if q["split"] == split_name
    }

    for method_key, result in results.items():

        rows = [
            r for r in result["per_query"]
            if (r["language"], r["query_id"])
            in split_keys
        ]

        print(f"\n{result['method']}")
        print(
            f"nDCG@10   = "
            f"{np.mean([r['ndcg10'] for r in rows]):.6f}"
        )
        print(
            f"MRR@10    = "
            f"{np.mean([r['mrr10'] for r in rows]):.6f}"
        )
        print(
            f"Recall@1  = "
            f"{np.mean([r['recall1'] for r in rows]):.6f}"
        )
        print(
            f"Recall@10 = "
            f"{np.mean([r['recall10'] for r in rows]):.6f}"
        )

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

RESULT_PATH = (
    BASE / "results/bge_m3_results.json"
)

with open(RESULT_PATH, "w") as f:
    json.dump(
        {
            "experiment": "002",
            "embedding_model": "BAAI/bge-m3",
            "maxmin_sigmoid_variant": "sigmoid(|C|-1)",
            "hard_thr": 0.6,
            "init_const": 1.5,
            "c": 0.9,
            "retrieval_scope": "own_language",
            "retrieval": "exact_dense_cosine",
            "parent_collapse": "max_chunk_score",
            "primary_metric": "nDCG@10",
            "results": results
        },
        f,
        ensure_ascii=False
    )

print("\n✓ BGE-M3 results saved:")
print(RESULT_PATH)
print("✓ Per-query results saved for all three methods")
print("✓ No bootstrap performed yet")

EXPERIMENT 002 — BGE-M3 RETRIEVAL EVALUATION
✓ BGE-Native complete
✓ BGE-Fixed2 complete
✓ BGE-Max–Min complete

DEV RESULTS

BGE-Native
nDCG@10   = 0.625900
MRR@10    = 0.546832
Recall@1  = 0.351111
Recall@10 = 0.928889

BGE-Fixed2
nDCG@10   = 0.604444
MRR@10    = 0.522168
Recall@1  = 0.324444
Recall@10 = 0.915556

BGE-MaxMin
nDCG@10   = 0.605003
MRR@10    = 0.522236
Recall@1  = 0.328889
Recall@10 = 0.915556

TEST RESULTS

BGE-Native
nDCG@10   = 0.631728
MRR@10    = 0.552262
Recall@1  = 0.363248
Recall@10 = 0.931624

BGE-Fixed2
nDCG@10   = 0.593838
MRR@10    = 0.502937
Recall@1  = 0.303419
Recall@10 = 0.918803

BGE-MaxMin
nDCG@10   = 0.598438
MRR@10    = 0.510864
Recall@1  = 0.311966
Recall@10 = 0.914530

✓ BGE-M3 results saved:
/content/drive/MyDrive/maxmin_research/results/bge_m3_results.json
✓ Per-query results saved for all three methods
✓ No bootstrap performed yet


In [ ]:
# Experiment 002 — BGE-M3
# Cell 20: Paired bootstrap for BGE-Native / Fixed-2 / Max–Min

import json
import numpy as np
from pathlib import Path

# ------------------------------------------------------------
# Load saved per-query results
# ------------------------------------------------------------

RESULT_PATH = (
    BASE / "results/bge_m3_results.json"
)

with open(RESULT_PATH, "r") as f:
    bge_results = json.load(f)

results = bge_results["results"]

method_rows = {
    "native": results["native"]["per_query"],
    "fixed2": results["fixed2"]["per_query"],
    "maxmin": results["maxmin"]["per_query"],
}

# ------------------------------------------------------------
# Verify query-key alignment
# ------------------------------------------------------------

def key(r):
    return (r["language"], r["query_id"])


key_sets = {
    name: {key(r) for r in rows}
    for name, rows in method_rows.items()
}

assert key_sets["native"] == key_sets["fixed2"]
assert key_sets["native"] == key_sets["maxmin"]
assert len(key_sets["native"]) == 459

print("=" * 75)
print("EXPERIMENT 002 — PAIRED BOOTSTRAP")
print("=" * 75)

print("Paired queries:", len(key_sets["native"]))
print("DEV:", 225)
print("TEST:", 234)

# ------------------------------------------------------------
# Bootstrap function
# ------------------------------------------------------------

RNG_SEED = 20261003
N_BOOT = 10000

def paired_bootstrap(
    rows_a,
    rows_b,
    split,
    metric,
    seed=RNG_SEED,
    n_boot=N_BOOT
):
    a_map = {
        key(r): r
        for r in rows_a
        if r["language"] + "_" + str(r["query_id"])
    }

    b_map = {
        key(r): r
        for r in rows_b
    }

    query_keys = [
        k for k in a_map
        if k[0] in {
            q["language"]
            for q in benchmark
            if q["split"] == split
        }
        and any(
            q["query_id"] == k[1]
            and q["language"] == k[0]
            and q["split"] == split
            for q in benchmark
        )
    ]

    # Stable deterministic ordering
    query_keys = sorted(query_keys)

    assert len(query_keys) == (
        225 if split == "dev" else 234
    )

    observed = np.mean([
        a_map[k][metric] - b_map[k][metric]
        for k in query_keys
    ])

    differences = np.asarray([
        a_map[k][metric] - b_map[k][metric]
        for k in query_keys
    ], dtype=np.float64)

    rng = np.random.default_rng(seed)

    sampled = rng.integers(
        0,
        len(differences),
        size=(n_boot, len(differences))
    )

    bootstrap_means = differences[sampled].mean(axis=1)

    ci_low, ci_high = np.percentile(
        bootstrap_means,
        [2.5, 97.5]
    )

    return {
        "difference": float(observed),
        "ci_low": float(ci_low),
        "ci_high": float(ci_high),
        "n_queries": len(query_keys),
        "n_boot": n_boot,
        "seed": seed
    }


# ------------------------------------------------------------
# Comparisons
# ------------------------------------------------------------

comparisons = {
    "native_minus_maxmin": ("native", "maxmin"),
    "fixed2_minus_maxmin": ("fixed2", "maxmin"),
    "native_minus_fixed2": ("native", "fixed2"),
}

metrics = [
    "ndcg10",
    "mrr10",
    "recall1",
    "recall10",
]

bootstrap_results = {}

for split in ["dev", "test"]:

    bootstrap_results[split] = {}

    print("\n" + "=" * 75)
    print(split.upper())
    print("=" * 75)

    for comparison_name, (a_name, b_name) in comparisons.items():

        bootstrap_results[split][comparison_name] = {}

        print(f"\n{comparison_name}")

        for metric in metrics:

            result = paired_bootstrap(
                method_rows[a_name],
                method_rows[b_name],
                split,
                metric
            )

            bootstrap_results[split][comparison_name][metric] = result

            print(
                f"{metric:10s} "
                f"{result['difference']:+.6f} "
                f"[{result['ci_low']:+.6f}, "
                f"{result['ci_high']:+.6f}]"
            )

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

OUT_PATH = (
    BASE / "results/bge_m3_paired_bootstrap_10000.json"
)

with open(OUT_PATH, "w") as f:
    json.dump(
        {
            "experiment": "002",
            "embedding_model": "BAAI/bge-m3",
            "n_boot": N_BOOT,
            "seed": RNG_SEED,
            "ci": "95_percentile",
            "paired_resampling": True,
            "results": bootstrap_results
        },
        f,
        indent=2
    )

print("\n✓ 10,000 paired bootstrap replicates completed")
print("✓ 95% percentile CIs computed")
print("✓ Results saved:")
print(OUT_PATH)

EXPERIMENT 002 — PAIRED BOOTSTRAP
Paired queries: 459
DEV: 225
TEST: 234

DEV

native_minus_maxmin
ndcg10     +0.020897 [+0.001643, +0.039858]
mrr10      +0.024596 [-0.001073, +0.049868]
recall1    +0.022222 [-0.017778, +0.062222]
recall10   +0.013333 [+0.000000, +0.031111]

fixed2_minus_maxmin
ndcg10     -0.000559 [-0.014941, +0.013757]
mrr10      -0.000069 [-0.018928, +0.018947]
recall1    -0.004444 [-0.031111, +0.022222]
recall10   +0.000000 [-0.013333, +0.013333]

native_minus_fixed2
ndcg10     +0.021457 [+0.000903, +0.042112]
mrr10      +0.024665 [-0.002155, +0.051580]
recall1    +0.026667 [-0.017778, +0.071111]
recall10   +0.013333 [-0.004444, +0.035556]

TEST

native_minus_maxmin
ndcg10     +0.033290 [+0.011371, +0.055359]
mrr10      +0.041399 [+0.012344, +0.070401]
recall1    +0.051282 [+0.004274, +0.098291]
recall10   +0.017094 [+0.004274, +0.034188]

fixed2_minus_maxmin
ndcg10     -0.004600 [-0.021812, +0.011899]
mrr10      -0.007926 [-0.030545, +0.013606]
recall1    -0.00854

In [ ]:
# Experiment 002
# Cell 21 — Per-language TEST performance

import numpy as np
import pandas as pd

test_queries = {
    (q["language"], q["query_id"])
    for q in benchmark
    if q["split"] == "test"
}

rows = []

for lang in languages:

    lang_keys = {
        k for k in test_queries
        if k[0] == lang
    }

    row = {"language": lang}

    for method_key, result in results.items():

        method_map = {
            (r["language"], r["query_id"]): r
            for r in result["per_query"]
        }

        method_rows = [
            method_map[k]
            for k in lang_keys
        ]

        row[f"{method_key}_ndcg10"] = np.mean([
            r["ndcg10"] for r in method_rows
        ])

        row[f"{method_key}_mrr10"] = np.mean([
            r["mrr10"] for r in method_rows
        ])

        row[f"{method_key}_recall10"] = np.mean([
            r["recall10"] for r in method_rows
        ])

    rows.append(row)

lang_df = pd.DataFrame(rows).sort_values("language")

print("=" * 100)
print("EXPERIMENT 002 — TEST RESULTS BY LANGUAGE")
print("=" * 100)

print(
    lang_df[
        [
            "language",
            "native_ndcg10",
            "fixed2_ndcg10",
            "maxmin_ndcg10",
            "native_mrr10",
            "fixed2_mrr10",
            "maxmin_mrr10",
            "native_recall10",
            "fixed2_recall10",
            "maxmin_recall10",
        ]
    ].to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)

print("\n✓ Per-language TEST analysis complete")

EXPERIMENT 002 — TEST RESULTS BY LANGUAGE
 language  native_ndcg10  fixed2_ndcg10  maxmin_ndcg10  native_mrr10  fixed2_mrr10  maxmin_mrr10  native_recall10  fixed2_recall10  maxmin_recall10
  bengali         0.6628         0.6376         0.6460        0.6202        0.5910        0.5892           0.8846           0.8462           0.8846
 gujarati         0.6524         0.6009         0.6290        0.5607        0.4943        0.5305           0.9615           0.9615           0.9615
    hindi         0.6866         0.6309         0.6293        0.6247        0.5408        0.5389           0.9615           0.9615           0.9615
  kannada         0.5286         0.5220         0.4933        0.4037        0.4062        0.3678           0.9231           0.8846           0.8846
malayalam         0.5871         0.5772         0.6045        0.4763        0.4741        0.5147           0.9231           0.8846           0.8846
  marathi         0.6535         0.6073         0.6012        0.5750  

In [ ]:
# Experiment 002
# Cell 22 — Per-language TEST deltas

lang_delta = lang_df.copy()

lang_delta["maxmin_minus_native_ndcg"] = (
    lang_delta["maxmin_ndcg10"]
    - lang_delta["native_ndcg10"]
)

lang_delta["maxmin_minus_fixed2_ndcg"] = (
    lang_delta["maxmin_ndcg10"]
    - lang_delta["fixed2_ndcg10"]
)

lang_delta["maxmin_minus_native_mrr"] = (
    lang_delta["maxmin_mrr10"]
    - lang_delta["native_mrr10"]
)

lang_delta["maxmin_minus_fixed2_mrr"] = (
    lang_delta["maxmin_mrr10"]
    - lang_delta["fixed2_mrr10"]
)

lang_delta["maxmin_minus_native_recall10"] = (
    lang_delta["maxmin_recall10"]
    - lang_delta["native_recall10"]
)

print("=" * 100)
print("EXPERIMENT 002 — TEST DELTAS BY LANGUAGE")
print("=" * 100)

print(
    lang_delta[
        [
            "language",
            "maxmin_minus_native_ndcg",
            "maxmin_minus_fixed2_ndcg",
            "maxmin_minus_native_mrr",
            "maxmin_minus_fixed2_mrr",
            "maxmin_minus_native_recall10",
        ]
    ].sort_values(
        "maxmin_minus_native_ndcg",
        ascending=False
    ).to_string(
        index=False,
        float_format=lambda x: f"{x:+.4f}"
    )
)

print("\n✓ Per-language deltas calculated")

EXPERIMENT 002 — TEST DELTAS BY LANGUAGE
 language  maxmin_minus_native_ndcg  maxmin_minus_fixed2_ndcg  maxmin_minus_native_mrr  maxmin_minus_fixed2_mrr  maxmin_minus_native_recall10
malayalam                   +0.0174                   +0.0273                  +0.0384                  +0.0407                       -0.0385
  bengali                   -0.0169                   +0.0083                  -0.0310                  -0.0018                       +0.0000
  punjabi                   -0.0197                   +0.0305                  -0.0122                  +0.0596                       -0.0385
 gujarati                   -0.0233                   +0.0281                  -0.0302                  +0.0362                       +0.0000
  kannada                   -0.0353                   -0.0287                  -0.0359                  -0.0384                       -0.0385
    tamil                   -0.0403                   -0.0083                  -0.0616                  -0.

In [ ]:
# Experiment 002
# Cell 23 — Per-query ΔnDCG: BGE-Max–Min − BGE-Native

native_map = {
    (r["language"], r["query_id"]): r
    for r in results["native"]["per_query"]
}

maxmin_map = {
    (r["language"], r["query_id"]): r
    for r in results["maxmin"]["per_query"]
}

fixed2_map = {
    (r["language"], r["query_id"]): r
    for r in results["fixed2"]["per_query"]
}

rows = []

for key in sorted(native_map):

    n = native_map[key]
    m = maxmin_map[key]
    f = fixed2_map[key]

    # Determine split from frozen benchmark
    split = next(
        q["split"]
        for q in benchmark
        if q["language"] == key[0]
        and q["query_id"] == key[1]
    )

    rows.append({
        "language": key[0],
        "query_id": key[1],
        "split": split,

        "native_ndcg": n["ndcg10"],
        "maxmin_ndcg": m["ndcg10"],
        "fixed2_ndcg": f["ndcg10"],

        "delta_maxmin_native":
            m["ndcg10"] - n["ndcg10"],

        "delta_maxmin_fixed2":
            m["ndcg10"] - f["ndcg10"],

        "delta_maxmin_native_mrr":
            m["mrr10"] - n["mrr10"],

        "delta_maxmin_native_recall10":
            m["recall10"] - n["recall10"],
    })

bge_pq = pd.DataFrame(rows)

# ------------------------------------------------------------
# Summary function
# ------------------------------------------------------------

def summarize_pq(df, name):

    d = df["delta_maxmin_native"]

    print("\n" + "=" * 75)
    print(name)
    print("=" * 75)

    print("Queries:", len(df))
    print(f"Mean ΔnDCG:        {d.mean():+.6f}")
    print(f"Median ΔnDCG:      {d.median():+.6f}")
    print(f"Std ΔnDCG:         {d.std():.6f}")
    print(f"Max improvement:   {d.max():+.6f}")
    print(f"Max degradation:   {d.min():+.6f}")

    improved = (d > 0).sum()
    unchanged = (d == 0).sum()
    degraded = (d < 0).sum()

    print(
        f"Improved:          {improved} "
        f"({improved/len(df)*100:.1f}%)"
    )
    print(
        f"Unchanged:         {unchanged} "
        f"({unchanged/len(df)*100:.1f}%)"
    )
    print(
        f"Degraded:          {degraded} "
        f"({degraded/len(df)*100:.1f}%)"
    )


summarize_pq(
    bge_pq[bge_pq["split"] == "dev"],
    "DEV — BGE-Max–Min vs BGE-Native"
)

summarize_pq(
    bge_pq[bge_pq["split"] == "test"],
    "TEST — BGE-Max–Min vs BGE-Native"
)

print("\n✓ BGE per-query ΔnDCG analysis complete")


DEV — BGE-Max–Min vs BGE-Native
Queries: 225
Mean ΔnDCG:        -0.020897
Median ΔnDCG:      +0.000000
Std ΔnDCG:         0.144153
Max improvement:   +0.569323
Max degradation:   -0.643793
Improved:          27 (12.0%)
Unchanged:         138 (61.3%)
Degraded:          60 (26.7%)

TEST — BGE-Max–Min vs BGE-Native
Queries: 234
Mean ΔnDCG:        -0.033290
Median ΔnDCG:      +0.000000
Std ΔnDCG:         0.173118
Max improvement:   +0.666667
Max degradation:   -0.643793
Improved:          29 (12.4%)
Unchanged:         148 (63.2%)
Degraded:          57 (24.4%)

✓ BGE per-query ΔnDCG analysis complete


In [ ]:
# Experiment 002
# Cell 24 — Largest BGE-Max–Min gains and degradations

def show_bge_extremes(df, split, n=10):

    part = df[df["split"] == split].copy()

    cols = [
        "language",
        "query_id",
        "native_ndcg",
        "maxmin_ndcg",
        "delta_maxmin_native"
    ]

    print("=" * 80)
    print(f"{split.upper()} — TOP {n} BGE-MAX–MIN IMPROVEMENTS")
    print("=" * 80)

    print(
        part.sort_values(
            "delta_maxmin_native",
            ascending=False
        )[cols]
        .head(n)
        .to_string(index=False)
    )

    print("\n" + "=" * 80)
    print(f"{split.upper()} — TOP {n} BGE-MAX–MIN DEGRADATIONS")
    print("=" * 80)

    print(
        part.sort_values(
            "delta_maxmin_native",
            ascending=True
        )[cols]
        .head(n)
        .to_string(index=False)
    )


show_bge_extremes(bge_pq, "dev")
show_bge_extremes(bge_pq, "test")

print("\n✓ BGE per-query extremes inspected")

DEV — TOP 10 BGE-MAX–MIN IMPROVEMENTS
 language  query_id  native_ndcg  maxmin_ndcg  delta_maxmin_native
    tamil     21860     0.430677      1.00000             0.569323
  kannada    265717     0.500000      1.00000             0.500000
 gujarati    300122     0.500000      1.00000             0.500000
    tamil    300122     0.500000      1.00000             0.500000
    hindi    300122     0.630930      1.00000             0.369070
  marathi     92849     0.630930      1.00000             0.369070
malayalam    229215     0.630930      1.00000             0.369070
  bengali    211441     0.630930      1.00000             0.369070
    tamil    113570     0.430677      0.63093             0.200253
malayalam    131336     0.500000      0.63093             0.130930

DEV — TOP 10 BGE-MAX–MIN DEGRADATIONS
language  query_id  native_ndcg  maxmin_ndcg  delta_maxmin_native
  telugu    116898          1.0     0.356207            -0.643793
  telugu   1060348          1.0     0.430677          

In [ ]:
# Experiment 002
# Cell 25 — Structural features vs BGE-Max–Min ΔnDCG

from scipy.stats import spearmanr
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# Build parent -> BGE sentence records
# ------------------------------------------------------------

parent_sentences_bge = {}

for row in sentence_corpus:
    parent_sentences_bge.setdefault(
        row["parent_id"], []
    ).append(row)

assert len(parent_sentences_bge) == 8973

# ------------------------------------------------------------
# BGE Max–Min segmentation lookup
# ------------------------------------------------------------

parent_bge_seg = {
    row["parent_id"]: row
    for row in bge_maxmin_segmentation
}

assert len(parent_bge_seg) == 8973

# ------------------------------------------------------------
# Sentence embedding lookup
# ------------------------------------------------------------

bge_sentence_id_to_idx = {
    row["sentence_id"]: i
    for i, row in enumerate(sentence_corpus)
}

# ------------------------------------------------------------
# Compute frozen structural features for every parent
# ------------------------------------------------------------

bge_structural = {}

for parent_id in parent_bge_seg:

    seg = parent_bge_seg[parent_id]
    sentences = parent_sentences_bge[parent_id]

    n_sentences = seg["num_sentences"]
    n_chunks = seg["num_chunks"]

    chunk_sizes = [
        c["num_sentences"]
        for c in seg["chunks"]
    ]

    singleton_pct = (
        sum(s == 1 for s in chunk_sizes) / n_chunks
        if n_chunks > 0 else np.nan
    )

    avg_sentences_per_chunk = np.mean(chunk_sizes)

    # Adjacent-sentence BGE cosine statistics
    if n_sentences >= 2:

        vecs = np.vstack([
            bge_sentence_embeddings[
                bge_sentence_id_to_idx[s["sentence_id"]]
            ]
            for s in sentences
        ])

        adjacent_sims = np.sum(
            vecs[:-1] * vecs[1:],
            axis=1
        )

        adjacent_sim_mean = float(
            np.mean(adjacent_sims)
        )

        adjacent_sim_std = float(
            np.std(adjacent_sims)
        )

    else:
        adjacent_sim_mean = np.nan
        adjacent_sim_std = 0.0

    bge_structural[parent_id] = {
        "n_sentences": n_sentences,
        "n_chunks": n_chunks,
        "avg_sentences_per_chunk": avg_sentences_per_chunk,
        "singleton_pct": singleton_pct,
        "was_split": int(n_chunks > 1),
        "adjacent_sim_mean": adjacent_sim_mean,
        "adjacent_sim_std": adjacent_sim_std
    }

print("Structural records:", len(bge_structural))

# ------------------------------------------------------------
# Map each query to its relevant parent passage(s)
# ------------------------------------------------------------

benchmark_map = {
    (q["language"], q["query_id"]): q
    for q in benchmark
}

rows = []

for _, pq in bge_pq.iterrows():

    key = (pq["language"], pq["query_id"])
    q = benchmark_map[key]

    relevant_parents = []

    for passage_idx, label in enumerate(q["is_selected"]):

        if label == 1:

            parent_key = (
                q["language"],
                q["query_id"],
                passage_idx
            )

            assert parent_key in passage_key_to_parent

            relevant_parents.append(
                passage_key_to_parent[parent_key]
            )

    assert len(relevant_parents) > 0

    features = [
        bge_structural[parent_id]
        for parent_id in relevant_parents
    ]

    row = pq.copy()

    row["relevant_passages"] = len(features)

    for feature in [
        "n_sentences",
        "n_chunks",
        "avg_sentences_per_chunk",
        "singleton_pct",
        "was_split",
        "adjacent_sim_mean",
        "adjacent_sim_std"
    ]:

        values = [
            f[feature]
            for f in features
            if not np.isnan(f[feature])
        ]

        row[f"rel_{feature}"] = (
            float(np.mean(values))
            if values else np.nan
        )

    rows.append(row)

bge_struct_pq = pd.DataFrame(rows)

assert len(bge_struct_pq) == 459

# ------------------------------------------------------------
# Correlation analysis
# ------------------------------------------------------------

features = [
    "rel_n_sentences",
    "rel_n_chunks",
    "rel_avg_sentences_per_chunk",
    "rel_singleton_pct",
    "rel_was_split",
    "rel_adjacent_sim_mean",
    "rel_adjacent_sim_std"
]

def correlation_table(df):

    results = []

    for feature in features:

        valid = df[
            [feature, "delta_maxmin_native"]
        ].dropna()

        rho, p = spearmanr(
            valid[feature],
            valid["delta_maxmin_native"]
        )

        results.append({
            "feature": feature,
            "spearman_rho": rho,
            "p_value": p,
            "n": len(valid)
        })

    return pd.DataFrame(results).sort_values(
        "spearman_rho",
        key=lambda x: x.abs(),
        ascending=False
    )

dev_bge_corr = correlation_table(
    bge_struct_pq[
        bge_struct_pq["split"] == "dev"
    ]
)

test_bge_corr = correlation_table(
    bge_struct_pq[
        bge_struct_pq["split"] == "test"
    ]
)

print("\n" + "=" * 75)
print("DEV — BGE STRUCTURE vs ΔnDCG")
print("=" * 75)
print(
    dev_bge_corr.to_string(
        index=False,
        float_format=lambda x: f"{x:.5f}"
    )
)

print("\n" + "=" * 75)
print("TEST — BGE STRUCTURE vs ΔnDCG")
print("=" * 75)
print(
    test_bge_corr.to_string(
        index=False,
        float_format=lambda x: f"{x:.5f}"
    )
)

print("\n✓ BGE structural features computed")
print("✓ Exact parent mapping used")
print("✓ BGE sentence similarities used")
print("✓ No retrieval recomputation")

Structural records: 8973

DEV — BGE STRUCTURE vs ΔnDCG
                    feature  spearman_rho  p_value   n
          rel_singleton_pct      -0.09707  0.14669 225
rel_avg_sentences_per_chunk       0.05393  0.42078 225
            rel_n_sentences       0.05262  0.43222 225
              rel_was_split       0.04700  0.48301 225
               rel_n_chunks       0.03962  0.55436 225
       rel_adjacent_sim_std       0.03220  0.63087 225
      rel_adjacent_sim_mean      -0.00042  0.99551 186

TEST — BGE STRUCTURE vs ΔnDCG
                    feature  spearman_rho  p_value   n
rel_avg_sentences_per_chunk       0.18206  0.00522 234
          rel_singleton_pct      -0.16486  0.01155 234
      rel_adjacent_sim_mean       0.16091  0.02249 201
              rel_was_split      -0.13521  0.03875 234
               rel_n_chunks      -0.10950  0.09470 234
            rel_n_sentences       0.07148  0.27617 234
       rel_adjacent_sim_std      -0.02055  0.75447 234

✓ BGE structural features compute

In [ ]:
# Experiment 002
# Cell 26 — Multiple-testing correction for structural correlations

from statsmodels.stats.multitest import multipletests
import pandas as pd

def add_fdr(corr_df):
    out = corr_df.copy()

    reject, qvals, _, _ = multipletests(
        out["p_value"].values,
        alpha=0.05,
        method="fdr_bh"
    )

    out["q_value_fdr_bh"] = qvals
    out["fdr_significant"] = reject

    # Bonferroni for a stricter reference
    out["bonferroni_p"] = np.minimum(
        out["p_value"] * len(out),
        1.0
    )

    out["bonferroni_significant"] = (
        out["bonferroni_p"] < 0.05
    )

    return out


dev_corr_fdr = add_fdr(dev_bge_corr)
test_corr_fdr = add_fdr(test_bge_corr)

print("=" * 85)
print("DEV — STRUCTURAL CORRELATIONS WITH MULTIPLE-TESTING CORRECTION")
print("=" * 85)

print(
    dev_corr_fdr[
        [
            "feature",
            "spearman_rho",
            "p_value",
            "q_value_fdr_bh",
            "fdr_significant",
            "bonferroni_p",
            "bonferroni_significant"
        ]
    ].to_string(
        index=False,
        float_format=lambda x: f"{x:.5f}"
    )
)

print("\n" + "=" * 85)
print("TEST — STRUCTURAL CORRELATIONS WITH MULTIPLE-TESTING CORRECTION")
print("=" * 85)

print(
    test_corr_fdr[
        [
            "feature",
            "spearman_rho",
            "p_value",
            "q_value_fdr_bh",
            "fdr_significant",
            "bonferroni_p",
            "bonferroni_significant"
        ]
    ].to_string(
        index=False,
        float_format=lambda x: f"{x:.5f}"
    )
)

print("\n✓ FDR and Bonferroni corrections applied")
print("✓ No new retrieval or segmentation performed")

DEV — STRUCTURAL CORRELATIONS WITH MULTIPLE-TESTING CORRECTION
                    feature  spearman_rho  p_value  q_value_fdr_bh  fdr_significant  bonferroni_p  bonferroni_significant
          rel_singleton_pct      -0.09707  0.14669         0.73601            False       1.00000                   False
rel_avg_sentences_per_chunk       0.05393  0.42078         0.73601            False       1.00000                   False
            rel_n_sentences       0.05262  0.43222         0.73601            False       1.00000                   False
              rel_was_split       0.04700  0.48301         0.73601            False       1.00000                   False
               rel_n_chunks       0.03962  0.55436         0.73601            False       1.00000                   False
       rel_adjacent_sim_std       0.03220  0.63087         0.73601            False       1.00000                   False
      rel_adjacent_sim_mean      -0.00042  0.99551         0.99551            False

In [ ]:
# Experiment 002
# Cell 27 — Test the H1 mechanism:
# low adjacent-sentence similarity -> finer Max–Min segmentation

from scipy.stats import spearmanr
import numpy as np
import pandas as pd

# TEST only, because TEST is the final evaluation set
test_struct = bge_struct_pq[
    bge_struct_pq["split"] == "test"
].copy()

# ------------------------------------------------------------
# 1. Semantic cohesion -> singleton proportion
# ------------------------------------------------------------

valid_singleton = test_struct[
    ["rel_adjacent_sim_mean", "rel_singleton_pct"]
].dropna()

rho_singleton, p_singleton = spearmanr(
    valid_singleton["rel_adjacent_sim_mean"],
    valid_singleton["rel_singleton_pct"]
)

# ------------------------------------------------------------
# 2. Semantic cohesion -> average sentences/chunk
# ------------------------------------------------------------

valid_granularity = test_struct[
    ["rel_adjacent_sim_mean",
     "rel_avg_sentences_per_chunk"]
].dropna()

rho_granularity, p_granularity = spearmanr(
    valid_granularity["rel_adjacent_sim_mean"],
    valid_granularity["rel_avg_sentences_per_chunk"]
)

# ------------------------------------------------------------
# 3. Semantic cohesion -> retrieval effect
# ------------------------------------------------------------

valid_delta = test_struct[
    ["rel_adjacent_sim_mean",
     "delta_maxmin_native"]
].dropna()

rho_delta, p_delta = spearmanr(
    valid_delta["rel_adjacent_sim_mean"],
    valid_delta["delta_maxmin_native"]
)

print("=" * 80)
print("H1 MECHANISM — BGE-M3 TEST")
print("=" * 80)

print("\n1. Semantic cohesion vs singleton proportion")
print(
    f"Spearman rho = {rho_singleton:+.5f}"
    f"   p = {p_singleton:.5f}"
    f"   n = {len(valid_singleton)}"
)

print("\n2. Semantic cohesion vs average sentences/chunk")
print(
    f"Spearman rho = {rho_granularity:+.5f}"
    f"   p = {p_granularity:.5f}"
    f"   n = {len(valid_granularity)}"
)

print("\n3. Semantic cohesion vs Max–Min ΔnDCG")
print(
    f"Spearman rho = {rho_delta:+.5f}"
    f"   p = {p_delta:.5f}"
    f"   n = {len(valid_delta)}"
)

print("\nInterpretation:")
print("Higher adjacent similarity = greater semantic cohesion.")
print("Lower sentences/chunk = finer segmentation.")
print("Positive ΔnDCG = Max–Min better than Native.")

print("\n✓ H1 mechanism correlations computed")
print("✓ TEST only")
print("✓ No retrieval or segmentation recomputed")

H1 MECHANISM — BGE-M3 TEST

1. Semantic cohesion vs singleton proportion
Spearman rho = -0.61420   p = 0.00000   n = 201

2. Semantic cohesion vs average sentences/chunk
Spearman rho = +0.66755   p = 0.00000   n = 201

3. Semantic cohesion vs Max–Min ΔnDCG
Spearman rho = +0.16091   p = 0.02249   n = 201

Interpretation:
Higher adjacent similarity = greater semantic cohesion.
Lower sentences/chunk = finer segmentation.
Positive ΔnDCG = Max–Min better than Native.

✓ H1 mechanism correlations computed
✓ TEST only
✓ No retrieval or segmentation recomputed


In [ ]:
# Context Fragmentation Analysis
# Cell 28 — Reload saved 002 artifacts + original Answer data

import json
import numpy as np
from pathlib import Path

from google.colab import drive
drive.mount("/content/drive")

BASE = Path("/content/drive/MyDrive/maxmin_research")

languages = [
    "hindi",
    "bengali",
    "tamil",
    "telugu",
    "malayalam",
    "kannada",
    "marathi",
    "gujarati",
    "punjabi"
]

# ------------------------------------------------------------
# Load frozen 002 artifacts
# ------------------------------------------------------------

with open(BASE / "frozen/msmarco_xi_frozen_benchmark.json", "r") as f:
    benchmark = json.load(f)

with open(BASE / "frozen/msmarco_xi_sentence_corpus.json", "r") as f:
    sentence_corpus = json.load(f)

with open(BASE / "frozen/maxmin_segmentation_bge_m3.json", "r") as f:
    bge_maxmin_segmentation = json.load(f)["segmentation"]

with open(BASE / "frozen/msmarco_xi_native_corpus.json", "r") as f:
    native_corpus = json.load(f)

with open(BASE / "results/bge_m3_results.json", "r") as f:
    bge_results = json.load(f)

# ------------------------------------------------------------
# Load original 100-query files containing Answer
# ------------------------------------------------------------

source_records = []

for lang in languages:

    path = BASE / "frozen" / f"msmarco_xi_{lang}_100.json"

    if not path.exists():
        raise FileNotFoundError(path)

    with open(path, "r") as f:
        source_records.extend(json.load(f))

# ------------------------------------------------------------
# Build exact query -> Answer lookup
# ------------------------------------------------------------

answer_lookup = {}

for r in source_records:

    key = (
        r["source_lang"].lower(),
        r["query_id"]
    )

    answer_lookup[key] = r["Answer"]

# ------------------------------------------------------------
# Verify all frozen benchmark queries have Answers
# ------------------------------------------------------------

missing = []

for q in benchmark:

    key = (
        q["language"],
        q["query_id"]
    )

    if key not in answer_lookup:
        missing.append(key)

assert len(benchmark) == 459
assert len(bge_maxmin_segmentation) == 8973
assert len(missing) == 0

# ------------------------------------------------------------
# Build parent/sentence lookups
# ------------------------------------------------------------

parent_sentences = {}

for row in sentence_corpus:
    parent_sentences.setdefault(
        row["parent_id"], []
    ).append(row)

parent_segmentation = {
    row["parent_id"]: row
    for row in bge_maxmin_segmentation
}

# ------------------------------------------------------------
# Basic report
# ------------------------------------------------------------

print("=" * 75)
print("CELL 28 — CONTEXT FRAGMENTATION SETUP")
print("=" * 75)

print("Benchmark queries:", len(benchmark))
print("Sentence records:", len(sentence_corpus))
print("Max–Min segmentation records:", len(bge_maxmin_segmentation))
print("Original source records:", len(source_records))
print("Queries with Answer:", len(answer_lookup))

print("\nExample Answer:")
example_q = benchmark[0]
example_key = (
    example_q["language"],
    example_q["query_id"]
)

print("Language:", example_q["language"])
print("Query ID:", example_q["query_id"])
print("Query:", example_q["query"])
print("Answer:", answer_lookup[example_key])

print("\n✓ All 459 frozen queries matched to original Answer data")
print("✓ BGE Max–Min segmentation restored")
print("✓ Sentence corpus restored")
print("✓ No embeddings or retrieval recomputed")
print("✓ Ready to identify answer-relevant sentences")

Mounted at /content/drive


FileNotFoundError: /content/drive/MyDrive/maxmin_research/frozen/msmarco_xi_bengali_100.json

In [ ]:
# Cell 29 — Inspect Drive for the original 100-query source files

from pathlib import Path

BASE = Path("/content/drive/MyDrive/maxmin_research")

print("=" * 75)
print("CELL 29 — LOCATING ORIGINAL SOURCE FILES")
print("=" * 75)

# Show JSON files directly under frozen/
frozen_files = sorted(
    p.name
    for p in (BASE / "frozen").glob("*.json")
)

print("\nJSON files in frozen/:")
for name in frozen_files:
    print(" ", name)

# Also check the main experiment directory for possible source files
root_files = sorted(
    p.name
    for p in BASE.glob("*.json")
)

print("\nJSON files in maxmin_research/:")
for name in root_files:
    print(" ", name)

# Search recursively for filenames containing '100'
matches_100 = sorted(
    str(p.relative_to(BASE))
    for p in BASE.rglob("*.json")
    if "100" in p.name.lower()
)

print("\nJSON files containing '100' anywhere under maxmin_research/:")
for name in matches_100:
    print(" ", name)

print("\n✓ No files modified")
print("✓ No embeddings or retrieval recomputed")

CELL 29 — LOCATING ORIGINAL SOURCE FILES

JSON files in frozen/:
  experiment_001_manifest.json
  fixed2_chunk_corpus_bge_m3.json
  maxmin_chunk_corpus_bge_m3.json
  maxmin_chunk_corpus_qwen_official.json
  maxmin_segmentation_bge_m3.json
  maxmin_segmentation_qwen_official.json
  msmarco_xi_ben_100.json
  msmarco_xi_frozen_benchmark.json
  msmarco_xi_guj_100.json
  msmarco_xi_hindi_100.json
  msmarco_xi_kan_100.json
  msmarco_xi_mal_100.json
  msmarco_xi_mar_100.json
  msmarco_xi_native_corpus.json
  msmarco_xi_pan_100.json
  msmarco_xi_sentence_corpus.json
  msmarco_xi_tam_100.json
  msmarco_xi_tel_100.json

JSON files in maxmin_research/:

JSON files containing '100' anywhere under maxmin_research/:
  frozen/msmarco_xi_ben_100.json
  frozen/msmarco_xi_guj_100.json
  frozen/msmarco_xi_hindi_100.json
  frozen/msmarco_xi_kan_100.json
  frozen/msmarco_xi_mal_100.json
  frozen/msmarco_xi_mar_100.json
  frozen/msmarco_xi_pan_100.json
  frozen/msmarco_xi_tam_100.json
  frozen/msmarco_xi_te

In [ ]:
# Cell 30 — Load original MSMARCO-XI source records correctly

import json
from pathlib import Path

BASE = Path("/content/drive/MyDrive/maxmin_research")
FROZEN = BASE / "frozen"

# ------------------------------------------------------------
# Correct frozen source-file names
# ------------------------------------------------------------

source_files = {
    "bengali": "msmarco_xi_ben_100.json",
    "gujarati": "msmarco_xi_guj_100.json",
    "hindi": "msmarco_xi_hindi_100.json",
    "kannada": "msmarco_xi_kan_100.json",
    "malayalam": "msmarco_xi_mal_100.json",
    "marathi": "msmarco_xi_mar_100.json",
    "punjabi": "msmarco_xi_pan_100.json",
    "tamil": "msmarco_xi_tam_100.json",
    "telugu": "msmarco_xi_tel_100.json",
}

source_records_by_lang = {}

for lang, filename in source_files.items():

    path = FROZEN / filename

    assert path.exists(), f"Missing file: {path}"

    with open(path, "r") as f:
        records = json.load(f)

    source_records_by_lang[lang] = records

    print(
        f"{lang:10s} "
        f"records={len(records):3d} "
        f"source_lang_values={sorted(set(
            str(r.get('source_lang'))
            for r in records
        ))}"
    )

# ------------------------------------------------------------
# Flatten records
# ------------------------------------------------------------

source_records = []

for lang in source_records_by_lang:
    source_records.extend(
        source_records_by_lang[lang]
    )

print("\nTotal source records:", len(source_records))

# ------------------------------------------------------------
# Inspect Answer structure
# ------------------------------------------------------------

example = source_records[0]

print("\nExample source record fields:")
print(list(example.keys()))

print("\nExample:")
print("source_lang:", example.get("source_lang"))
print("query_id:", example.get("query_id"))
print("query:", example.get("query"))
print("Answer type:", type(example.get("Answer")))
print("Answer:", example.get("Answer"))

# ------------------------------------------------------------
# Basic expected-count check
# ------------------------------------------------------------

assert len(source_records) == 900

for lang, records in source_records_by_lang.items():
    assert len(records) == 100

print("\n✓ All 9 original 100-query files loaded")
print("✓ 900 source records available")
print("✓ Answer field inspected")
print("✓ No embeddings or retrieval recomputed")

bengali    records=100 source_lang_values=['None']
gujarati   records=100 source_lang_values=['None']
hindi      records=100 source_lang_values=['None']
kannada    records=100 source_lang_values=['None']
malayalam  records=100 source_lang_values=['None']
marathi    records=100 source_lang_values=['None']
punjabi    records=100 source_lang_values=['None']
tamil      records=100 source_lang_values=['None']
telugu     records=100 source_lang_values=['None']

Total source records: 900

Example source record fields:
['query_id', 'query', 'passages', 'is_selected']

Example:
source_lang: None
query_id: 1102432
query: কর্পোরেশন কী?
Answer type: <class 'NoneType'>
Answer: None

✓ All 9 original 100-query files loaded
✓ 900 source records available
✓ Answer field inspected
✓ No embeddings or retrieval recomputed


In [ ]:
# Cell 31 — Reload saved BGE artifacts for query-evidence fragmentation

import json
import numpy as np
from pathlib import Path

BASE = Path("/content/drive/MyDrive/maxmin_research")

# ------------------------------------------------------------
# Load frozen benchmark / sentence corpus / BGE Max–Min
# ------------------------------------------------------------

with open(BASE / "frozen/msmarco_xi_frozen_benchmark.json", "r") as f:
    benchmark = json.load(f)

with open(BASE / "frozen/msmarco_xi_sentence_corpus.json", "r") as f:
    sentence_corpus = json.load(f)

with open(BASE / "frozen/maxmin_segmentation_bge_m3.json", "r") as f:
    bge_segmentation = json.load(f)["segmentation"]

# ------------------------------------------------------------
# Load saved BGE embeddings
# ------------------------------------------------------------

bge_sentence_embeddings = np.load(
    BASE / "cache/bge_m3_sentence_embeddings/"
    "bge_m3_sentence_embeddings.npy"
)

bge_query_embeddings = np.load(
    BASE / "cache/bge_m3_query_embeddings/"
    "bge_m3_query_embeddings.npy"
)

# ------------------------------------------------------------
# Basic integrity checks
# ------------------------------------------------------------

assert len(benchmark) == 459
assert len(sentence_corpus) == 31792
assert len(bge_segmentation) == 8973
assert bge_sentence_embeddings.shape == (31792, 1024)
assert bge_query_embeddings.shape == (459, 1024)

# ------------------------------------------------------------
# Build lookups
# ------------------------------------------------------------

sentence_id_to_idx = {
    row["sentence_id"]: i
    for i, row in enumerate(sentence_corpus)
}

parent_sentences = {}

for row in sentence_corpus:
    parent_sentences.setdefault(
        row["parent_id"], []
    ).append(row)

parent_segmentation = {
    row["parent_id"]: row
    for row in bge_segmentation
}

query_key_to_idx = {
    (q["language"], q["query_id"]): i
    for i, q in enumerate(benchmark)
}

# ------------------------------------------------------------
# Verify alignment
# ------------------------------------------------------------

assert len(parent_sentences) == 8973
assert len(parent_segmentation) == 8973
assert len(query_key_to_idx) == 459

print("=" * 75)
print("CELL 31 — QUERY-EVIDENCE FRAGMENTATION SETUP")
print("=" * 75)

print("Frozen queries:", len(benchmark))
print("Sentence embeddings:", bge_sentence_embeddings.shape)
print("Query embeddings:", bge_query_embeddings.shape)
print("BGE Max–Min passages:", len(parent_segmentation))

print("\nImportant:")
print("The source benchmark contains no Answer field.")
print("Therefore this analysis uses query-semantic evidence")
print("as a proxy for answer-bearing evidence.")

print("\n✓ All saved BGE artifacts restored")
print("✓ No embeddings recomputed")
print("✓ No retrieval recomputed")
print("✓ Ready to test query-evidence fragmentation")

CELL 31 — QUERY-EVIDENCE FRAGMENTATION SETUP
Frozen queries: 459
Sentence embeddings: (31792, 1024)
Query embeddings: (459, 1024)
BGE Max–Min passages: 8973

Important:
The source benchmark contains no Answer field.
Therefore this analysis uses query-semantic evidence
as a proxy for answer-bearing evidence.

✓ All saved BGE artifacts restored
✓ No embeddings recomputed
✓ No retrieval recomputed
✓ Ready to test query-evidence fragmentation


In [ ]:
# Cell 32 — Query-evidence fragmentation proxy
#
# Definition:
# For each relevant passage, identify the 2 sentences with
# highest BGE cosine similarity to the query.
# If those 2 sentences fall into different Max–Min chunks,
# mark the passage as fragmented.

import json
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# Load exact native parent mapping + BGE results
# ------------------------------------------------------------

with open(BASE / "frozen/msmarco_xi_native_corpus.json", "r") as f:
    native_corpus = json.load(f)

with open(BASE / "results/bge_m3_results.json", "r") as f:
    bge_results = json.load(f)

# Exact frozen relevance mapping
passage_key_to_parent = {}

for passage in native_corpus:
    key = (
        passage["language"],
        passage["source_query_id"],
        passage["passage_idx"]
    )

    assert key not in passage_key_to_parent
    passage_key_to_parent[key] = passage["parent_id"]

# ------------------------------------------------------------
# Build query-level Max–Min vs Native delta
# ------------------------------------------------------------

native_map = {
    (r["language"], r["query_id"]): r
    for r in bge_results["results"]["native"]["per_query"]
}

maxmin_map = {
    (r["language"], r["query_id"]): r
    for r in bge_results["results"]["maxmin"]["per_query"]
}

# ------------------------------------------------------------
# Measure fragmentation
# ------------------------------------------------------------

fragmentation_rows = []

for q in benchmark:

    qkey = (q["language"], q["query_id"])
    q_idx = query_key_to_idx[qkey]

    query_vec = bge_query_embeddings[q_idx]

    relevant_passage_results = []

    for passage_idx, label in enumerate(q["is_selected"]):

        if label != 1:
            continue

        parent_key = (
            q["language"],
            q["query_id"],
            passage_idx
        )

        assert parent_key in passage_key_to_parent

        parent_id = passage_key_to_parent[parent_key]

        sentences = parent_sentences[parent_id]
        seg = parent_segmentation[parent_id]

        # ----------------------------------------------------
        # Sentence-level query similarity
        # ----------------------------------------------------

        sentence_vectors = np.vstack([
            bge_sentence_embeddings[
                sentence_id_to_idx[s["sentence_id"]]
            ]
            for s in sentences
        ])

        sentence_scores = sentence_vectors @ query_vec

        # Top 2 query-similar sentences
        top_k = min(2, len(sentence_scores))

        top_indices = np.argsort(
            -sentence_scores,
            kind="mergesort"
        )[:top_k]

        # ----------------------------------------------------
        # Map sentence index -> Max–Min chunk index
        # ----------------------------------------------------

        sentence_to_chunk = {}

        for chunk in seg["chunks"]:
            for sentence_idx in chunk["sentence_indices"]:
                sentence_to_chunk[sentence_idx] = chunk["chunk_idx"]

        top_chunk_ids = [
            sentence_to_chunk[int(idx)]
            for idx in top_indices
        ]

        fragmented = (
            len(set(top_chunk_ids)) > 1
            if len(top_chunk_ids) >= 2
            else False
        )

        relevant_passage_results.append({
            "parent_id": parent_id,
            "num_sentences": len(sentences),
            "num_chunks": seg["num_chunks"],
            "top_sentence_indices": [
                int(x) for x in top_indices
            ],
            "top_sentence_scores": [
                float(sentence_scores[x])
                for x in top_indices
            ],
            "top_chunk_ids": top_chunk_ids,
            "fragmented": int(fragmented)
        })

    # --------------------------------------------------------
    # Aggregate relevant passages for this query
    # --------------------------------------------------------

    fragment_rate = np.mean([
        r["fragmented"]
        for r in relevant_passage_results
    ])

    delta_ndcg = (
        maxmin_map[qkey]["ndcg10"]
        - native_map[qkey]["ndcg10"]
    )

    fragmentation_rows.append({
        "language": q["language"],
        "query_id": q["query_id"],
        "split": q["split"],
        "delta_ndcg": delta_ndcg,
        "relevant_passages": len(
            relevant_passage_results
        ),
        "fragmented_relevant_passages": sum(
            r["fragmented"]
            for r in relevant_passage_results
        ),
        "fragmentation_rate": fragment_rate
    })

fragmentation_df = pd.DataFrame(
    fragmentation_rows
)

assert len(fragmentation_df) == 459

print("=" * 80)
print("CELL 32 — QUERY-EVIDENCE FRAGMENTATION")
print("=" * 80)

print(
    "Queries:", len(fragmentation_df)
)

print(
    "\nRelevant passages whose top-2 query-evidence "
    "sentences are split across Max–Min chunks:"
)

print(
    f"{(fragmentation_df['fragmentation_rate'] > 0).sum()} / 459 "
    f"queries "
    f"({(fragmentation_df['fragmentation_rate'] > 0).mean()*100:.1f}%)"
)

print(
    "\nOverall relevant-passage fragmentation rate:",
    f"{fragmentation_df['fragmentation_rate'].mean()*100:.2f}%"
)

print("\nExample rows:")
print(
    fragmentation_df.head(10).to_string(
        index=False
    )
)

print(
    "\n✓ Exact parent mapping used"
)
print(
    "✓ BGE query/sentence similarities used"
)
print(
    "✓ Max–Min chunk assignments used"
)
print(
    "✓ No retrieval or embedding recomputed"
)

CELL 32 — QUERY-EVIDENCE FRAGMENTATION
Queries: 459

Relevant passages whose top-2 query-evidence sentences are split across Max–Min chunks:
135 / 459 queries (29.4%)

Overall relevant-passage fragmentation rate: 27.45%

Example rows:
language  query_id split  delta_ndcg  relevant_passages  fragmented_relevant_passages  fragmentation_rate
   hindi   1090350   dev     0.00000                  1                             0                 0.0
   hindi    158720   dev     0.00000                  1                             1                 1.0
   hindi    373460   dev    -0.50000                  1                             1                 1.0
   hindi    113570   dev     0.00000                  1                             0                 0.0
   hindi    233826   dev     0.00000                  1                             0                 0.0
   hindi    265717   dev     0.13093                  1                             1                 1.0
   hindi    190327   de

In [ ]:
# Cell 33 — Test whether query-evidence fragmentation is associated
# with Max–Min retrieval degradation

from scipy.stats import spearmanr
import numpy as np
import pandas as pd

df = fragmentation_df.copy()

# ------------------------------------------------------------
# Continuous relationship
# fragmentation_rate -> ΔnDCG
# ------------------------------------------------------------

valid = df[
    ["fragmentation_rate", "delta_ndcg"]
].dropna()

rho, p = spearmanr(
    valid["fragmentation_rate"],
    valid["delta_ndcg"]
)

print("=" * 80)
print("CELL 33 — FRAGMENTATION vs RETRIEVAL EFFECT")
print("=" * 80)

print("\nContinuous relationship:")
print(
    f"Spearman rho = {rho:+.5f}"
    f"   p = {p:.5f}"
    f"   n = {len(valid)}"
)

# ------------------------------------------------------------
# Binary comparison:
# any fragmentation vs no fragmentation
# ------------------------------------------------------------

fragmented = df[
    df["fragmentation_rate"] > 0
]["delta_ndcg"]

not_fragmented = df[
    df["fragmentation_rate"] == 0
]["delta_ndcg"]

print("\nQueries WITH fragmentation:")
print("n:", len(fragmented))
print("Mean ΔnDCG:", f"{fragmented.mean():+.6f}")
print("Median ΔnDCG:", f"{fragmented.median():+.6f}")

print("\nQueries WITHOUT fragmentation:")
print("n:", len(not_fragmented))
print("Mean ΔnDCG:", f"{not_fragmented.mean():+.6f}")
print("Median ΔnDCG:", f"{not_fragmented.median():+.6f}")

print(
    "\nMean difference "
    "(fragmented − non-fragmented):",
    f"{fragmented.mean() - not_fragmented.mean():+.6f}"
)

# ------------------------------------------------------------
# Split by TEST only
# ------------------------------------------------------------

test_df = df[df["split"] == "test"].copy()

test_fragmented = test_df[
    test_df["fragmentation_rate"] > 0
]["delta_ndcg"]

test_not_fragmented = test_df[
    test_df["fragmentation_rate"] == 0
]["delta_ndcg"]

print("\n" + "=" * 80)
print("TEST ONLY")
print("=" * 80)

print(
    "Fragmented queries:",
    len(test_fragmented),
    f"mean ΔnDCG={test_fragmented.mean():+.6f}"
)

print(
    "Non-fragmented queries:",
    len(test_not_fragmented),
    f"mean ΔnDCG={test_not_fragmented.mean():+.6f}"
)

print(
    "Difference:",
    f"{test_fragmented.mean() - test_not_fragmented.mean():+.6f}"
)

print("\n✓ Fragmentation effect analyzed")
print("✓ No embeddings or retrieval recomputed")

CELL 33 — FRAGMENTATION vs RETRIEVAL EFFECT

Continuous relationship:
Spearman rho = -0.06488   p = 0.16525   n = 459

Queries WITH fragmentation:
n: 135
Mean ΔnDCG: -0.040697
Median ΔnDCG: +0.000000

Queries WITHOUT fragmentation:
n: 324
Mean ΔnDCG: -0.021598
Median ΔnDCG: +0.000000

Mean difference (fragmented − non-fragmented): -0.019099

TEST ONLY
Fragmented queries: 76 mean ΔnDCG=-0.066547
Non-fragmented queries: 158 mean ΔnDCG=-0.017293
Difference: -0.049254

✓ Fragmentation effect analyzed
✓ No embeddings or retrieval recomputed


In [ ]:
# Cell 34 — Statistical test:
# Does query-evidence fragmentation predict lower ΔnDCG?

import numpy as np
from scipy.stats import permutation_test, mannwhitneyu

test_df = fragmentation_df[
    fragmentation_df["split"] == "test"
].copy()

fragmented = test_df[
    test_df["fragmentation_rate"] > 0
]["delta_ndcg"].to_numpy()

non_fragmented = test_df[
    test_df["fragmentation_rate"] == 0
]["delta_ndcg"].to_numpy()

# ------------------------------------------------------------
# Observed difference
# ------------------------------------------------------------

observed_diff = (
    fragmented.mean()
    - non_fragmented.mean()
)

# ------------------------------------------------------------
# Permutation test
# H0: fragmentation status does not affect ΔnDCG
# Two-sided test, exact group sizes preserved.
# ------------------------------------------------------------

rng = np.random.default_rng(20261003)

all_values = np.concatenate([
    fragmented,
    non_fragmented
])

n_frag = len(fragmented)
n_total = len(all_values)

n_permutations = 10000

perm_diffs = np.empty(n_permutations)

for i in range(n_permutations):

    shuffled = rng.permutation(all_values)

    perm_diffs[i] = (
        shuffled[:n_frag].mean()
        - shuffled[n_frag:].mean()
    )

p_perm = (
    np.mean(
        np.abs(perm_diffs) >= abs(observed_diff)
    )
)

# ------------------------------------------------------------
# Bootstrap 95% CI for mean difference
# Fragmented − non-fragmented
# ------------------------------------------------------------

bootstrap_diffs = np.empty(n_permutations)

for i in range(n_permutations):

    frag_sample = rng.choice(
        fragmented,
        size=len(fragmented),
        replace=True
    )

    nonfrag_sample = rng.choice(
        non_fragmented,
        size=len(non_fragmented),
        replace=True
    )

    bootstrap_diffs[i] = (
        frag_sample.mean()
        - nonfrag_sample.mean()
    )

ci_low, ci_high = np.percentile(
    bootstrap_diffs,
    [2.5, 97.5]
)

# ------------------------------------------------------------
# Mann-Whitney U as a distributional robustness check
# ------------------------------------------------------------

mw = mannwhitneyu(
    fragmented,
    non_fragmented,
    alternative="two-sided"
)

# ------------------------------------------------------------
# Report
# ------------------------------------------------------------

print("=" * 80)
print("CELL 34 — TEST: FRAGMENTATION EFFECT")
print("=" * 80)

print("\nGroups:")
print("Fragmented:", len(fragmented))
print("Non-fragmented:", len(non_fragmented))

print(
    "\nObserved mean difference "
    "(fragmented − non-fragmented):",
    f"{observed_diff:+.6f}"
)

print(
    "95% bootstrap CI:",
    f"[{ci_low:+.6f}, {ci_high:+.6f}]"
)

print(
    "\nPermutation p-value:",
    f"{p_perm:.6f}"
)

print(
    "\nMann–Whitney U p-value:",
    f"{mw.pvalue:.6f}"
)

print("\n✓ TEST-only statistical comparison complete")
print("✓ No embeddings or retrieval recomputed")

CELL 34 — TEST: FRAGMENTATION EFFECT

Groups:
Fragmented: 76
Non-fragmented: 158

Observed mean difference (fragmented − non-fragmented): -0.049254
95% bootstrap CI: [-0.108017, +0.007612]

Permutation p-value: 0.037500

Mann–Whitney U p-value: 0.011122

✓ TEST-only statistical comparison complete
✓ No embeddings or retrieval recomputed


In [ ]:
##2nd hypothesis testing

In [ ]:
# Cell 35 — Reload saved artifacts needed for H2

import json
import numpy as np
from pathlib import Path

BASE = Path("/content/drive/MyDrive/maxmin_research")

# ------------------------------------------------------------
# Load saved BGE Max–Min chunk corpus
# ------------------------------------------------------------

with open(
    BASE / "frozen/maxmin_chunk_corpus_bge_m3.json",
    "r"
) as f:
    bge_mm_corpus = json.load(f)

# ------------------------------------------------------------
# Load frozen benchmark
# ------------------------------------------------------------

with open(
    BASE / "frozen/msmarco_xi_frozen_benchmark.json",
    "r"
) as f:
    benchmark = json.load(f)

# ------------------------------------------------------------
# Load saved BGE per-query results
# ------------------------------------------------------------

with open(
    BASE / "results/bge_m3_results.json",
    "r"
) as f:
    bge_results = json.load(f)

# ------------------------------------------------------------
# Load frozen native corpus for exact parent mapping
# ------------------------------------------------------------

with open(
    BASE / "frozen/msmarco_xi_native_corpus.json",
    "r"
) as f:
    native_corpus = json.load(f)

# ------------------------------------------------------------
# Build exact parent mapping
# ------------------------------------------------------------

passage_key_to_parent = {}

for passage in native_corpus:

    key = (
        passage["language"],
        passage["source_query_id"],
        passage["passage_idx"]
    )

    assert key not in passage_key_to_parent

    passage_key_to_parent[key] = passage["parent_id"]

# ------------------------------------------------------------
# Build query lookup
# ------------------------------------------------------------

benchmark_map = {
    (q["language"], q["query_id"]): q
    for q in benchmark
}

native_map = {
    (r["language"], r["query_id"]): r
    for r in bge_results["results"]["native"]["per_query"]
}

maxmin_map = {
    (r["language"], r["query_id"]): r
    for r in bge_results["results"]["maxmin"]["per_query"]
}

# ------------------------------------------------------------
# Verify
# ------------------------------------------------------------

assert len(bge_mm_corpus) == 16763
assert len(benchmark) == 459
assert len(native_map) == 459
assert len(maxmin_map) == 459
assert len(passage_key_to_parent) == 8973

print("=" * 75)
print("CELL 35 — H2 INPUTS RELOADED")
print("=" * 75)

print("BGE Max–Min chunks:", len(bge_mm_corpus))
print("Benchmark queries:", len(benchmark))
print("Native per-query results:", len(native_map))
print("Max–Min per-query results:", len(maxmin_map))
print("Exact parent mappings:", len(passage_key_to_parent))

print("\n✓ Saved BGE Max–Min corpus restored")
print("✓ Frozen benchmark restored")
print("✓ Saved retrieval results restored")
print("✓ Exact parent mapping restored")
print("✓ No embeddings or retrieval recomputed")
print("✓ Ready for H2 analysis")

CELL 35 — H2 INPUTS RELOADED
BGE Max–Min chunks: 16763
Benchmark queries: 459
Native per-query results: 459
Max–Min per-query results: 459
Exact parent mappings: 8973

✓ Saved BGE Max–Min corpus restored
✓ Frozen benchmark restored
✓ Saved retrieval results restored
✓ Exact parent mapping restored
✓ No embeddings or retrieval recomputed
✓ Ready for H2 analysis


In [ ]:
# Cell 36 — H2: Does large Max–Min chunk size relate to retrieval degradation?

import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from transformers import AutoTokenizer

# ------------------------------------------------------------
# Load BGE tokenizer
# ------------------------------------------------------------

tokenizer_h2 = AutoTokenizer.from_pretrained(
    "BAAI/bge-m3"
)

MAX_LEN = 8192

# ------------------------------------------------------------
# Tokenize all saved BGE Max–Min chunks
# ------------------------------------------------------------

mm_texts = [
    row["text"]
    for row in bge_mm_corpus
]

mm_token_lengths = []

for i in range(0, len(mm_texts), 256):

    batch = mm_texts[i:i + 256]

    encoded = tokenizer_h2(
        batch,
        add_special_tokens=True,
        padding=False,
        truncation=False
    )

    mm_token_lengths.extend(
        len(ids)
        for ids in encoded["input_ids"]
    )

mm_token_lengths = np.asarray(mm_token_lengths)

assert len(mm_token_lengths) == 16763
assert mm_token_lengths.max() <= MAX_LEN

# ------------------------------------------------------------
# Parent -> chunk sizes
# ------------------------------------------------------------

parent_chunk_tokens = {}

for row, tok_len in zip(
    bge_mm_corpus,
    mm_token_lengths
):
    parent_chunk_tokens.setdefault(
        row["parent_id"], []
    ).append(int(tok_len))

assert len(parent_chunk_tokens) == 8973

# ------------------------------------------------------------
# Build query-level H2 features
# ------------------------------------------------------------

rows = []

for q in benchmark:

    qkey = (
        q["language"],
        q["query_id"]
    )

    relevant_parents = []

    for passage_idx, label in enumerate(
        q["is_selected"]
    ):

        if label != 1:
            continue

        key = (
            q["language"],
            q["query_id"],
            passage_idx
        )

        assert key in passage_key_to_parent

        relevant_parents.append(
            passage_key_to_parent[key]
        )

    # Max / mean chunk size across relevant passages
    all_relevant_chunk_tokens = []

    max_chunk_sentences = []

    for parent_id in relevant_parents:

        all_relevant_chunk_tokens.extend(
            parent_chunk_tokens[parent_id]
        )

        seg = next(
            s for s in bge_maxmin_segmentation
            if s["parent_id"] == parent_id
        )

        max_chunk_sentences.append(
            max(
                c["num_sentences"]
                for c in seg["chunks"]
            )
        )

    delta_ndcg = (
        maxmin_map[qkey]["ndcg10"]
        - native_map[qkey]["ndcg10"]
    )

    rows.append({
        "language": q["language"],
        "query_id": q["query_id"],
        "split": q["split"],
        "delta_ndcg": delta_ndcg,
        "rel_max_chunk_tokens": max(
            all_relevant_chunk_tokens
        ),
        "rel_mean_chunk_tokens": np.mean(
            all_relevant_chunk_tokens
        ),
        "rel_max_chunk_sentences": max(
            max_chunk_sentences
        )
    })

h2_df = pd.DataFrame(rows)

assert len(h2_df) == 459

# ------------------------------------------------------------
# Overall Max–Min chunk-size distribution
# ------------------------------------------------------------

print("=" * 80)
print("H2 — BGE MAX–MIN CHUNK SIZE")
print("=" * 80)

print("\nAll 16,763 chunks:")
print("Mean tokens:", f"{mm_token_lengths.mean():.2f}")
print("Median:", np.median(mm_token_lengths))
print("P90:", np.percentile(mm_token_lengths, 90))
print("P95:", np.percentile(mm_token_lengths, 95))
print("P99:", np.percentile(mm_token_lengths, 99))
print("Maximum:", mm_token_lengths.max())

for threshold in [256, 512, 1024, 2048]:

    print(
        f">{threshold} tokens:",
        int((mm_token_lengths > threshold).sum()),
        f"({(mm_token_lengths > threshold).mean()*100:.2f}%)"
    )

# ------------------------------------------------------------
# TEST correlations
# ------------------------------------------------------------

test = h2_df[
    h2_df["split"] == "test"
].copy()

print("\n" + "=" * 80)
print("H2 — TEST: RELEVANT CHUNK SIZE vs ΔnDCG")
print("=" * 80)

for feature in [
    "rel_max_chunk_tokens",
    "rel_mean_chunk_tokens",
    "rel_max_chunk_sentences"
]:

    valid = test[
        [feature, "delta_ndcg"]
    ].dropna()

    rho, p = spearmanr(
        valid[feature],
        valid["delta_ndcg"]
    )

    print(
        f"{feature:28s} "
        f"rho={rho:+.5f} "
        f"p={p:.5f} "
        f"n={len(valid)}"
    )

# ------------------------------------------------------------
# Fixed descriptive size bins
# ------------------------------------------------------------

test["size_bin"] = pd.cut(
    test["rel_max_chunk_tokens"],
    bins=[0, 128, 256, 512, 1024, np.inf],
    labels=[
        "<=128",
        "129-256",
        "257-512",
        "513-1024",
        ">1024"
    ],
    include_lowest=True
)

summary = (
    test.groupby("size_bin", observed=True)["delta_ndcg"]
    .agg(
        queries="count",
        mean_delta="mean",
        median_delta="median",
        improved=lambda x: (x > 0).sum(),
        degraded=lambda x: (x < 0).sum()
    )
    .reset_index()
)

print("\n" + "=" * 80)
print("H2 — TEST: ΔnDCG BY MAXIMUM RELEVANT CHUNK SIZE")
print("=" * 80)

print(
    summary.to_string(
        index=False,
        float_format=lambda x: f"{x:+.5f}"
    )
)

print("\n✓ H2 chunk-size analysis complete")
print("✓ No embeddings recomputed")
print("✓ No retrieval recomputed")

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 5.07MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

H2 — BGE MAX–MIN CHUNK SIZE

All 16,763 chunks:
Mean tokens: 57.03
Median: 51.0
P90: 102.0
P95: 117.0
P99: 163.0
Maximum: 2663
>256 tokens: 33 (0.20%)
>512 tokens: 21 (0.13%)
>1024 tokens: 6 (0.04%)
>2048 tokens: 3 (0.02%)

H2 — TEST: RELEVANT CHUNK SIZE vs ΔnDCG
rel_max_chunk_tokens         rho=+0.16083 p=0.01377 n=234
rel_mean_chunk_tokens        rho=+0.14048 p=0.03170 n=234
rel_max_chunk_sentences      rho=+0.12593 p=0.05438 n=234

H2 — TEST: ΔnDCG BY MAXIMUM RELEVANT CHUNK SIZE
size_bin  queries  mean_delta  median_delta  improved  degraded
   <=128      214    -0.03457      +0.00000        26        55
 129-256       16    -0.01398      +0.00000         2         1
 257-512        2    +0.10013      +0.10013         1         0
   >1024        2    -0.18454      -0.18454         0         1

✓ H2 chunk-size analysis complete
✓ No embeddings recomputed
✓ No retrieval recomputed


In [ ]:
## PHASE 2

In [ ]:
# Experiment 002 — Random Boundary-Matched Control
# Cell 30: Reload BGE-002 frozen inputs

import json
import numpy as np
from pathlib import Path
import importlib.util

BASE = Path("/content/drive/MyDrive/maxmin_research")

# ------------------------------------------------------------
# Frozen inputs
# ------------------------------------------------------------

with open(
    BASE / "frozen/msmarco_xi_native_corpus.json", "r"
) as f:
    native_corpus = json.load(f)

with open(
    BASE / "frozen/msmarco_xi_sentence_corpus.json", "r"
) as f:
    sentence_corpus = json.load(f)

with open(
    BASE / "frozen/msmarco_xi_frozen_benchmark.json", "r"
) as f:
    benchmark = json.load(f)

# ------------------------------------------------------------
# BGE-Max–Min segmentation from Experiment 002
# ------------------------------------------------------------

with open(
    BASE / "frozen/maxmin_segmentation_bge_m3.json", "r"
) as f:
    bge_mm_raw = json.load(f)

bge_mm_segmentation = bge_mm_raw["segmentation"]

# ------------------------------------------------------------
# Exact frozen sentence splitter
# ------------------------------------------------------------

splitter_path = BASE / "frozen/sentence_splitter.py"

spec = importlib.util.spec_from_file_location(
    "sentence_splitter",
    splitter_path
)

sentence_splitter = importlib.util.module_from_spec(spec)
spec.loader.exec_module(sentence_splitter)

split_sentences = sentence_splitter.split_sentences

# ------------------------------------------------------------
# Parent -> sentence records
# ------------------------------------------------------------

parent_sentences = {}

for row in sentence_corpus:
    parent_sentences.setdefault(
        row["parent_id"], []
    ).append(row)

# ------------------------------------------------------------
# BGE-Max–Min reference chunk counts
# ------------------------------------------------------------

bge_mm_chunk_counts = {
    row["parent_id"]: row["num_chunks"]
    for row in bge_mm_segmentation
}

bge_mm_num_sentences = {
    row["parent_id"]: row["num_sentences"]
    for row in bge_mm_segmentation
}

# ------------------------------------------------------------
# Verify frozen setup
# ------------------------------------------------------------

assert len(native_corpus) == 8973
assert len(sentence_corpus) == 31792
assert len(benchmark) == 459
assert len(bge_mm_segmentation) == 8973

assert sum(bge_mm_chunk_counts.values()) == 16763

assert len(parent_sentences) == 8973

# Verify splitter against frozen sentence corpus
sample = native_corpus[0]

assert split_sentences(sample["text"]) == [
    x["text"]
    for x in parent_sentences[sample["parent_id"]]
]

print("=" * 75)
print("EXPERIMENT 002 — BGE RANDOM CONTROL SETUP")
print("=" * 75)

print("Embedding model: BAAI/bge-m3")
print("Native passages:", len(native_corpus))
print("Sentence records:", len(sentence_corpus))
print("Frozen queries:", len(benchmark))

print(
    "BGE-Max–Min reference chunks:",
    sum(bge_mm_chunk_counts.values())
)

print(
    "BGE-Max–Min mean chunks/passage:",
    np.mean(
        list(bge_mm_chunk_counts.values())
    )
)

print("\nReference segmentation:")
print("Passages:", len(bge_mm_chunk_counts))
print(
    "Total chunks:",
    sum(bge_mm_chunk_counts.values())
)

print("\n✓ BGE-002 segmentation loaded")
print("✓ Exact 001 sentence splitter verified")
print("✓ Reference = BGE-Max–Min (16,763 chunks)")
print("✓ No embeddings performed")
print("✓ Ready to generate Random Boundary seed 42")

EXPERIMENT 002 — BGE RANDOM CONTROL SETUP
Embedding model: BAAI/bge-m3
Native passages: 8973
Sentence records: 31792
Frozen queries: 459
BGE-Max–Min reference chunks: 16763
BGE-Max–Min mean chunks/passage: 1.8681600356625432

Reference segmentation:
Passages: 8973
Total chunks: 16763

✓ BGE-002 segmentation loaded
✓ Exact 001 sentence splitter verified
✓ Reference = BGE-Max–Min (16,763 chunks)
✓ No embeddings performed
✓ Ready to generate Random Boundary seed 42


In [ ]:
# Experiment 002 — Random Boundary-Matched Control
# Cell 31: Generate BGE Random Boundary Control — SEED 42
#
# Same per-passage chunk count as BGE-Max–Min.
# Only boundary positions are randomized.
# No embeddings performed.

import json
import numpy as np
from collections import Counter

SEED = 42
rng = np.random.default_rng(SEED)

random42_bge_corpus = []

# ------------------------------------------------------------
# Generate random boundary-matched chunks
# ------------------------------------------------------------

for passage in native_corpus:

    parent_id = passage["parent_id"]
    language = passage["language"]

    sentences = parent_sentences[parent_id]

    n = len(sentences)

    # Reference chunk count from BGE-Max–Min
    k = bge_mm_chunk_counts[parent_id]

    assert n == bge_mm_num_sentences[parent_id]
    assert 1 <= k <= n

    # --------------------------------------------------------
    # k == 1 → keep entire passage unchanged
    # --------------------------------------------------------

    if k == 1:

        boundaries = [0, n]

    else:

        # Choose exactly k-1 distinct internal sentence gaps.
        #
        # Example:
        # n = 5
        #
        # S1 | S2 | S3 | S4 | S5
        #    1    2    3    4
        #
        # For k=3, choose exactly 2 of these 4 gaps.

        cut_positions = np.sort(
            rng.choice(
                np.arange(1, n),
                size=k - 1,
                replace=False
            )
        )

        boundaries = (
            [0]
            + cut_positions.tolist()
            + [n]
        )

    assert len(boundaries) == k + 1

    # --------------------------------------------------------
    # Create chunks
    # --------------------------------------------------------

    for chunk_idx in range(k):

        start = boundaries[chunk_idx]
        end = boundaries[chunk_idx + 1]

        sentence_slice = sentences[start:end]

        assert len(sentence_slice) > 0

        sentence_indices = list(
            range(start, end)
        )

        chunk_text = " ".join(
            s["text"]
            for s in sentence_slice
        )

        random42_bge_corpus.append({
            "chunk_id": (
                f"{parent_id}_random42_bge_{chunk_idx}"
            ),
            "parent_id": parent_id,
            "language": language,
            "chunk_idx": chunk_idx,
            "text": chunk_text,
            "num_sentences": len(sentence_slice),
            "sentence_indices": sentence_indices
        })


# ------------------------------------------------------------
# Exact per-passage count verification
# ------------------------------------------------------------

random_counts = Counter(
    row["parent_id"]
    for row in random42_bge_corpus
)

assert len(random_counts) == 8973

for parent_id, expected_k in bge_mm_chunk_counts.items():
    assert random_counts[parent_id] == expected_k

# ------------------------------------------------------------
# Global count verification
# ------------------------------------------------------------

assert len(random42_bge_corpus) == 16763

# ------------------------------------------------------------
# Structural statistics
# ------------------------------------------------------------

chunk_sizes = np.asarray([
    row["num_sentences"]
    for row in random42_bge_corpus
])

singleton_count = int(
    np.sum(chunk_sizes == 1)
)

one_chunk_passages = sum(
    k == 1
    for k in bge_mm_chunk_counts.values()
)

# ------------------------------------------------------------
# Compare with BGE-Max–Min reference structure
# ------------------------------------------------------------

reference_chunk_sizes = np.asarray([
    c["num_sentences"]
    for seg in bge_mm_segmentation
    for c in seg["chunks"]
])

assert len(reference_chunk_sizes) == 16763

# ------------------------------------------------------------
# Report
# ------------------------------------------------------------

print("=" * 80)
print("BGE RANDOM BOUNDARY CONTROL — SEED 42")
print("=" * 80)

print("Embedding model:", "BAAI/bge-m3")
print("Seed:", SEED)
print("Passages:", len(native_corpus))
print("Random chunks:", len(random42_bge_corpus))

print(
    "Mean chunks/passage:",
    f"{len(random42_bge_corpus) / len(native_corpus):.6f}"
)

print("\nSentences/chunk:")
print("Mean:", f"{chunk_sizes.mean():.6f}")
print("Median:", f"{np.median(chunk_sizes):.1f}")
print("P90:", f"{np.percentile(chunk_sizes, 90):.1f}")
print("P95:", f"{np.percentile(chunk_sizes, 95):.1f}")
print("P99:", f"{np.percentile(chunk_sizes, 99):.1f}")
print("Max:", int(chunk_sizes.max()))

print("\nSingleton chunks:")
print(
    singleton_count,
    f"({singleton_count / len(random42_bge_corpus) * 100:.2f}%)"
)

print("\nOne-chunk passages:")
print(
    one_chunk_passages,
    f"({one_chunk_passages / len(native_corpus) * 100:.2f}%)"
)

# ------------------------------------------------------------
# Final assertions
# ------------------------------------------------------------

assert (
    len(random42_bge_corpus)
    == len(reference_chunk_sizes)
    == 16763
)

assert np.all(
    [
        random_counts[parent_id]
        == bge_mm_chunk_counts[parent_id]
        for parent_id in bge_mm_chunk_counts
    ]
)

print("\n✓ Every passage has EXACTLY the BGE-Max–Min chunk count")
print("✓ Total chunks = 16,763")
print("✓ Sentence order preserved")
print("✓ Only boundary positions randomized")
print("✓ Seed explicitly fixed at 42")
print("✓ No embeddings performed")

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

OUT_PATH = (
    BASE /
    "frozen/random_boundary_bge_seed42.json"
)

with open(OUT_PATH, "w") as f:
    json.dump(
        {
            "method": "random_boundary_matched",
            "embedding_model": "BAAI/bge-m3",
            "seed": SEED,
            "reference": "BGE-Max-Min official-code segmentation",
            "num_passages": len(native_corpus),
            "num_chunks": len(random42_bge_corpus),
            "segmentation": random42_bge_corpus
        },
        f,
        ensure_ascii=False
    )

print("\n✓ Seed-42 BGE Random corpus saved:")
print(OUT_PATH)

BGE RANDOM BOUNDARY CONTROL — SEED 42
Embedding model: BAAI/bge-m3
Seed: 42
Passages: 8973
Random chunks: 16763
Mean chunks/passage: 1.868160

Sentences/chunk:
Mean: 1.896558
Median: 2.0
P90: 3.0
P95: 4.0
P99: 6.0
Max: 135

Singleton chunks:
7555 (45.07%)

One-chunk passages:
4179 (46.57%)

✓ Every passage has EXACTLY the BGE-Max–Min chunk count
✓ Total chunks = 16,763
✓ Sentence order preserved
✓ Only boundary positions randomized
✓ Seed explicitly fixed at 42
✓ No embeddings performed

✓ Seed-42 BGE Random corpus saved:
/content/drive/MyDrive/maxmin_research/frozen/random_boundary_bge_seed42.json


In [ ]:
# Experiment 002 — Random Boundary-Matched Control
# Cell 32: Determine BGE embedding reuse vs new workload

import json
import hashlib
from collections import defaultdict

# ------------------------------------------------------------
# Load previously saved BGE chunk corpora from 002
# ------------------------------------------------------------

with open(
    BASE / "frozen/fixed2_chunk_corpus_bge_m3.json",
    "r"
) as f:
    bge_fixed2_corpus = json.load(f)

# Max–Min BGE corpus was saved as a plain list
with open(
    BASE / "frozen/maxmin_chunk_corpus_bge_m3.json",
    "r"
) as f:
    bge_maxmin_corpus = json.load(f)

assert len(bge_fixed2_corpus) == 18171
assert len(bge_maxmin_corpus) == 16763
assert len(random42_bge_corpus) == 16763

# ------------------------------------------------------------
# Exact SHA-256 text hash
# ------------------------------------------------------------

def text_hash(text):
    return hashlib.sha256(
        text.encode("utf-8")
    ).hexdigest()

# ------------------------------------------------------------
# Build text-hash lookup tables for existing BGE embeddings
# ------------------------------------------------------------

sentence_hashes = defaultdict(list)

for row in sentence_corpus:
    sentence_hashes[text_hash(row["text"])].append(
        row["sentence_id"]
    )

native_hashes = defaultdict(list)

for row in native_corpus:
    native_hashes[text_hash(row["text"])].append(
        row["parent_id"]
    )

fixed2_hashes = defaultdict(list)

for row in bge_fixed2_corpus:
    fixed2_hashes[text_hash(row["text"])].append(
        row["chunk_id"]
    )

maxmin_hashes = defaultdict(list)

for row in bge_maxmin_corpus:
    maxmin_hashes[text_hash(row["text"])].append(
        row["chunk_id"]
    )

# ------------------------------------------------------------
# Determine reuse / new workload
# ------------------------------------------------------------

reuse_sentence = 0
reuse_native = 0
reuse_fixed2 = 0
reuse_maxmin = 0
needs_new = 0

new_unique_hashes = []
new_unique_set = set()

for row in random42_bge_corpus:

    h = text_hash(row["text"])

    # 1. Single-sentence chunk
    if (
        row["num_sentences"] == 1
        and h in sentence_hashes
    ):
        reuse_sentence += 1

    # 2. Exact whole native passage
    elif h in native_hashes:
        reuse_native += 1

    # 3. Identical to an already embedded Fixed-2 chunk
    elif h in fixed2_hashes:
        reuse_fixed2 += 1

    # 4. Identical to an already embedded Max–Min chunk
    elif h in maxmin_hashes:
        reuse_maxmin += 1

    # 5. Genuinely new text
    else:
        needs_new += 1

        if h not in new_unique_set:
            new_unique_set.add(h)
            new_unique_hashes.append(h)

# ------------------------------------------------------------
# Verify accounting
# ------------------------------------------------------------

total = (
    reuse_sentence
    + reuse_native
    + reuse_fixed2
    + reuse_maxmin
    + needs_new
)

assert total == 16763

# Genuinely new texts must be multi-sentence
new_hash_set = set(new_unique_hashes)

for row in random42_bge_corpus:
    if text_hash(row["text"]) in new_hash_set:
        assert row["num_sentences"] >= 2

print("=" * 80)
print("BGE RANDOM BOUNDARY SEED 42 — EMBEDDING REUSE PLAN")
print("=" * 80)

print("Total Random-42 chunks:", len(random42_bge_corpus))

print("\nReuse:")
print("Sentence embeddings:", reuse_sentence)
print("Native embeddings:", reuse_native)
print("BGE Fixed-2 embeddings:", reuse_fixed2)
print("BGE Max–Min embeddings:", reuse_maxmin)

print("\nNew:")
print("New chunk instances:", needs_new)
print("New unique chunk texts:", len(new_unique_hashes))

print("\nTotal accounted for:", total)

print("\n✓ All 16,763 Random-42 chunks accounted for")
print("✓ Reuse determined by exact SHA-256 text match")
print("✓ Only genuinely new multi-sentence texts require embedding")
print("✓ No embeddings performed")

# ------------------------------------------------------------
# Save exact new-text hash order
# ------------------------------------------------------------

RANDOM_CACHE = (
    BASE / "cache/bge_m3_random_boundary_embeddings"
)

RANDOM_CACHE.mkdir(
    parents=True,
    exist_ok=True
)

HASH_PATH = (
    RANDOM_CACHE /
    "seed42_new_unique_text_hashes.json"
)

with open(HASH_PATH, "w") as f:
    json.dump(new_unique_hashes, f)

print("\n✓ New-text hash list saved:")
print(HASH_PATH)

BGE RANDOM BOUNDARY SEED 42 — EMBEDDING REUSE PLAN
Total Random-42 chunks: 16763

Reuse:
Sentence embeddings: 7555
Native embeddings: 3174
BGE Fixed-2 embeddings: 2508
BGE Max–Min embeddings: 610

New:
New chunk instances: 2916
New unique chunk texts: 2913

Total accounted for: 16763

✓ All 16,763 Random-42 chunks accounted for
✓ Reuse determined by exact SHA-256 text match
✓ Only genuinely new multi-sentence texts require embedding
✓ No embeddings performed

✓ New-text hash list saved:
/content/drive/MyDrive/maxmin_research/cache/bge_m3_random_boundary_embeddings/seed42_new_unique_text_hashes.json


In [ ]:
# Experiment 002 — Random Boundary-Matched Control
# Cell 33: Token-length check for NEW BGE Random-42 texts

import json
import numpy as np
from transformers import AutoTokenizer

RANDOM_CACHE = (
    BASE / "cache/bge_m3_random_boundary_embeddings"
)

HASH_PATH = (
    RANDOM_CACHE /
    "seed42_new_unique_text_hashes.json"
)

# ------------------------------------------------------------
# Load exact new-text hashes
# ------------------------------------------------------------

with open(HASH_PATH, "r") as f:
    random_new_hashes = json.load(f)

assert len(random_new_hashes) == 2913

new_hash_set = set(random_new_hashes)

# ------------------------------------------------------------
# Rebuild exact hash -> text mapping
# ------------------------------------------------------------

hash_to_text = {}

for row in random42_bge_corpus:
    h = text_hash(row["text"])

    if h in new_hash_set:
        hash_to_text[h] = row["text"]

assert len(hash_to_text) == 2913

new_texts = [
    hash_to_text[h]
    for h in random_new_hashes
]

assert len(new_texts) == 2913

# ------------------------------------------------------------
# Load BGE tokenizer
# ------------------------------------------------------------

tokenizer_bge = AutoTokenizer.from_pretrained(
    "BAAI/bge-m3"
)

MAX_LEN = 8192

# ------------------------------------------------------------
# Token lengths
# ------------------------------------------------------------

lengths = []

for i in range(0, len(new_texts), 256):

    batch = new_texts[i:i + 256]

    encoded = tokenizer_bge(
        batch,
        add_special_tokens=True,
        padding=False,
        truncation=False
    )

    lengths.extend(
        len(ids)
        for ids in encoded["input_ids"]
    )

lengths = np.asarray(lengths)

assert len(lengths) == 2913

over_limit = lengths > MAX_LEN

print("=" * 80)
print("BGE RANDOM BOUNDARY SEED 42 — NEW TEXT LENGTH CHECK")
print("=" * 80)

print("Unique new texts:", len(lengths))
print("Minimum:", lengths.min())
print("Mean:", f"{lengths.mean():.2f}")
print("Median:", np.median(lengths))
print("P90:", np.percentile(lengths, 90))
print("P95:", np.percentile(lengths, 95))
print("P99:", np.percentile(lengths, 99))
print("Maximum:", lengths.max())

print(f"\n> {MAX_LEN} tokens:", int(over_limit.sum()))
print(
    f"% > {MAX_LEN}:",
    f"{over_limit.mean() * 100:.4f}%"
)

assert over_limit.sum() == 0

# Save exact token lengths for the embedding cell
LENGTH_PATH = (
    RANDOM_CACHE /
    "seed42_new_unique_token_lengths.npy"
)

np.save(LENGTH_PATH, lengths)

print("\n✓ 2,913 unique new texts verified")
print("✓ No text exceeds BGE-M3's 8,192-token limit")
print("✓ No truncation required")
print("✓ Exact token lengths saved")
print("✓ Ready for seed-42 BGE embedding")

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 5.07MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

BGE RANDOM BOUNDARY SEED 42 — NEW TEXT LENGTH CHECK
Unique new texts: 2913
Minimum: 6
Mean: 71.65
Median: 68.0
P90: 110.0
P95: 127.0
P99: 161.0
Maximum: 822

> 8192 tokens: 0
% > 8192: 0.0000%

✓ 2,913 unique new texts verified
✓ No text exceeds BGE-M3's 8,192-token limit
✓ No truncation required
✓ Exact token lengths saved
✓ Ready for seed-42 BGE embedding


In [ ]:
# Experiment 002 — Random Boundary-Matched Control
# Cell 34: Embed ONLY the 2,913 new unique BGE Random-42 texts

import time
import json
import numpy as np
import torch
from sentence_transformers import SentenceTransformer

RANDOM_CACHE = (
    BASE / "cache/bge_m3_random_boundary_embeddings"
)

HASH_PATH = (
    RANDOM_CACHE / "seed42_new_unique_text_hashes.json"
)

EMB_PATH = (
    RANDOM_CACHE / "seed42_new_unique_text_embeddings.npy"
)

# ------------------------------------------------------------
# Load exact new-text hashes
# ------------------------------------------------------------

with open(HASH_PATH, "r") as f:
    random_new_hashes = json.load(f)

assert len(random_new_hashes) == 2913

new_hash_set = set(random_new_hashes)

# Exact hash -> text mapping
hash_to_text = {}

for row in random42_bge_corpus:
    h = text_hash(row["text"])

    if h in new_hash_set:
        hash_to_text[h] = row["text"]

assert len(hash_to_text) == 2913

new_texts = [
    hash_to_text[h]
    for h in random_new_hashes
]

assert len(new_texts) == 2913

# ------------------------------------------------------------
# Resume protection
# ------------------------------------------------------------

if EMB_PATH.exists():

    print("Existing Random-42 BGE embedding cache found.")

    random_new_embeddings = np.load(EMB_PATH)

    print("Shape:", random_new_embeddings.shape)

    assert random_new_embeddings.shape == (2913, 1024)

    norms = np.linalg.norm(
        random_new_embeddings,
        axis=1
    )

    print(
        "Norms:",
        f"min={norms.min():.7f}, "
        f"mean={norms.mean():.7f}, "
        f"max={norms.max():.7f}"
    )

    assert np.allclose(norms, 1.0, atol=1e-5)

    print("\n✓ Existing cache verified")
    print("✓ No embedding recomputation needed")

else:

    # --------------------------------------------------------
    # Load BGE-M3
    # --------------------------------------------------------

    print("Loading BGE-M3...")

    assert torch.cuda.is_available()

    bge_model_random = SentenceTransformer(
        "BAAI/bge-m3",
        device="cuda"
    )

    bge_model_random.half()

    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )
    print(
        "Embedding dimension:",
        bge_model_random.get_sentence_embedding_dimension()
    )
    print(
        "Model dtype:",
        next(
            bge_model_random.parameters()
        ).dtype
    )

    assert (
        bge_model_random.get_sentence_embedding_dimension()
        == 1024
    )

    # --------------------------------------------------------
    # Length-sorted embedding
    # --------------------------------------------------------

    tokenizer_random = bge_model_random.tokenizer

    lengths = []

    for i in range(0, len(new_texts), 256):

        batch = new_texts[i:i + 256]

        encoded = tokenizer_random(
            batch,
            add_special_tokens=True,
            padding=False,
            truncation=False
        )

        lengths.extend(
            len(x)
            for x in encoded["input_ids"]
        )

    lengths = np.asarray(lengths)

    assert len(lengths) == 2913
    assert lengths.max() <= 8192

    sorted_indices = np.argsort(lengths)

    def choose_batch_size(max_tokens):
        if max_tokens <= 128:
            return 64
        elif max_tokens <= 256:
            return 32
        elif max_tokens <= 512:
            return 16
        else:
            return 8

    start = time.time()

    embeddings_sorted = []
    sorted_original_indices = []

    pos = 0
    total = len(sorted_indices)

    while pos < total:

        first_idx = sorted_indices[pos]

        batch_size = choose_batch_size(
            int(lengths[first_idx])
        )

        end = min(
            pos + batch_size,
            total
        )

        batch_indices = sorted_indices[pos:end]

        batch_texts = [
            new_texts[i]
            for i in batch_indices
        ]

        batch_emb = bge_model_random.encode(
            batch_texts,
            batch_size=len(batch_texts),
            normalize_embeddings=False,
            convert_to_numpy=True,
            show_progress_bar=False
        ).astype(np.float32)

        # Explicit L2 normalization
        batch_emb /= np.linalg.norm(
            batch_emb,
            axis=1,
            keepdims=True
        )

        embeddings_sorted.append(batch_emb)
        sorted_original_indices.extend(
            batch_indices.tolist()
        )

        pos = end

        if pos % 500 < batch_size or pos == total:

            elapsed = time.time() - start

            print(
                f"{pos:4d}/{total} "
                f"({pos/total*100:6.2f}%) | "
                f"elapsed={elapsed:.2f} sec"
            )

    # --------------------------------------------------------
    # Restore exact hash order
    # --------------------------------------------------------

    sorted_embeddings = np.vstack(
        embeddings_sorted
    )

    random_new_embeddings = np.empty_like(
        sorted_embeddings
    )

    for sorted_pos, original_idx in enumerate(
        sorted_original_indices
    ):
        random_new_embeddings[original_idx] = (
            sorted_embeddings[sorted_pos]
        )

    # --------------------------------------------------------
    # Integrity
    # --------------------------------------------------------

    assert random_new_embeddings.shape == (
        2913, 1024
    )

    norms = np.linalg.norm(
        random_new_embeddings,
        axis=1
    )

    elapsed = time.time() - start

    print("\nEmbedding complete.")
    print(f"Elapsed: {elapsed:.2f} sec")
    print("Shape:", random_new_embeddings.shape)
    print(
        "Norms:",
        f"min={norms.min():.7f}, "
        f"mean={norms.mean():.7f}, "
        f"max={norms.max():.7f}"
    )

    assert np.allclose(
        norms,
        1.0,
        atol=1e-5
    )

    # --------------------------------------------------------
    # Save
    # --------------------------------------------------------

    np.save(
        EMB_PATH,
        random_new_embeddings
    )

    print("\n✓ Saved 2,913 new BGE Random-42 embeddings")
    print("✓ Hash order preserved")
    print("✓ Explicit L2 normalization verified")

Loading BGE-M3...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

GPU: Tesla T4
Embedding dimension: 1024
Model dtype: torch.float16


/tmp/ipykernel_868/3041861199.py:105: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  bge_model_random.get_sentence_embedding_dimension()
/tmp/ipykernel_868/3041861199.py:115: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  bge_model_random.get_sentence_embedding_dimension()


 512/2913 ( 17.58%) | elapsed=2.47 sec
1024/2913 ( 35.15%) | elapsed=3.41 sec
1536/2913 ( 52.73%) | elapsed=4.58 sec
2048/2913 ( 70.31%) | elapsed=5.93 sec
2560/2913 ( 87.88%) | elapsed=7.60 sec
2913/2913 (100.00%) | elapsed=9.90 sec

Embedding complete.
Elapsed: 9.92 sec
Shape: (2913, 1024)
Norms: min=0.9999999, mean=1.0000000, max=1.0000001

✓ Saved 2,913 new BGE Random-42 embeddings
✓ Hash order preserved
✓ Explicit L2 normalization verified


In [ ]:
# Experiment 002 — Random Boundary-Matched Control
# Cell 35: Assemble complete BGE Random-42 embedding matrix

import json
import hashlib
import numpy as np
from collections import defaultdict

RANDOM_CACHE = (
    BASE / "cache/bge_m3_random_boundary_embeddings"
)

# ------------------------------------------------------------
# Load existing BGE embedding matrices
# ------------------------------------------------------------

bge_sentence_embeddings = np.load(
    BASE / "cache/bge_m3_sentence_embeddings/"
    "bge_m3_sentence_embeddings.npy"
)

bge_native_embeddings = np.load(
    BASE / "cache/bge_m3_native_embeddings/"
    "bge_m3_native_embeddings.npy"
)

bge_fixed2_embeddings = np.load(
    BASE / "cache/bge_m3_fixed2_embeddings/"
    "bge_m3_fixed2_embeddings.npy"
)

bge_maxmin_embeddings = np.load(
    BASE / "cache/bge_m3_maxmin_embeddings/"
    "bge_m3_maxmin_embeddings.npy"
)

bge_random_new_embeddings = np.load(
    RANDOM_CACHE /
    "seed42_new_unique_text_embeddings.npy"
)

with open(
    RANDOM_CACHE /
    "seed42_new_unique_text_hashes.json",
    "r"
) as f:
    random_new_hashes = json.load(f)

# ------------------------------------------------------------
# Load existing chunk corpora
# ------------------------------------------------------------

with open(
    BASE / "frozen/fixed2_chunk_corpus_bge_m3.json",
    "r"
) as f:
    bge_fixed2_corpus = json.load(f)

with open(
    BASE / "frozen/maxmin_chunk_corpus_bge_m3.json",
    "r"
) as f:
    bge_maxmin_corpus = json.load(f)

# ------------------------------------------------------------
# Verify dimensions
# ------------------------------------------------------------

assert bge_sentence_embeddings.shape == (31792, 1024)
assert bge_native_embeddings.shape == (8973, 1024)
assert bge_fixed2_embeddings.shape == (18171, 1024)
assert bge_maxmin_embeddings.shape == (16763, 1024)
assert bge_random_new_embeddings.shape == (2913, 1024)

assert len(random42_bge_corpus) == 16763
assert len(random_new_hashes) == 2913

# ------------------------------------------------------------
# Exact text hash
# ------------------------------------------------------------

def text_hash(text):
    return hashlib.sha256(
        text.encode("utf-8")
    ).hexdigest()

# ------------------------------------------------------------
# Build text -> embedding-row lookups
# ------------------------------------------------------------

sentence_hashes = defaultdict(list)

for i, row in enumerate(sentence_corpus):
    sentence_hashes[text_hash(row["text"])].append(i)

native_hashes = defaultdict(list)

for i, row in enumerate(native_corpus):
    native_hashes[text_hash(row["text"])].append(i)

fixed2_hashes = defaultdict(list)

for i, row in enumerate(bge_fixed2_corpus):
    fixed2_hashes[text_hash(row["text"])].append(i)

maxmin_hashes = defaultdict(list)

for i, row in enumerate(bge_maxmin_corpus):
    maxmin_hashes[text_hash(row["text"])].append(i)

new_hash_to_idx = {
    h: i
    for i, h in enumerate(random_new_hashes)
}

# ------------------------------------------------------------
# Assemble complete Random-42 matrix
# ------------------------------------------------------------

bge_random42_embeddings = np.empty(
    (16763, 1024),
    dtype=np.float32
)

random42_chunk_ids = []

source_sentence = 0
source_native = 0
source_fixed2 = 0
source_maxmin = 0
source_new = 0

for i, row in enumerate(random42_bge_corpus):

    h = text_hash(row["text"])

    # 1. Singleton → cached sentence embedding
    if (
        row["num_sentences"] == 1
        and h in sentence_hashes
    ):
        idx = sentence_hashes[h][0]

        bge_random42_embeddings[i] = (
            bge_sentence_embeddings[idx]
        )

        source_sentence += 1

    # 2. Exact native passage
    elif h in native_hashes:
        idx = native_hashes[h][0]

        bge_random42_embeddings[i] = (
            bge_native_embeddings[idx]
        )

        source_native += 1

    # 3. Existing Fixed-2 text
    elif h in fixed2_hashes:
        idx = fixed2_hashes[h][0]

        bge_random42_embeddings[i] = (
            bge_fixed2_embeddings[idx]
        )

        source_fixed2 += 1

    # 4. Existing Max–Min text
    elif h in maxmin_hashes:
        idx = maxmin_hashes[h][0]

        bge_random42_embeddings[i] = (
            bge_maxmin_embeddings[idx]
        )

        source_maxmin += 1

    # 5. Newly embedded Random text
    else:
        assert h in new_hash_to_idx

        idx = new_hash_to_idx[h]

        bge_random42_embeddings[i] = (
            bge_random_new_embeddings[idx]
        )

        source_new += 1

    random42_chunk_ids.append(
        row["chunk_id"]
    )

# ------------------------------------------------------------
# Integrity checks
# ------------------------------------------------------------

assert bge_random42_embeddings.shape == (
    16763, 1024
)

assert len(random42_chunk_ids) == 16763
assert len(set(random42_chunk_ids)) == 16763

total_sources = (
    source_sentence
    + source_native
    + source_fixed2
    + source_maxmin
    + source_new
)

assert total_sources == 16763
assert source_new == 2916

norms = np.linalg.norm(
    bge_random42_embeddings,
    axis=1
)

assert np.allclose(
    norms,
    1.0,
    atol=1e-5
)

# ------------------------------------------------------------
# Report
# ------------------------------------------------------------

print("=" * 80)
print("BGE RANDOM BOUNDARY SEED 42 — COMPLETE EMBEDDING ASSEMBLY")
print("=" * 80)

print("Shape:", bge_random42_embeddings.shape)

print("\nEmbedding sources:")
print("Sentence embeddings:", source_sentence)
print("Native embeddings:", source_native)
print("Fixed-2 embeddings:", source_fixed2)
print("Max–Min embeddings:", source_maxmin)
print("New Random embeddings:", source_new)

print("\nTotal:", total_sources)

print(
    "\nNorms:",
    f"min={norms.min():.7f}, "
    f"mean={norms.mean():.7f}, "
    f"max={norms.max():.7f}"
)

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

FULL_EMB_PATH = (
    RANDOM_CACHE /
    "seed42_bge_random_embeddings.npy"
)

FULL_ID_PATH = (
    RANDOM_CACHE /
    "seed42_random_chunk_ids.json"
)

np.save(
    FULL_EMB_PATH,
    bge_random42_embeddings
)

with open(FULL_ID_PATH, "w") as f:
    json.dump(
        random42_chunk_ids,
        f
    )

print("\n✓ Complete 16,763 × 1,024 Random-42 matrix saved")
print("✓ Chunk IDs saved")
print("✓ All embeddings verified as L2-normalized")
print("✓ No new embeddings generated")
print("✓ No retrieval performed")

BGE RANDOM BOUNDARY SEED 42 — COMPLETE EMBEDDING ASSEMBLY
Shape: (16763, 1024)

Embedding sources:
Sentence embeddings: 7555
Native embeddings: 3174
Fixed-2 embeddings: 2508
Max–Min embeddings: 610
New Random embeddings: 2916

Total: 16763

Norms: min=0.9999999, mean=1.0000000, max=1.0000001

✓ Complete 16,763 × 1,024 Random-42 matrix saved
✓ Chunk IDs saved
✓ All embeddings verified as L2-normalized
✓ No new embeddings generated
✓ No retrieval performed


In [ ]:
print("=== native_corpus row ===")
print(native_corpus[0])
print("native keys:", list(native_corpus[0].keys()))

print("\n=== benchmark row ===")
print(benchmark[0])
print("benchmark keys:", list(benchmark[0].keys()))

print("\n=== random42_bge_corpus row ===")
print(random42_bge_corpus[0])
print("random keys:", list(random42_bge_corpus[0].keys()))

=== native_corpus row ===
{'parent_id': 'hindi_1102432_p00', 'language': 'hindi', 'source_query_id': 1102432, 'query_idx': 0, 'passage_idx': 0, 'text': 'एक कंपनी एक विशिष्ट देश में निगमित होती है, अक्सर उस देश के एक छोटे उपसमूह, जैसे कि एक राज्य या प्रांत, की सीमाओं के भीतर। निगम तब उस राज्य में निगमन के कानूनों द्वारा शासित होता है। एक निगम या तो निजी या सार्वजनिक स्टॉक जारी कर सकता है, या इसे गैर-स्टॉक निगम के रूप में वर्गीकृत किया जा सकता है। यदि स्टॉक जारी किया जाता है, तो निगम आमतौर पर अपने शेयरधारकों द्वारा सीधे या परोक्ष रूप से शासित होगा।', 'label': 0}
native keys: ['parent_id', 'language', 'source_query_id', 'query_idx', 'passage_idx', 'text', 'label']

=== benchmark row ===
{'language': 'hindi', 'query_id': 1090350, 'query': 'वयस्कों में विस्थापन में सीमावर्ती व्यक्तित्व विकार के लक्षण', 'passages': ['सीमावर्ती विकार के लिए नौ डीएसएम-IV मानदंडों में से तीन इस समूह में आते हैं: 1. प्रभावी (भावनात्मक) अस्थिरता, जिसमें तीव्र, प्रकरणिक भावनात्मक पीड़ा, चिड़चिड़ापन और चिंता/घबराहट

In [ ]:
# Experiment 002 — Random Boundary-Matched Control
# Cell 36: Retrieval + exact parent-collapse evaluation

import json
import numpy as np
from collections import defaultdict

# ------------------------------------------------------------
# Load Random-42 embeddings
# ------------------------------------------------------------

RANDOM_CACHE = (
    BASE / "cache/bge_m3_random_boundary_embeddings"
)

bge_random42_embeddings = np.load(
    RANDOM_CACHE / "seed42_bge_random_embeddings.npy"
)

with open(
    RANDOM_CACHE / "seed42_random_chunk_ids.json",
    "r"
) as f:
    random42_chunk_ids = json.load(f)

assert bge_random42_embeddings.shape == (16763, 1024)
assert len(random42_chunk_ids) == 16763
assert len(random42_bge_corpus) == 16763

# ------------------------------------------------------------
# Build chunk indices by language
# ------------------------------------------------------------

chunk_indices_by_language = defaultdict(list)

for i, row in enumerate(random42_bge_corpus):
    chunk_indices_by_language[row["language"]].append(i)

# ------------------------------------------------------------
# Exact native parent lookup
#
# native_corpus:
#   language
#   source_query_id
#   passage_idx
#
# benchmark:
#   language
#   query_id
#   passages
#   is_selected
# ------------------------------------------------------------

native_parent_lookup = {}

for row in native_corpus:

    key = (
        row["language"],
        row["source_query_id"],
        row["passage_idx"]
    )

    assert key not in native_parent_lookup, (
        f"Duplicate native key found: {key}"
    )

    native_parent_lookup[key] = row["parent_id"]

# ------------------------------------------------------------
# Build relevant parent set for every benchmark query
# ------------------------------------------------------------

relevant_parent_ids = {}

for item in benchmark:

    language = item["language"]
    query_id = item["query_id"]

    passages = item["passages"]
    selections = item["is_selected"]

    assert len(passages) == len(selections), (
        f"Passage/selection mismatch for "
        f"{language}, query {query_id}"
    )

    relevant_parents = set()

    for passage_idx, selected in enumerate(selections):

        if int(selected) == 1:

            key = (
                language,
                query_id,
                passage_idx
            )

            assert key in native_parent_lookup, (
                f"Missing native parent mapping: {key}"
            )

            relevant_parents.add(
                native_parent_lookup[key]
            )

    assert len(relevant_parents) > 0, (
        f"No relevant parent found for "
        f"{language}, query {query_id}"
    )

    relevant_parent_ids[
        (language, query_id)
    ] = relevant_parents

# ------------------------------------------------------------
# Stable query embedding lookup
# ------------------------------------------------------------

assert bge_query_embeddings.shape[0] == len(benchmark)

query_embedding_lookup = {}

for i, item in enumerate(benchmark):

    key = (
        item["language"],
        item["query_id"]
    )

    assert key not in query_embedding_lookup, (
        f"Duplicate benchmark query key: {key}"
    )

    query_embedding_lookup[key] = (
        bge_query_embeddings[i].astype(np.float32)
    )

# ------------------------------------------------------------
# Metric functions
# ------------------------------------------------------------

def dcg_at_k(relevances, k=10):

    relevances = relevances[:k]

    return sum(
        rel / np.log2(rank + 2)
        for rank, rel in enumerate(relevances)
    )


def ndcg_at_k(ranked_parent_ids, relevant, k=10):

    if not relevant:
        return 0.0

    relevance_vector = [
        1 if pid in relevant else 0
        for pid in ranked_parent_ids
    ]

    dcg = dcg_at_k(
        relevance_vector,
        k
    )

    ideal = [
        1
    ] * min(len(relevant), k)

    idcg = dcg_at_k(
        ideal,
        k
    )

    return dcg / idcg if idcg > 0 else 0.0


def mrr_at_k(ranked_parent_ids, relevant, k=10):

    for rank, pid in enumerate(
        ranked_parent_ids[:k],
        start=1
    ):
        if pid in relevant:
            return 1.0 / rank

    return 0.0


def recall_at_k(ranked_parent_ids, relevant, k=10):

    if not relevant:
        return 0.0

    retrieved = set(
        ranked_parent_ids[:k]
    )

    return (
        len(retrieved & relevant)
        / len(relevant)
    )

# ------------------------------------------------------------
# Evaluate one split
# ------------------------------------------------------------

def evaluate_split(split_name, split_items):

    per_query = []

    ndcgs = []
    mrrs = []
    r1s = []
    r10s = []

    for item in split_items:

        language = item["language"]
        query_id = item["query_id"]

        query_key = (
            language,
            query_id
        )

        query_embedding = (
            query_embedding_lookup[query_key]
        )

        # ----------------------------------------------------
        # Own-language retrieval only
        # ----------------------------------------------------

        candidate_idx = np.asarray(
            chunk_indices_by_language[language],
            dtype=np.int64
        )

        candidate_embeddings = (
            bge_random42_embeddings[candidate_idx]
        )

        # All vectors are L2-normalized
        # Therefore dot product = cosine similarity
        scores = (
            candidate_embeddings @ query_embedding
        )

        # ----------------------------------------------------
        # Retrieve top 100 chunks
        # then collapse to parents
        # ----------------------------------------------------

        top_n = min(
            100,
            len(scores)
        )

        top_local = np.argpartition(
            -scores,
            top_n - 1
        )[:top_n]

        top_local = top_local[
            np.argsort(
                -scores[top_local]
            )
        ]

        top_chunk_indices = (
            candidate_idx[top_local]
        )

        top_chunk_scores = (
            scores[top_local]
        )

        # ----------------------------------------------------
        # Parent collapse:
        # maximum chunk score per parent
        # ----------------------------------------------------

        parent_best_score = {}

        for chunk_idx, score in zip(
            top_chunk_indices,
            top_chunk_scores
        ):

            parent_id = random42_bge_corpus[
                int(chunk_idx)
            ]["parent_id"]

            score = float(score)

            if (
                parent_id not in parent_best_score
                or score > parent_best_score[parent_id]
            ):
                parent_best_score[parent_id] = score

        ranked_parents = [
            pid
            for pid, _ in sorted(
                parent_best_score.items(),
                key=lambda x: x[1],
                reverse=True
            )
        ]

        # ----------------------------------------------------
        # Ground-truth parents
        # ----------------------------------------------------

        relevant = relevant_parent_ids[
            query_key
        ]

        # ----------------------------------------------------
        # Metrics
        # ----------------------------------------------------

        ndcg = ndcg_at_k(
            ranked_parents,
            relevant,
            10
        )

        mrr = mrr_at_k(
            ranked_parents,
            relevant,
            10
        )

        r1 = recall_at_k(
            ranked_parents,
            relevant,
            1
        )

        r10 = recall_at_k(
            ranked_parents,
            relevant,
            10
        )

        ndcgs.append(ndcg)
        mrrs.append(mrr)
        r1s.append(r1)
        r10s.append(r10)

        per_query.append({
            "language": language,
            "query_id": query_id,
            "n_relevant_parents": len(relevant),
            "n_retrieved_parents": len(ranked_parents),
            "ndcg@10": ndcg,
            "mrr@10": mrr,
            "recall@1": r1,
            "recall@10": r10,
        })

    return {
        "split": split_name,
        "n_queries": len(split_items),
        "ndcg@10": float(np.mean(ndcgs)),
        "mrr@10": float(np.mean(mrrs)),
        "recall@1": float(np.mean(r1s)),
        "recall@10": float(np.mean(r10s)),
        "per_query": per_query,
    }

# ------------------------------------------------------------
# Frozen DEV / TEST split
# ------------------------------------------------------------

dev_items = [
    x for x in benchmark
    if x["split"].lower() == "dev"
]

test_items = [
    x for x in benchmark
    if x["split"].lower() == "test"
]

assert len(dev_items) == 225
assert len(test_items) == 234

# ------------------------------------------------------------
# Run
# ------------------------------------------------------------

print("=" * 80)
print("BGE-M3 RANDOM BOUNDARY-MATCHED CONTROL — SEED 42")
print("=" * 80)

random_dev_results = evaluate_split(
    "DEV",
    dev_items
)

random_test_results = evaluate_split(
    "TEST",
    test_items
)

print("\nDEV")
print(
    f"nDCG@10   = {random_dev_results['ndcg@10']:.6f}"
)
print(
    f"MRR@10    = {random_dev_results['mrr@10']:.6f}"
)
print(
    f"Recall@1  = {random_dev_results['recall@1']:.6f}"
)
print(
    f"Recall@10 = {random_dev_results['recall@10']:.6f}"
)

print("\nTEST")
print(
    f"nDCG@10   = {random_test_results['ndcg@10']:.6f}"
)
print(
    f"MRR@10    = {random_test_results['mrr@10']:.6f}"
)
print(
    f"Recall@1  = {random_test_results['recall@1']:.6f}"
)
print(
    f"Recall@10 = {random_test_results['recall@10']:.6f}"
)

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

random_results = {
    "experiment": "002",
    "method": "random_boundary",
    "seed": 42,
    "embedding_model": "BAAI/bge-m3",
    "n_chunks": 16763,
    "dev": random_dev_results,
    "test": random_test_results,
}

RESULT_PATH = (
    BASE /
    "results/bge_m3_random_boundary_seed42_results.json"
)

with open(RESULT_PATH, "w") as f:
    json.dump(
        random_results,
        f,
        indent=2
    )

print("\n✓ Results saved:", RESULT_PATH)
print("✓ No new embeddings generated")

NameError: name 'bge_query_embeddings' is not defined

In [ ]:
# Cell 36A — Reload cached BGE-M3 query embeddings

from pathlib import Path
import numpy as np

candidates = sorted(
    BASE.glob("cache/**/*query*embedding*.npy")
)

print("Candidate query-embedding files:")
for p in candidates:
    print(" -", p)

assert len(candidates) >= 1, (
    "No cached BGE query embedding file found."
)

# Prefer the file whose shape is 459 x 1024
valid = []

for p in candidates:
    try:
        arr = np.load(p, mmap_mode="r")
        if arr.shape == (459, 1024):
            valid.append(p)
    except Exception:
        pass

assert len(valid) >= 1, (
    "Could not find a cached query embedding matrix "
    "with shape (459, 1024)."
)

QUERY_PATH = valid[0]

bge_query_embeddings = np.load(
    QUERY_PATH
).astype(np.float32)

assert bge_query_embeddings.shape == (459, 1024)

print("\n✓ Loaded:", QUERY_PATH)
print("✓ Shape:", bge_query_embeddings.shape)

norms = np.linalg.norm(
    bge_query_embeddings,
    axis=1
)

print(
    "✓ Norms:",
    f"min={norms.min():.7f}, "
    f"mean={norms.mean():.7f}, "
    f"max={norms.max():.7f}"
)

Candidate query-embedding files:
 - /content/drive/MyDrive/maxmin_research/cache/bge_m3_query_embeddings/bge_m3_query_embeddings.npy
 - /content/drive/MyDrive/maxmin_research/cache/qwen_query_embeddings/qwen_query_embeddings.npy

✓ Loaded: /content/drive/MyDrive/maxmin_research/cache/bge_m3_query_embeddings/bge_m3_query_embeddings.npy
✓ Shape: (459, 1024)
✓ Norms: min=0.9999999, mean=1.0000000, max=1.0000001


In [ ]:
# Experiment 002 — Random Boundary-Matched Control
# Cell 36B: Retrieval + exact parent-collapse evaluation

import json
import numpy as np
from collections import defaultdict

# ------------------------------------------------------------
# Build language -> chunk indices
# ------------------------------------------------------------

chunk_indices_by_language = defaultdict(list)

for i, row in enumerate(random42_bge_corpus):
    chunk_indices_by_language[row["language"]].append(i)

assert len(random42_bge_corpus) == 16763
assert bge_random42_embeddings.shape == (16763, 1024)

# ------------------------------------------------------------
# Exact native parent lookup
# ------------------------------------------------------------

native_parent_lookup = {}

for row in native_corpus:
    key = (
        row["language"],
        row["source_query_id"],
        row["passage_idx"]
    )

    assert key not in native_parent_lookup
    native_parent_lookup[key] = row["parent_id"]

# ------------------------------------------------------------
# Query -> relevant parent IDs
# ------------------------------------------------------------

relevant_parent_ids = {}

for item in benchmark:

    language = item["language"]
    query_id = item["query_id"]

    selections = item["is_selected"]

    relevant = set()

    for passage_idx, selected in enumerate(selections):

        if int(selected) == 1:

            key = (
                language,
                query_id,
                passage_idx
            )

            assert key in native_parent_lookup

            relevant.add(
                native_parent_lookup[key]
            )

    assert len(relevant) > 0

    relevant_parent_ids[
        (language, query_id)
    ] = relevant

assert len(relevant_parent_ids) == 459

# ------------------------------------------------------------
# Query embedding lookup
# ------------------------------------------------------------

assert bge_query_embeddings.shape == (459, 1024)

query_embedding_lookup = {}

for i, item in enumerate(benchmark):

    key = (
        item["language"],
        item["query_id"]
    )

    assert key not in query_embedding_lookup

    query_embedding_lookup[key] = (
        bge_query_embeddings[i]
    )

# ------------------------------------------------------------
# Metrics
# ------------------------------------------------------------

def dcg_at_k(rels, k=10):
    return sum(
        rel / np.log2(rank + 2)
        for rank, rel in enumerate(rels[:k])
    )


def ndcg_at_k(ranked, relevant, k=10):

    if not relevant:
        return 0.0

    rels = [
        1 if pid in relevant else 0
        for pid in ranked
    ]

    dcg = dcg_at_k(rels, k)

    ideal = [1] * min(len(relevant), k)

    idcg = dcg_at_k(ideal, k)

    return dcg / idcg if idcg else 0.0


def mrr_at_k(ranked, relevant, k=10):

    for rank, pid in enumerate(ranked[:k], 1):
        if pid in relevant:
            return 1.0 / rank

    return 0.0


def recall_at_k(ranked, relevant, k=10):

    if not relevant:
        return 0.0

    return (
        len(set(ranked[:k]) & relevant)
        / len(relevant)
    )

# ------------------------------------------------------------
# Evaluate
# ------------------------------------------------------------

def evaluate_split(split_items):

    ndcgs = []
    mrrs = []
    recalls1 = []
    recalls10 = []

    per_query = []

    for item in split_items:

        language = item["language"]
        query_id = item["query_id"]

        key = (
            language,
            query_id
        )

        query_vec = query_embedding_lookup[key]

        candidate_idx = np.asarray(
            chunk_indices_by_language[language],
            dtype=np.int64
        )

        candidate_vecs = (
            bge_random42_embeddings[candidate_idx]
        )

        # Normalized vectors -> cosine = dot product
        scores = candidate_vecs @ query_vec

        # Same 100-chunk candidate depth used before collapse
        top_n = min(100, len(scores))

        top_local = np.argpartition(
            -scores,
            top_n - 1
        )[:top_n]

        top_local = top_local[
            np.argsort(-scores[top_local])
        ]

        # ----------------------------------------------------
        # Parent collapse: max chunk score
        # ----------------------------------------------------

        parent_best = {}

        for local_idx in top_local:

            chunk_idx = int(
                candidate_idx[local_idx]
            )

            parent_id = random42_bge_corpus[
                chunk_idx
            ]["parent_id"]

            score = float(
                scores[local_idx]
            )

            if (
                parent_id not in parent_best
                or score > parent_best[parent_id]
            ):
                parent_best[parent_id] = score

        ranked_parents = [
            pid
            for pid, score in sorted(
                parent_best.items(),
                key=lambda x: x[1],
                reverse=True
            )
        ]

        relevant = relevant_parent_ids[key]

        ndcg = ndcg_at_k(
            ranked_parents,
            relevant,
            10
        )

        mrr = mrr_at_k(
            ranked_parents,
            relevant,
            10
        )

        r1 = recall_at_k(
            ranked_parents,
            relevant,
            1
        )

        r10 = recall_at_k(
            ranked_parents,
            relevant,
            10
        )

        ndcgs.append(ndcg)
        mrrs.append(mrr)
        recalls1.append(r1)
        recalls10.append(r10)

        per_query.append({
            "language": language,
            "query_id": query_id,
            "ndcg@10": ndcg,
            "mrr@10": mrr,
            "recall@1": r1,
            "recall@10": r10
        })

    return {
        "n_queries": len(split_items),
        "ndcg@10": float(np.mean(ndcgs)),
        "mrr@10": float(np.mean(mrrs)),
        "recall@1": float(np.mean(recalls1)),
        "recall@10": float(np.mean(recalls10)),
        "per_query": per_query
    }

# ------------------------------------------------------------
# Frozen splits
# ------------------------------------------------------------

dev_items = [
    x for x in benchmark
    if x["split"].lower() == "dev"
]

test_items = [
    x for x in benchmark
    if x["split"].lower() == "test"
]

assert len(dev_items) == 225
assert len(test_items) == 234

# ------------------------------------------------------------
# Run
# ------------------------------------------------------------

print("=" * 80)
print("BGE-M3 RANDOM BOUNDARY CONTROL — SEED 42")
print("=" * 80)

random_dev_results = evaluate_split(dev_items)
random_test_results = evaluate_split(test_items)

print("\nDEV")
print(f"nDCG@10   = {random_dev_results['ndcg@10']:.6f}")
print(f"MRR@10    = {random_dev_results['mrr@10']:.6f}")
print(f"Recall@1  = {random_dev_results['recall@1']:.6f}")
print(f"Recall@10 = {random_dev_results['recall@10']:.6f}")

print("\nTEST")
print(f"nDCG@10   = {random_test_results['ndcg@10']:.6f}")
print(f"MRR@10    = {random_test_results['mrr@10']:.6f}")
print(f"Recall@1  = {random_test_results['recall@1']:.6f}")
print(f"Recall@10 = {random_test_results['recall@10']:.6f}")

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

random_results = {
    "experiment": "002",
    "method": "random_boundary",
    "seed": 42,
    "embedding_model": "BAAI/bge-m3",
    "n_chunks": 16763,
    "dev": random_dev_results,
    "test": random_test_results
}

RESULT_PATH = (
    BASE /
    "results/bge_m3_random_boundary_seed42_results.json"
)

with open(RESULT_PATH, "w") as f:
    json.dump(
        random_results,
        f,
        indent=2
    )

print("\n✓ Results saved")
print("✓ No embeddings generated")
print("✓ No new data created")

BGE-M3 RANDOM BOUNDARY CONTROL — SEED 42

DEV
nDCG@10   = 0.610034
MRR@10    = 0.526691
Recall@1  = 0.317778
Recall@10 = 0.904444

TEST
nDCG@10   = 0.609006
MRR@10    = 0.528566
Recall@1  = 0.322650
Recall@10 = 0.895299

✓ Results saved
✓ No embeddings generated
✓ No new data created


In [ ]:
# Experiment 002
# Cell 37: Paired bootstrap — Random-42 vs Max–Min
# TEST split, 10,000 paired resamples

import json
import numpy as np

# ------------------------------------------------------------
# Load saved results
# ------------------------------------------------------------

random_path = (
    BASE / "results/bge_m3_random_boundary_seed42_results.json"
)

bge_path = (
    BASE / "results/bge_m3_results.json"
)

with open(random_path, "r") as f:
    random_results_saved = json.load(f)

with open(bge_path, "r") as f:
    bge_results_saved = json.load(f)

# ------------------------------------------------------------
# Get TEST Random-42 per-query results
# ------------------------------------------------------------

random_test_rows = (
    random_results_saved["test"]["per_query"]
)

assert len(random_test_rows) == 234

# ------------------------------------------------------------
# Get ALL Max–Min per-query results
# ------------------------------------------------------------

maxmin_rows = (
    bge_results_saved["results"]["maxmin"]["per_query"]
)

assert len(maxmin_rows) == 459

# ------------------------------------------------------------
# Index Max–Min results by exact query key
# ------------------------------------------------------------

maxmin_lookup = {
    (
        row["language"],
        row["query_id"]
    ): row
    for row in maxmin_rows
}

# ------------------------------------------------------------
# Align exactly to Random TEST queries
# ------------------------------------------------------------

random_lookup = {
    (
        row["language"],
        row["query_id"]
    ): row
    for row in random_test_rows
}

test_keys = list(random_lookup.keys())

assert len(test_keys) == 234
assert all(key in maxmin_lookup for key in test_keys)

print("✓ Random TEST queries:", len(test_keys))
print("✓ Matching Max–Min queries:", len(test_keys))

# ------------------------------------------------------------
# Build paired differences
#
# Positive = Random > Max–Min
# ------------------------------------------------------------

metrics = [
    "ndcg@10",
    "mrr@10",
    "recall@1",
    "recall@10"
]

differences = {}

for metric in metrics:

    random_values = np.array([
        random_lookup[key][metric]
        for key in test_keys
    ], dtype=np.float64)

    maxmin_values = np.array([
        maxmin_lookup[key][metric]
        for key in test_keys
    ], dtype=np.float64)

    differences[metric] = (
        random_values - maxmin_values
    )

# ------------------------------------------------------------
# Paired bootstrap
# ------------------------------------------------------------

N_BOOT = 10_000
SEED = 20261003

rng = np.random.default_rng(SEED)

n = len(test_keys)

bootstrap_results = {}

for metric in metrics:

    diff = differences[metric]

    observed = float(
        np.mean(diff)
    )

    # Paired query resampling
    indices = rng.integers(
        0,
        n,
        size=(N_BOOT, n)
    )

    bootstrap_means = (
        diff[indices].mean(axis=1)
    )

    ci_low, ci_high = np.percentile(
        bootstrap_means,
        [2.5, 97.5]
    )

    bootstrap_results[metric] = {
        "random_minus_maxmin": observed,
        "ci_95_low": float(ci_low),
        "ci_95_high": float(ci_high),
        "significant_95": bool(
            ci_low > 0 or ci_high < 0
        )
    }

# ------------------------------------------------------------
# Print
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("RANDOM-42 vs MAX–MIN — TEST PAIRED BOOTSTRAP")
print("=" * 80)

print("Queries:", n)
print("Bootstrap replicates:", N_BOOT)
print("Seed:", SEED)
print("95% percentile CI")
print("Positive = Random better than Max–Min")

for metric in metrics:

    result = bootstrap_results[metric]

    print(
        f"\n{metric.upper()}"
        f"\n  Random − Max–Min = "
        f"{result['random_minus_maxmin']:+.6f}"
        f"\n  95% CI = "
        f"[{result['ci_95_low']:+.6f}, "
        f"{result['ci_95_high']:+.6f}]"
        f"\n  Significant = "
        f"{result['significant_95']}"
    )

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

output = {
    "experiment": "002",
    "comparison": "random_boundary_seed42_vs_maxmin",
    "split": "TEST",
    "embedding_model": "BAAI/bge-m3",
    "random_seed": 42,
    "bootstrap_seed": SEED,
    "n_bootstrap": N_BOOT,
    "n_queries": n,
    "confidence_level": 0.95,
    "metrics": bootstrap_results
}

output_path = (
    BASE /
    "results/bge_m3_random_vs_maxmin_bootstrap_10000.json"
)

with open(output_path, "w") as f:
    json.dump(
        output,
        f,
        indent=2
    )

print("\n✓ Saved:", output_path)

✓ Random TEST queries: 234
✓ Matching Max–Min queries: 234


KeyError: 'ndcg@10'

In [ ]:
# Cell 37A — inspect actual Max–Min per-query schema

print("Max–Min row:")
print(maxmin_rows[0])

print("\nKeys:")
print(list(maxmin_rows[0].keys()))

print("\nRandom row:")
print(random_test_rows[0])

print("\nRandom keys:")
print(list(random_test_rows[0].keys()))

Max–Min row:
{'language': 'hindi', 'query_id': 1090350, 'ndcg10': 1.0, 'mrr10': 1.0, 'recall1': 1.0, 'recall10': 1.0}

Keys:
['language', 'query_id', 'ndcg10', 'mrr10', 'recall1', 'recall10']

Random row:
{'language': 'hindi', 'query_id': 264507, 'ndcg@10': 0.43067655807339306, 'mrr@10': 0.25, 'recall@1': 0.0, 'recall@10': 1.0}

Random keys:
['language', 'query_id', 'ndcg@10', 'mrr@10', 'recall@1', 'recall@10']


In [ ]:
# Experiment 002
# Cell 37: Random-42 vs Max–Min paired bootstrap
# TEST split — 10,000 paired resamples

import json
import numpy as np

# ------------------------------------------------------------
# Load saved results
# ------------------------------------------------------------

random_path = (
    BASE / "results/bge_m3_random_boundary_seed42_results.json"
)

bge_path = (
    BASE / "results/bge_m3_results.json"
)

with open(random_path, "r") as f:
    random_results_saved = json.load(f)

with open(bge_path, "r") as f:
    bge_results_saved = json.load(f)

# ------------------------------------------------------------
# TEST Random-42 results
# ------------------------------------------------------------

random_test_rows = (
    random_results_saved["test"]["per_query"]
)

assert len(random_test_rows) == 234

# ------------------------------------------------------------
# ALL Max–Min per-query results
# ------------------------------------------------------------

maxmin_rows = (
    bge_results_saved["results"]["maxmin"]["per_query"]
)

assert len(maxmin_rows) == 459

# ------------------------------------------------------------
# Exact query alignment
# ------------------------------------------------------------

random_lookup = {
    (
        row["language"],
        row["query_id"]
    ): row
    for row in random_test_rows
}

maxmin_lookup = {
    (
        row["language"],
        row["query_id"]
    ): row
    for row in maxmin_rows
}

test_keys = list(random_lookup.keys())

assert len(test_keys) == 234
assert len(set(test_keys)) == 234
assert all(
    key in maxmin_lookup
    for key in test_keys
)

print("✓ Random TEST queries:", len(test_keys))
print("✓ Matching Max–Min queries:", len(test_keys))

# ------------------------------------------------------------
# Metric name mapping
# ------------------------------------------------------------

metric_map = {
    "ndcg@10": "ndcg10",
    "mrr@10": "mrr10",
    "recall@1": "recall1",
    "recall@10": "recall10"
}

# ------------------------------------------------------------
# Paired differences
#
# Positive = Random better than Max–Min
# ------------------------------------------------------------

differences = {}

for random_metric, maxmin_metric in metric_map.items():

    random_values = np.array(
        [
            random_lookup[key][random_metric]
            for key in test_keys
        ],
        dtype=np.float64
    )

    maxmin_values = np.array(
        [
            maxmin_lookup[key][maxmin_metric]
            for key in test_keys
        ],
        dtype=np.float64
    )

    assert random_values.shape == (234,)
    assert maxmin_values.shape == (234,)

    differences[random_metric] = (
        random_values - maxmin_values
    )

# ------------------------------------------------------------
# Paired bootstrap
# ------------------------------------------------------------

N_BOOT = 10_000
SEED = 20261003

rng = np.random.default_rng(SEED)

n = len(test_keys)

bootstrap_results = {}

for metric in metric_map:

    diff = differences[metric]

    observed = float(
        np.mean(diff)
    )

    # Resample queries, preserving the pairing
    indices = rng.integers(
        0,
        n,
        size=(N_BOOT, n)
    )

    boot_means = (
        diff[indices].mean(axis=1)
    )

    ci_low, ci_high = np.percentile(
        boot_means,
        [2.5, 97.5]
    )

    bootstrap_results[metric] = {
        "random_minus_maxmin": observed,
        "ci_95_low": float(ci_low),
        "ci_95_high": float(ci_high),
        "significant_95": bool(
            ci_low > 0 or ci_high < 0
        )
    }

# ------------------------------------------------------------
# Print results
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("RANDOM-42 vs MAX–MIN — TEST PAIRED BOOTSTRAP")
print("=" * 80)

print("Queries:", n)
print("Bootstrap replicates:", N_BOOT)
print("Seed:", SEED)
print("CI: 95% percentile")
print("Positive difference = Random > Max–Min")

for metric in metric_map:

    r = bootstrap_results[metric]

    print(
        f"\n{metric.upper()}"
        f"\n  Random − Max–Min = "
        f"{r['random_minus_maxmin']:+.6f}"
        f"\n  95% CI = "
        f"[{r['ci_95_low']:+.6f}, "
        f"{r['ci_95_high']:+.6f}]"
        f"\n  Significant = "
        f"{r['significant_95']}"
    )

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

output = {
    "experiment": "002",
    "comparison": "random_boundary_seed42_vs_maxmin",
    "split": "TEST",
    "embedding_model": "BAAI/bge-m3",
    "random_seed": 42,
    "bootstrap_seed": SEED,
    "n_bootstrap": N_BOOT,
    "n_queries": n,
    "confidence_level": 0.95,
    "metrics": bootstrap_results
}

output_path = (
    BASE /
    "results/bge_m3_random_vs_maxmin_bootstrap_10000.json"
)

with open(output_path, "w") as f:
    json.dump(
        output,
        f,
        indent=2
    )

print("\n✓ Bootstrap results saved:")
print(output_path)

✓ Random TEST queries: 234
✓ Matching Max–Min queries: 234

RANDOM-42 vs MAX–MIN — TEST PAIRED BOOTSTRAP
Queries: 234
Bootstrap replicates: 10000
Seed: 20261003
CI: 95% percentile
Positive difference = Random > Max–Min

NDCG@10
  Random − Max–Min = +0.010568
  95% CI = [-0.006658, +0.028993]
  Significant = False

MRR@10
  Random − Max–Min = +0.017703
  95% CI = [-0.005151, +0.041754]
  Significant = False

RECALL@1
  Random − Max–Min = +0.010684
  95% CI = [-0.029915, +0.051282]
  Significant = False

RECALL@10
  Random − Max–Min = -0.019231
  95% CI = [-0.038462, -0.002137]
  Significant = True

✓ Bootstrap results saved:
/content/drive/MyDrive/maxmin_research/results/bge_m3_random_vs_maxmin_bootstrap_10000.json


In [ ]:
# ============================================================
# FINAL AUDIT — CELL 1: Reconnect + inspect existing results
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import json
import os

BASE = Path('/content/drive/MyDrive/maxmin_research')

print("BASE:", BASE)
print("Exists:", BASE.exists())

# Relevant result files
candidates = [
    BASE / "results" / "maxmin_qwen_official_results.json",
    BASE / "results" / "bge_m3_results.json",
    BASE / "results" / "bge_m3_paired_bootstrap_10000.json",
    BASE / "results" / "maxmin_qwen_official_results.json",
]

for p in candidates:
    print(f"\n{p.relative_to(BASE)}")
    print("  exists:", p.exists())
    if p.exists():
        print("  size:", round(p.stat().st_size / 1024, 2), "KB")

# Show result directory
results_dir = BASE / "results"
if results_dir.exists():
    print("\n=== RESULTS DIRECTORY ===")
    for p in sorted(results_dir.iterdir()):
        print(p.name)

Mounted at /content/drive
BASE: /content/drive/MyDrive/maxmin_research
Exists: True

results/maxmin_qwen_official_results.json
  exists: True
  size: 419.67 KB

results/bge_m3_results.json
  exists: True
  size: 159.44 KB

results/bge_m3_paired_bootstrap_10000.json
  exists: True
  size: 6.13 KB

results/maxmin_qwen_official_results.json
  exists: True
  size: 419.67 KB

=== RESULTS DIRECTORY ===
bge_m3_paired_bootstrap_10000.json
bge_m3_random_boundary_seed42_results.json
bge_m3_random_vs_maxmin_bootstrap_10000.json
bge_m3_results.json
checkpoint_native_complete.json
fixed2_per_query_results.json
fixed2_results.json
maxmin_qwen_official_results.json
native_baseline_corrected.json
native_baseline_corrected.txt
native_per_query_results.json
paired_bootstrap_10000_results.json


In [ ]:
# ============================================================
# EXPERIMENT 002 — CELL 2
# Inspect per-query result structure for cluster bootstrap
# ============================================================

import json
from collections import Counter

path = BASE / "results" / "bge_m3_results.json"

with open(path, "r", encoding="utf-8") as f:
    bge_results = json.load(f)

print("Top-level type:", type(bge_results).__name__)

if isinstance(bge_results, dict):
    print("\nTop-level keys:")
    for k, v in bge_results.items():
        if isinstance(v, (list, dict)):
            print(f"  {k}: {type(v).__name__}, len={len(v)}")
        else:
            print(f"  {k}: {type(v).__name__} = {v}")

# Find likely per-query containers
print("\n=== Candidate per-query sections ===")

if isinstance(bge_results, dict):
    for k, v in bge_results.items():
        if isinstance(v, list) and len(v) > 0:
            print(f"\n[{k}]")
            print("Length:", len(v))
            print("First item type:", type(v[0]).__name__)
            print("First item:", v[0])
        elif isinstance(v, dict):
            # Check for nested per-query lists
            for k2, v2 in v.items():
                if isinstance(v2, list) and len(v2) > 0:
                    print(f"\n[{k} -> {k2}]")
                    print("Length:", len(v2))
                    print("First item:", v2[0])

Top-level type: dict

Top-level keys:
  experiment: str = 002
  embedding_model: str = BAAI/bge-m3
  maxmin_sigmoid_variant: str = sigmoid(|C|-1)
  hard_thr: float = 0.6
  init_const: float = 1.5
  c: float = 0.9
  retrieval_scope: str = own_language
  retrieval: str = exact_dense_cosine
  parent_collapse: str = max_chunk_score
  primary_metric: str = nDCG@10
  results: dict, len=3

=== Candidate per-query sections ===


In [ ]:
# ============================================================
# EXPERIMENT 002 — CELL 3
# Inspect the three retrieval result sections
# ============================================================

for method, data in bge_results["results"].items():
    print(f"\n{'='*60}")
    print(f"METHOD: {method}")
    print(f"{'='*60}")
    print("Type:", type(data).__name__)

    if isinstance(data, dict):
        print("Keys:")
        for k, v in data.items():
            if isinstance(v, (list, dict)):
                print(f"  {k}: {type(v).__name__}, len={len(v)}")
            else:
                print(f"  {k}: {type(v).__name__} = {v}")

        # Show likely per-query records
        for k, v in data.items():
            if isinstance(v, list) and len(v) > 0:
                print(f"\nFirst item from '{k}':")
                print(v[0])


METHOD: native
Type: dict
Keys:
  method: str = BGE-Native
  n_queries: int = 459
  ndcg10: float = 0.6288715137429991
  mrr10: float = 0.5496005809731299
  recall1: float = 0.3572984749455338
  recall10: float = 0.9302832244008714
  per_query: list, len=459

First item from 'per_query':
{'language': 'hindi', 'query_id': 1090350, 'ndcg10': 1.0, 'mrr10': 1.0, 'recall1': 1.0, 'recall10': 1.0}

METHOD: fixed2
Type: dict
Keys:
  method: str = BGE-Fixed2
  n_queries: int = 459
  ndcg10: float = 0.5990369129383952
  mrr10: float = 0.512363834422658
  recall1: float = 0.3137254901960784
  recall10: float = 0.9172113289760349
  per_query: list, len=459

First item from 'per_query':
{'language': 'hindi', 'query_id': 1090350, 'ndcg10': 1.0, 'mrr10': 1.0, 'recall1': 1.0, 'recall10': 1.0}

METHOD: maxmin
Type: dict
Keys:
  method: str = BGE-MaxMin
  n_queries: int = 459
  ndcg10: float = 0.60165635914422
  mrr10: float = 0.5164384272227409
  recall1: float = 0.3202614379084967
  recall10: float =

In [ ]:
# ============================================================
# EXPERIMENT 002 — CELL 4
# Verify query_id clustering structure
# ============================================================

results = bge_results["results"]

for method in ["native", "fixed2", "maxmin"]:
    rows = results[method]["per_query"]

    query_ids = [r["query_id"] for r in rows]
    counts = Counter(query_ids)

    duplicated = {qid: n for qid, n in counts.items() if n > 1}

    print(f"\n{method.upper()}")
    print("-" * 40)
    print("Rows:", len(rows))
    print("Unique query_ids:", len(counts))
    print("Duplicated query_ids:", len(duplicated))

    if duplicated:
        print("Example duplicates:", list(duplicated.items())[:10])

# Check that all three methods contain exactly the same query IDs
id_sets = {
    method: {r["query_id"] for r in results[method]["per_query"]}
    for method in ["native", "fixed2", "maxmin"]
}

print("\n=== CROSS-METHOD ALIGNMENT ===")
print("Native == Fixed2:", id_sets["native"] == id_sets["fixed2"])
print("Native == MaxMin:", id_sets["native"] == id_sets["maxmin"])
print("Fixed2 == MaxMin:", id_sets["fixed2"] == id_sets["maxmin"])

print("\nQuery IDs per language:")
lang_counts = Counter(
    (r["language"] for r in results["native"]["per_query"])
)

print(dict(sorted(lang_counts.items())))


NATIVE
----------------------------------------
Rows: 459
Unique query_ids: 51
Duplicated query_ids: 51
Example duplicates: [(1090350, 9), (158720, 9), (373460, 9), (113570, 9), (233826, 9), (265717, 9), (190327, 9), (1060361, 9), (131336, 9), (1090352, 9)]

FIXED2
----------------------------------------
Rows: 459
Unique query_ids: 51
Duplicated query_ids: 51
Example duplicates: [(1090350, 9), (158720, 9), (373460, 9), (113570, 9), (233826, 9), (265717, 9), (190327, 9), (1060361, 9), (131336, 9), (1090352, 9)]

MAXMIN
----------------------------------------
Rows: 459
Unique query_ids: 51
Duplicated query_ids: 51
Example duplicates: [(1090350, 9), (158720, 9), (373460, 9), (113570, 9), (233826, 9), (265717, 9), (190327, 9), (1060361, 9), (131336, 9), (1090352, 9)]

=== CROSS-METHOD ALIGNMENT ===
Native == Fixed2: True
Native == MaxMin: True
Fixed2 == MaxMin: True

Query IDs per language:
{'bengali': 51, 'gujarati': 51, 'hindi': 51, 'kannada': 51, 'malayalam': 51, 'marathi': 51, 'punj

In [ ]:
# ============================================================
# EXPERIMENT 002 — CELL 5
# Query-ID Cluster Bootstrap
# 51 clusters × 9 language observations
# ============================================================

import numpy as np
import json

SEED = 20261003
N_BOOT = 10_000
ALPHA = 0.05

rng = np.random.default_rng(SEED)

methods = ["native", "fixed2", "maxmin"]
metrics = ["ndcg10", "mrr10", "recall1", "recall10"]

# ------------------------------------------------------------
# Build aligned rows:
# one row = one (query_id, language) observation
# ------------------------------------------------------------

rows_by_method = {
    m: results[m]["per_query"]
    for m in methods
}

# Key by (query_id, language) to guarantee exact alignment
key_sets = {
    m: {(r["query_id"], r["language"]) for r in rows_by_method[m]}
    for m in methods
}

assert key_sets["native"] == key_sets["fixed2"] == key_sets["maxmin"]

keys = sorted(key_sets["native"])

print("Total observations :", len(keys))

# Query IDs define the clusters
query_ids = sorted(set(qid for qid, lang in keys))

print("Unique query_id clusters:", len(query_ids))

# Verify each cluster contains 9 languages
cluster_sizes = {
    qid: sum(1 for k in keys if k[0] == qid)
    for qid in query_ids
}

print("Cluster sizes:", sorted(set(cluster_sizes.values())))

assert len(query_ids) == 51
assert set(cluster_sizes.values()) == {9}

# ------------------------------------------------------------
# Build arrays:
# shape = (51 clusters, 9 language observations)
# ------------------------------------------------------------

languages = sorted(set(lang for qid, lang in keys))

assert len(languages) == 9

data = {}

for method in methods:
    for metric in metrics:
        lookup = {
            (r["query_id"], r["language"]): r[metric]
            for r in rows_by_method[method]
        }

        matrix = np.array([
            [
                lookup[(qid, lang)]
                for lang in languages
            ]
            for qid in query_ids
        ], dtype=float)

        assert matrix.shape == (51, 9)
        data[(method, metric)] = matrix

# ------------------------------------------------------------
# Cluster bootstrap
#
# Each replicate samples 51 query_id clusters WITH replacement.
# All 9 language observations inside a selected cluster move
# together.
# ------------------------------------------------------------

comparisons = {
    "native_minus_maxmin": ("native", "maxmin"),
    "fixed2_minus_maxmin": ("fixed2", "maxmin"),
    "native_minus_fixed2": ("native", "fixed2"),
}

cluster_results = {}

for comparison_name, (a, b) in comparisons.items():

    cluster_results[comparison_name] = {}

    for metric in metrics:

        A = data[(a, metric)]
        B = data[(b, metric)]

        # Observed difference over all 459 observations
        observed = np.mean(A - B)

        bootstrap = np.empty(N_BOOT, dtype=float)

        for i in range(N_BOOT):
            sampled_clusters = rng.integers(
                0,
                len(query_ids),
                size=len(query_ids)
            )

            # Resample whole clusters, retaining all 9 languages
            sampled_A = A[sampled_clusters]
            sampled_B = B[sampled_clusters]

            bootstrap[i] = np.mean(sampled_A - sampled_B)

        ci_low, ci_high = np.percentile(
            bootstrap,
            [100 * ALPHA / 2,
             100 * (1 - ALPHA / 2)]
        )

        cluster_results[comparison_name][metric] = {
            "observed_difference": float(observed),
            "ci_95_lower": float(ci_low),
            "ci_95_upper": float(ci_high),
            "significant": bool(
                ci_low > 0 or ci_high < 0
            )
        }

# ------------------------------------------------------------
# Print final results
# ------------------------------------------------------------

print("\n" + "=" * 72)
print("QUERY-ID CLUSTER BOOTSTRAP — EXPERIMENT 002")
print("=" * 72)

print(f"Clusters       : {len(query_ids)}")
print(f"Observations   : {len(keys)}")
print(f"Languages/cluster: {len(languages)}")
print(f"Bootstrap reps : {N_BOOT}")
print(f"Seed           : {SEED}")
print(f"CI             : 95% percentile")

for comparison, metric_results in cluster_results.items():

    print(f"\n{comparison.upper()}")
    print("-" * 72)

    for metric, r in metric_results.items():

        print(
            f"{metric:10s} "
            f"diff={r['observed_difference']:+.6f}  "
            f"CI=[{r['ci_95_lower']:+.6f}, "
            f"{r['ci_95_upper']:+.6f}]  "
            f"significant={r['significant']}"
        )

# Save results
out_path = BASE / "results" / "bge_m3_cluster_bootstrap_queryid_10000.json"

payload = {
    "experiment": "002",
    "embedding_model": "BAAI/bge-m3",
    "bootstrap_type": "cluster_bootstrap_by_query_id",
    "n_clusters": len(query_ids),
    "observations_per_cluster": len(languages),
    "n_observations": len(keys),
    "n_bootstrap": N_BOOT,
    "seed": SEED,
    "confidence_level": 0.95,
    "comparisons": cluster_results,
}

with open(out_path, "w", encoding="utf-8") as f:
    json.dump(payload, f, indent=2)

print("\nSaved:")
print(out_path)

Total observations : 459
Unique query_id clusters: 51
Cluster sizes: [9]

QUERY-ID CLUSTER BOOTSTRAP — EXPERIMENT 002
Clusters       : 51
Observations   : 459
Languages/cluster: 9
Bootstrap reps : 10000
Seed           : 20261003
CI             : 95% percentile

NATIVE_MINUS_MAXMIN
------------------------------------------------------------------------
ndcg10     diff=+0.027215  CI=[+0.004054, +0.055794]  significant=True
mrr10      diff=+0.033162  CI=[+0.002306, +0.070407]  significant=True
recall1    diff=+0.037037  CI=[-0.010893, +0.093736]  significant=False
recall10   diff=+0.015251  CI=[+0.004357, +0.028322]  significant=True

FIXED2_MINUS_MAXMIN
------------------------------------------------------------------------
ndcg10     diff=-0.002619  CI=[-0.014940, +0.009690]  significant=False
mrr10      diff=-0.004075  CI=[-0.020540, +0.012279]  significant=False
recall1    diff=-0.006536  CI=[-0.032680, +0.017429]  significant=False
recall10   diff=+0.002179  CI=[-0.004357, +0.01089

In [ ]:
# ============================================================
# EXPERIMENT 002 — FREEZE FINAL CLUSTER-BOOTSTRAP RESULT
# ============================================================

import json
from pathlib import Path

bootstrap_path = (
    BASE / "results" /
    "bge_m3_cluster_bootstrap_queryid_10000.json"
)

assert bootstrap_path.exists(), "Cluster bootstrap file not found."

with open(bootstrap_path, "r", encoding="utf-8") as f:
    frozen_bootstrap = json.load(f)

print("=" * 72)
print("EXPERIMENT 002 — FINAL BOOTSTRAP AUDIT")
print("=" * 72)

# Core metadata
required_metadata = {
    "experiment": "002",
    "embedding_model": "BAAI/bge-m3",
    "bootstrap_type": "cluster_bootstrap_by_query_id",
    "n_clusters": 51,
    "observations_per_cluster": 9,
    "n_observations": 459,
    "n_bootstrap": 10000,
    "seed": 20261003,
    "confidence_level": 0.95,
}

print("\nMETADATA CHECK")
for key, expected in required_metadata.items():
    actual = frozen_bootstrap.get(key)
    status = "✓" if actual == expected else "✗"
    print(f"{status} {key}: {actual}")

# Comparison structure
expected_comparisons = [
    "native_minus_maxmin",
    "fixed2_minus_maxmin",
    "native_minus_fixed2",
]

expected_metrics = [
    "ndcg10",
    "mrr10",
    "recall1",
    "recall10",
]

print("\nRESULT STRUCTURE CHECK")

comparisons = frozen_bootstrap["comparisons"]

for comparison in expected_comparisons:
    assert comparison in comparisons
    for metric in expected_metrics:
        assert metric in comparisons[comparison]

    print(f"✓ {comparison}: 4 metrics present")

# Print frozen values
print("\nFROZEN RESULTS")
print("-" * 72)

for comparison in expected_comparisons:
    print(f"\n{comparison.upper()}")

    for metric in expected_metrics:
        r = comparisons[comparison][metric]

        print(
            f"  {metric:10s} "
            f"diff={r['observed_difference']:+.6f}  "
            f"CI=[{r['ci_95_lower']:+.6f}, "
            f"{r['ci_95_upper']:+.6f}]  "
            f"significant={r['significant']}"
        )

print("\n" + "=" * 72)
print("✓ EXPERIMENT 002 CLUSTER BOOTSTRAP VERIFIED")
print("=" * 72)
print("\nFrozen source:")
print(bootstrap_path)

EXPERIMENT 002 — FINAL BOOTSTRAP AUDIT

METADATA CHECK
✓ experiment: 002
✓ embedding_model: BAAI/bge-m3
✓ bootstrap_type: cluster_bootstrap_by_query_id
✓ n_clusters: 51
✓ observations_per_cluster: 9
✓ n_observations: 459
✓ n_bootstrap: 10000
✓ seed: 20261003
✓ confidence_level: 0.95

RESULT STRUCTURE CHECK
✓ native_minus_maxmin: 4 metrics present
✓ fixed2_minus_maxmin: 4 metrics present
✓ native_minus_fixed2: 4 metrics present

FROZEN RESULTS
------------------------------------------------------------------------

NATIVE_MINUS_MAXMIN
  ndcg10     diff=+0.027215  CI=[+0.004054, +0.055794]  significant=True
  mrr10      diff=+0.033162  CI=[+0.002306, +0.070407]  significant=True
  recall1    diff=+0.037037  CI=[-0.010893, +0.093736]  significant=False
  recall10   diff=+0.015251  CI=[+0.004357, +0.028322]  significant=True

FIXED2_MINUS_MAXMIN
  ndcg10     diff=-0.002619  CI=[-0.014940, +0.009690]  significant=False
  mrr10      diff=-0.004075  CI=[-0.020540, +0.012279]  significant=Fal